# Mounting

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import geopandas as gpd
from shapely.geometry import Polygon
from shapely.geometry import Point
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.cm as cm
from sklearn import preprocessing
from sklearn.cluster import Birch
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.special import rel_entr, kl_div
from scipy.stats import entropy
import math
import time
import random
from sklearn.metrics import r2_score

import warnings
warnings.filterwarnings('ignore')

path_seoul_data = '/content/drive/MyDrive/Colab Notebooks/datasets/Seoul/'
path_lta_data = '/content/drive/MyDrive/Colab Notebooks/data_fusion_fcl/lta_data/'

path_data = '/content/drive/MyDrive/Colab Notebooks/Workspace/tour_generation/data_tour/'
path_figure = '/content/drive/MyDrive/Colab Notebooks/Workspace/tour_generation/re1_figure/'
path_result = '/content/drive/MyDrive/Colab Notebooks/Workspace/tour_generation/re1_result/'
path_segregation = '/content/drive/MyDrive/Colab Notebooks/Workspace/tour_generation/re1_segregation/'

def reduce_mem_usage(props):
    start_mem_usg = props.memory_usage().sum() / 1024**2
    #print("Memory usage of properties dataframe is :",start_mem_usg," MB")
    NAlist = [] # Keeps track of columns that have missing values filled in.
    for col in props.columns:
        if (props[col].dtype != object) & (props[col].dtype != 'category'):  # Exclude strings

            # make variables for Int, max and min
            IsInt = False
            mx = props[col].max()
            mn = props[col].min()

            # Integer does not support NA, therefore, NA needs to be filled
            if not np.isfinite(props[col]).all():
                NAlist.append(col)
                props[col].fillna(mn-1,inplace=True)

            # test if column can be converted to an integer
            asint = props[col].fillna(0).astype(np.int64)
            result = (props[col] - asint)
            result = result.sum()
            if result > -0.01 and result < 0.01:
                IsInt = True

            if IsInt:
                if mn >= 0:
                    if mx < 255:
                        props[col] = props[col].astype(np.uint8)
                    elif mx < 65535:
                        props[col] = props[col].astype(np.uint16)
                    elif mx < 4294967295:
                        props[col] = props[col].astype(np.uint32)
                    else:
                        props[col] = props[col].astype(np.uint64)
                else:
                    if mn > np.iinfo(np.int8).min and mx < np.iinfo(np.int8).max:
                        props[col] = props[col].astype(np.int8)
                    elif mn > np.iinfo(np.int16).min and mx < np.iinfo(np.int16).max:
                        props[col] = props[col].astype(np.int16)
                    elif mn > np.iinfo(np.int32).min and mx < np.iinfo(np.int32).max:
                        props[col] = props[col].astype(np.int32)
                    elif mn > np.iinfo(np.int64).min and mx < np.iinfo(np.int64).max:
                        props[col] = props[col].astype(np.int64)
            else:
                props[col] = props[col].astype(np.float32)

    mem_usg = props.memory_usage().sum() / 1024**2
    return props

Mounted at /content/drive


# ***

# Controlled experiment: Sampling

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def rename_columns(df, i):
    df.rename(columns = {'TRIP_PURPOSE':'TRIP_PURPOSE_' + str(i)}, inplace = True)
    df.rename(columns = {'TRAVEL_MODE':'TRAVEL_MODE_' + str(i)}, inplace = True)
    df.rename(columns = {'ORIGIN_DISTANCE':'ORIGIN_DISTANCE_' + str(i)}, inplace = True)
    df.rename(columns = {'DESTINATION_DISTANCE':'DESTINATION_DISTANCE_' + str(i)}, inplace = True)
    df.rename(columns = {'ORIGIN_SUBZONE':'ORIGIN_SUBZONE_' + str(i)}, inplace = True)
    df.rename(columns = {'DESTINATION_SUBZONE':'DESTINATION_SUBZONE_' + str(i)}, inplace = True)
    df.rename(columns = {'TRIP_STARTTIME':'TRIP_STARTTIME_' + str(i)}, inplace = True)
    df.rename(columns = {'TRIP_ENDTIME':'TRIP_ENDTIME_' + str(i)}, inplace = True)
    df.rename(columns = {'ORIGIN_SUBZONE_X':'ORIGIN_SUBZONE_X_' + str(i)}, inplace = True)
    df.rename(columns = {'ORIGIN_SUBZONE_Y':'ORIGIN_SUBZONE_Y_' + str(i)}, inplace = True)
    df.rename(columns = {'DESTINATION_SUBZONE_X':'DESTINATION_SUBZONE_X_' + str(i)}, inplace = True)
    df.rename(columns = {'DESTINATION_SUBZONE_Y':'DESTINATION_SUBZONE_Y_' + str(i)}, inplace = True)

def generate_tour_hts(df_hts):
    df_final = pd.DataFrame()
    df_hts = df_hts[att]
    for z in range(2, TRIP_MAX_ + 1):
        df_hts_ = df_hts[(df_hts['TRIP_MAX'] == z)]
        df_result = df_hts_[df_hts_['TRIP_CNT'] == 1]
        df_result = df_result.drop(columns=['TRIP_CNT'])
        rename_columns(df_result, 1)
        for i in range(2, z+1):
            df_hts_i = df_hts_[df_hts_['TRIP_CNT'] == i]
            df_hts_i = df_hts_i.drop(columns=['TRIP_CNT'])
            rename_columns(df_hts_i, i)
            df_result = pd.merge(df_result, df_hts_i, on=['ID','AGE','GENDER', 'INCOME', 'TRIP_MAX'])
            df_result = df_result[df_result['TRIP_STARTTIME_' + str(i)] >= df_result['TRIP_ENDTIME_' + str(i-1)]]
        if z == 2:
            df_final = df_result
        else:
            df_final = pd.concat([df_final, df_result])
    return df_final

def distance(df, att):
    lon1 = np.radians(df[att[0]])
    lon2 = np.radians(df[att[1]])
    lat1 = np.radians(df[att[2]])
    lat2 = np.radians(df[att[3]])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return 6371*2 * np.arcsin(np.sqrt(a))

def find_non_closed_tours(df_long: pd.DataFrame) -> pd.DataFrame:
    """
    Return IDs where origin of the first trip != destination of the last trip.

    Assumes df_long contains:
      ID, TRIP_COUNT, ORIGIN_SUBZONE, DESTINATION_SUBZONE
    """
    required = {"ID", "TRIP_CNT", "ORIGIN_SUBZONE", "DESTINATION_SUBZONE"}
    missing = required - set(df_long.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    # Get first trip origin and last trip destination per ID
    g = df_long.sort_values(["ID", "TRIP_CNT"], kind="mergesort").groupby("ID", sort=False)

    first_origin = g["ORIGIN_SUBZONE"].first().rename("FIRST_ORIGIN")
    last_dest = g["DESTINATION_SUBZONE"].last().rename("LAST_DEST")

    out = pd.concat([first_origin, last_dest], axis=1).reset_index()

    # Flag non-closed tours
    out["IS_CLOSED"] = out["FIRST_ORIGIN"] == out["LAST_DEST"]
    non_closed = out.loc[~out["IS_CLOSED"]].copy()

    return non_closed



#####################################################################

att = ['ID', 'AGE', 'GENDER', 'INCOME', 'TRIP_CNT', 'TRIP_MAX', 'TRIP_PURPOSE', 'TRAVEL_MODE', 'TRIP_STARTTIME', 'TRIP_ENDTIME',
        'ORIGIN_SUBZONE', 'ORIGIN_SUBZONE_X', 'ORIGIN_SUBZONE_Y', 'DESTINATION_SUBZONE', 'DESTINATION_SUBZONE_X', 'DESTINATION_SUBZONE_Y']

TRIP_MAX_ = 7

df_hts = pd.read_csv(path_data + 'data_sgp_hts_trip.csv')
age_mapping = {0: 0, 1: 0, 2: 1, 3: 1, 4: 2, 5: 2, 6: 3, 7: 3}
df_hts["AGE"] = df_hts["AGE"].replace(age_mapping)

#df_hts = df_hts[df_hts['INCOME'] < 3]
non_closed_ids = find_non_closed_tours(df_hts)
non_closed_ids = non_closed_ids['ID'].drop_duplicates()
df_hts = df_hts[~df_hts['ID'].isin(non_closed_ids)]
df_hts_tour = generate_tour_hts(df_hts)


df_hts.to_csv(path_data + 'val_data_true_trip.csv', index = False)
df_hts_tour.to_csv(path_data + 'val_data_true_tour.csv', index = False)

df_id = df_hts[['ID']].drop_duplicates()

for ff in [0.1]:#[0.2, 0.4, 0.6]:
    print(f'fraction {ff}...')
    df_hts_ = df_id.sample(frac = ff)
    df_hts__ = pd.merge(df_hts_, df_hts)
    df_hts_tour__ = generate_tour_hts(df_hts__)
    df_hts__.to_csv(path_data + f'val_data_hts_trip_{ff}.csv', index = False)
    df_hts_tour__.to_csv(path_data + f'val_data_hts_tour_{ff}.csv', index = False)

    print(df_hts__.info())

    df_pcm = df_hts[['TRIP_STARTTIME', 'TRIP_ENDTIME', 'ORIGIN_SUBZONE', 'DESTINATION_SUBZONE',
                    'ORIGIN_SUBZONE_X', 'ORIGIN_SUBZONE_Y', 'DESTINATION_SUBZONE_X', 'DESTINATION_SUBZONE_Y']]

    df_pcm = reduce_mem_usage(df_pcm)
    df_pcm.to_csv(path_data + 'val_data_pcm_trip.csv', index = False)

    print('Done!')

fraction 0.1...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3894 entries, 0 to 3893
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ID                     3894 non-null   object 
 1   AGE                    3894 non-null   int64  
 2   GENDER                 3894 non-null   int64  
 3   INCOME                 3894 non-null   int64  
 4   TRIP_CNT               3894 non-null   int64  
 5   TRIP_MAX               3894 non-null   int64  
 6   TRIP_PURPOSE           3894 non-null   int64  
 7   TRIP_STARTTIME         3894 non-null   int64  
 8   TRIP_ENDTIME           3894 non-null   int64  
 9   TRAVEL_MODE            3894 non-null   int64  
 10  ORIGIN_SUBZONE         3894 non-null   object 
 11  ORIGIN_SUBZONE_X       3894 non-null   float64
 12  ORIGIN_SUBZONE_Y       3894 non-null   float64
 13  DESTINATION_SUBZONE    3894 non-null   object 
 14  DESTINATION_SUBZONE_X  3894 non-null   f

In [ ]:
df_hts_tour = pd.read_csv(path_data + 'val_data_true_tour.csv')
print(df_hts_tour.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15358 entries, 0 to 15357
Data columns (total 75 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       15358 non-null  object 
 1   AGE                      15358 non-null  int64  
 2   GENDER                   15358 non-null  int64  
 3   INCOME                   15358 non-null  int64  
 4   TRIP_MAX                 15358 non-null  int64  
 5   TRIP_PURPOSE_1           15358 non-null  int64  
 6   TRAVEL_MODE_1            15358 non-null  int64  
 7   TRIP_STARTTIME_1         15358 non-null  int64  
 8   TRIP_ENDTIME_1           15358 non-null  int64  
 9   ORIGIN_SUBZONE_1         15358 non-null  object 
 10  ORIGIN_SUBZONE_X_1       15358 non-null  float64
 11  ORIGIN_SUBZONE_Y_1       15358 non-null  float64
 12  DESTINATION_SUBZONE_1    15358 non-null  object 
 13  DESTINATION_SUBZONE_X_1  15358 non-null  float64
 14  DESTINATION_SUBZONE_Y_

# Controlled experiment: Stage 1 Model Embeding

In [ ]:
# ============================================================
# Discrete-latent encoder-only fusion (non-param decoder)
#   + Embedding encoder (Φ = [D,S,E,C] integer indices)
#   + Sparse-per-Φ support for q(H|Φ)
# ============================================================

import os, math, json, re, random
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional, Callable

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.cluster import KMeans

# ------------------------------------------------------------
# 0) Utilities: feature builders & small helpers
# ------------------------------------------------------------

def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast numerics to reduce RAM usage (simple & safe)."""
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if pd.api.types.is_numeric_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()
            if pd.api.types.is_float_dtype(col_type):
                df[col] = pd.to_numeric(df[col], downcast='float')
            else:
                df[col] = pd.to_numeric(df[col], downcast='integer')
    end_mem = df.memory_usage().sum() / 1024**2
    # print(f"[reduce_mem] {start_mem:.2f} -> {end_mem:.2f} MB")
    return df

def distance(df, att):
    """Great-circle distance (km). att = [lon_o, lon_d, lat_o, lat_d] (deg)."""
    lon1 = np.radians(df[att[0]]); lon2 = np.radians(df[att[1]])
    lat1 = np.radians(df[att[2]]); lat2 = np.radians(df[att[3]])
    dlon = lon2 - lon1; dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 6371*2*np.arcsin(np.sqrt(a))


def land_use(df_hts_, df_pcm_):
    """DESTINATION_SUBZONE -> LU_0..LU_5. Fills missing PCM zones with HTS global averages."""
    # 1. Calculate probabilities from HTS
    df = df_hts_[['DESTINATION_SUBZONE','TRIP_PURPOSE']].copy()
    df['Numbers'] = 1.0
    df = df.groupby(['DESTINATION_SUBZONE','TRIP_PURPOSE'])['Numbers'].sum().reset_index()

    # Normalize per zone
    df['Zone_Total'] = df.groupby('DESTINATION_SUBZONE')['Numbers'].transform('sum')
    df['Prob'] = df['Numbers'] / df['Zone_Total']

    # Pivot to LU_0...LU_5
    df_lu = df.pivot(index='DESTINATION_SUBZONE', columns='TRIP_PURPOSE', values='Prob').reset_index()
    df_lu.columns.name = None
    df_lu = df_lu.rename(columns={i: f'LU_{i}' for i in range(6)})
    df_lu = df_lu.fillna(0.0)

    # 2. Handle missing zones using PCM universe
    pcm_zones = pd.DataFrame({'DESTINATION_SUBZONE': df_pcm_['DESTINATION_SUBZONE'].unique()})
    df_final = pd.merge(pcm_zones, df_lu, on='DESTINATION_SUBZONE', how='left')

    # 3. Fill NaNs with the average of existing zones
    lu_cols = [f'LU_{i}' for i in range(6)]
    avg_values = df_lu[lu_cols].mean()
    df_final[lu_cols] = df_final[lu_cols].fillna(avg_values)

    return df_final

def mode_share(df_hts_, df_pcm_):
    """DESTINATION_SUBZONE -> TRANSIT_RATIO. Fills missing PCM zones with global HTS mean."""
    # 1. Calculate transit ratio per zone from HTS
    df = df_hts_[['DESTINATION_SUBZONE','TRAVEL_MODE']].copy()
    df['is_transit'] = (df['TRAVEL_MODE'] == 1).astype(float)

    # Group by zone and get mean (which is the probability P(mode=1|zone))
    df_ms = df.groupby('DESTINATION_SUBZONE')['is_transit'].mean().reset_index()
    df_ms.rename(columns={'is_transit': 'TRANSIT_RATIO'}, inplace=True)

    # 2. Handle missing zones using PCM universe
    pcm_zones = pd.DataFrame({'DESTINATION_SUBZONE': df_pcm_['DESTINATION_SUBZONE'].unique()})
    df_final = pd.merge(pcm_zones, df_ms, on='DESTINATION_SUBZONE', how='left')

    # 3. Fill NaNs with the global average transit ratio from known zones
    global_avg = df_ms['TRANSIT_RATIO'].mean()
    df_final['TRANSIT_RATIO'] = df_final['TRANSIT_RATIO'].fillna(global_avg)

    return df_final

# ------------------------------------------------------------
# 0.1) Discrete-Φ helpers (bins + clusters → indices)
# ------------------------------------------------------------

@dataclass
class DiscreteSpec:
    n_dist: int = 6
    n_start: int = 8
    n_end: int = 8
    n_lu_mode_clusters: int = 16
    dist_binning: str = "quantile"  # or "uniform"
    time_binning: str = "quantile"  # or "uniform"

def _fit_bins(x: np.ndarray, n: int, kind: str) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    if kind == "uniform":
        lo, hi = np.nanmin(x), np.nanmax(x)
        edges = np.linspace(lo, hi, n + 1)
    else:
        qs = np.linspace(0, 100, n + 1)
        edges = np.percentile(x, qs)
        edges[0]  = min(edges[0],  x.min()) - 1e-9
        edges[-1] = max(edges[-1], x.max()) + 1e-9
    return edges

def _digitize(x: np.ndarray, edges: np.ndarray) -> np.ndarray:
    return np.clip(np.digitize(x, edges, right=False) - 1, 0, len(edges) - 2)

def fit_semantic_spec_on_hts(
    df_hts: pd.DataFrame,
    spec: DiscreteSpec,
    distance_col="TRIP_DISTANCE",
    start_col="TRIP_STARTTIME",
    end_col="TRIP_ENDTIME",
    lu_cols=("LU_0","LU_1","LU_2","LU_3","LU_4","LU_5"),
    tr_col="TRANSIT_RATIO",
    km_random_state=42
) -> Dict:
    dist_edges  = _fit_bins(df_hts[distance_col].values, spec.n_dist,  spec.dist_binning)
    start_edges = _fit_bins(df_hts[start_col].values,   spec.n_start, spec.time_binning)
    end_edges   = _fit_bins(df_hts[end_col].values,     spec.n_end,   spec.time_binning)

    feat_cols = list(lu_cols) + [tr_col]
    X = df_hts[feat_cols].to_numpy(dtype=float)

    eps = 1e-8
    mu = np.nanmean(X, axis=0)
    sd = np.nanstd(X, axis=0)
    sd = np.where(sd < eps, 1.0, sd)
    Xz = (X - mu) / sd

    kmeans = KMeans(n_clusters=spec.n_lu_mode_clusters, n_init="auto", random_state=km_random_state)
    kmeans.fit(Xz)

    return {
        "dist_edges": dist_edges.tolist(),
        "start_edges": start_edges.tolist(),
        "end_edges": end_edges.tolist(),
        "lu_cols": list(lu_cols),
        "tr_col": tr_col,
        "lu_norm": {"mean": mu.tolist(), "std": sd.tolist(), "eps": eps, "feat_order": feat_cols},
        "kmeans_centers": kmeans.cluster_centers_.tolist(),
        "phi_segments": {
            "n_dist": spec.n_dist,
            "n_start": spec.n_start,
            "n_end": spec.n_end,
            "n_cluster": spec.n_lu_mode_clusters
        }
    }

def apply_semantic_spec(
    df: pd.DataFrame,
    fitted: Dict,
    distance_col="TRIP_DISTANCE",
    start_col="TRIP_STARTTIME",
    end_col="TRIP_ENDTIME"
) -> pd.DataFrame:
    df = df.copy()
    dist_edges  = np.array(fitted["dist_edges"], dtype=float)
    start_edges = np.array(fitted["start_edges"], dtype=float)
    end_edges   = np.array(fitted["end_edges"], dtype=float)

    d_bin = _digitize(df[distance_col].to_numpy(dtype=float), dist_edges)
    s_bin = _digitize(df[start_col].to_numpy(dtype=float),    start_edges)
    e_bin = _digitize(df[end_col].to_numpy(dtype=float),      end_edges)

    lu_meta = fitted["lu_norm"]
    feat_cols = lu_meta["feat_order"]
    mu = np.array(lu_meta["mean"], dtype=float)
    sd = np.array(lu_meta["std"], dtype=float)
    eps = float(lu_meta.get("eps", 1e-8))

    X = df[feat_cols].to_numpy(dtype=float)
    sd_safe = np.where(sd < eps, 1.0, sd)
    Xz = (X - mu) / sd_safe

    centers = np.array(fitted["kmeans_centers"], dtype=float)
    d2 = ((Xz[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    c_id = d2.argmin(axis=1)

    df["D_BIN"] = d_bin.astype(int)
    df["S_BIN"] = s_bin.astype(int)
    df["E_BIN"] = e_bin.astype(int)
    df["C_ID"]  = c_id.astype(int)
    return df

def build_phi_indices_from_bins(
    df: pd.DataFrame,
    cols=("D_BIN","S_BIN","E_BIN","C_ID")
) -> Tuple[pd.DataFrame, int]:
    """Store Φ as integer indices [D,S,E,C]."""
    df = df.copy()
    D = df[cols[0]].astype(int).to_numpy()
    S = df[cols[1]].astype(int).to_numpy()
    E = df[cols[2]].astype(int).to_numpy()
    C = df[cols[3]].astype(int).to_numpy()
    df["PHI_IDX"] = [[int(D[i]), int(S[i]), int(E[i]), int(C[i])] for i in range(len(df))]
    return df, 4  # four categorical fields

# ------------------------------------------------------------
# 1) Data containers (index-based Φ)
# ------------------------------------------------------------

class HTSDataset(Dataset):
    """One HTS row = (phi_idx[4], xy_id)."""
    def __init__(self, df_hts: pd.DataFrame, xy_ids: np.ndarray):
        self.phi_idx = np.vstack(df_hts["PHI_IDX"].to_numpy()).astype(np.int64)  # [N,4]
        self.xy = xy_ids.astype(np.int64)
    def __len__(self): return self.phi_idx.shape[0]
    def __getitem__(self, i):
        return self.phi_idx[i], self.xy[i]

class PCMDataset(Dataset):
    """One PCM row = (phi_idx[4], count)."""
    def __init__(self, df_pcm: pd.DataFrame):
        self.phi_idx = np.vstack(df_pcm["PHI_IDX"].to_numpy()).astype(np.int64)  # [N,4]
        self.count = df_pcm["COUNT"].to_numpy(dtype=np.float32)
    def __len__(self): return self.phi_idx.shape[0]
    def __getitem__(self, i):
        return self.phi_idx[i], self.count[i]

# ------------------------------------------------------------
# 2) Embedding Encoder (Φ indices -> embeddings -> MLP -> logits[K])
# ------------------------------------------------------------

class EncoderEmbed(nn.Module):
    """
    Embedding encoder: [D,S,E,C] indices -> concat(emb_D, emb_S, emb_E, emb_C) -> MLP -> logits[K].
    """
    def __init__(
        self,
        cardinals: Tuple[int, int, int, int],  # (n_dist, n_start, n_end, n_cluster)
        K: int,
        emb_dims: Tuple[int, int, int, int] = (16, 16, 16, 16),
        hidden: int = 256,
        num_layers: int = 2,
        dropout: float = 0.0
    ):
        super().__init__()
        nD, nS, nE, nC = map(int, cardinals)
        eD, eS, eE, eC = emb_dims

        self.emb_D = nn.Embedding(nD, eD)
        self.emb_S = nn.Embedding(nS, eS)
        self.emb_E = nn.Embedding(nE, eE)
        self.emb_C = nn.Embedding(nC, eC)

        in_dim = eD + eS + eE + eC
        layers: List[nn.Module] = []
        dims = [in_dim] + [hidden]*(num_layers-1) + [K]
        for i in range(len(dims)-2):
            layers += [nn.Linear(dims[i], dims[i+1]), nn.ReLU(inplace=True)]
            if dropout > 0:
                layers += [nn.Dropout(dropout)]
        layers += [nn.Linear(dims[-2], dims[-1])]
        self.net = nn.Sequential(*layers)

    def forward(self, phi_idx: torch.Tensor) -> torch.Tensor:
        """
        phi_idx: LongTensor [B,4] with columns [D,S,E,C].
        returns logits [B,K]
        """
        D = phi_idx[:, 0]; S = phi_idx[:, 1]; E = phi_idx[:, 2]; C = phi_idx[:, 3]
        z = torch.cat([self.emb_D(D), self.emb_S(S), self.emb_E(E), self.emb_C(C)], dim=-1)
        return self.net(z)

# ------------------------------------------------------------
# 3) Numerics, schedules, divergences, masked softmax
# ------------------------------------------------------------

def safe_log(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return torch.log(x.clamp_min(eps))

def entropy_categorical(probs: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return -(probs * safe_log(probs, eps)).sum(dim=-1)

def js_divergence(p: torch.Tensor, q: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    p = p / p.sum().clamp_min(eps)
    q = q / q.sum().clamp_min(eps)
    m = 0.5 * (p + q)
    kl_pm = (p * (safe_log(p, eps) - safe_log(m, eps))).sum()
    kl_qm = (q * (safe_log(q, eps) - safe_log(m, eps))).sum()
    return 0.5 * (kl_pm + kl_qm)

def softmax_tau(logits: torch.Tensor, tau: float) -> torch.Tensor:
    return F.softmax(logits / max(tau, 1e-6), dim=-1)

def get_tau(ep: int, total_ep: int, tau_start: float, tau_end: float, schedule: str = "linear") -> float:
    if total_ep <= 1: return tau_end
    t = (ep - 1) / (total_ep - 1)  # 0..1
    if schedule == "linear":
        s = t
    elif schedule == "cosine":
        s = 0.5 * (1 - math.cos(math.pi * t))
    elif schedule == "exp":
        s = 1 - math.exp(-5 * t)
    else:
        s = t
    return (1 - s) * tau_start + s * tau_end

# ------------------------------------------------------------
# 3.1) Sparse support policies & masked softmax
# ------------------------------------------------------------

SupportPolicy = Callable[[torch.Tensor, Optional[torch.Tensor]], torch.Tensor]
# signature: mask = policy(logits, phi_idx_b) -> bool tensor [B,K], True = allowed

def masked_softmax(logits: torch.Tensor,
                   mask: torch.Tensor,
                   tau: float = 1.0,
                   eps: float = 1e-12) -> torch.Tensor:
    B, K = logits.shape
    scaled = logits / max(tau, 1e-6)
    scaled = scaled.masked_fill(~mask, -1e9)
    q = F.softmax(scaled, dim=-1)
    q = q * mask.float()
    z = q.sum(dim=-1, keepdim=True).clamp_min(eps)
    return q / z

def topk_support_policy(k: int) -> SupportPolicy:
    def policy(logits: torch.Tensor, phi_idx_b: Optional[torch.Tensor] = None) -> torch.Tensor:
        B, K = logits.shape
        k_eff = min(max(1, k), K)
        idx = torch.topk(logits, k=k_eff, dim=-1).indices
        mask = torch.zeros_like(logits, dtype=torch.bool)
        mask.scatter_(1, idx, True)
        return mask
    return policy

def prob_threshold_policy(thr: float) -> SupportPolicy:
    def policy(logits: torch.Tensor, phi_idx_b: Optional[torch.Tensor] = None) -> torch.Tensor:
        p = F.softmax(logits, dim=-1)
        return p >= thr
    return policy

# ---- Bucketed support mining (index-based) ----

from collections import defaultdict

def build_phi_bucket_candidates_embed(
    encoder: nn.Module,
    hts_loader: DataLoader,
    K: int,
    tau: float,
    topk: int = 8,
    device: Optional[str] = None
) -> Dict[Tuple[int, int, int, int], np.ndarray]:
    """Aggregate q(H|Φ) over HTS and keep top-k H per Φ bucket. Φ is [D,S,E,C] indices."""
    enc_device = next(encoder.parameters()).device if device is None else torch.device(device)
    encoder.eval()
    acc = defaultdict(lambda: np.zeros(K, dtype=np.float64))

    with torch.no_grad():
        for phi_idx_b, _ in hts_loader:
            phi_idx_b = phi_idx_b.to(enc_device)              # [B,4] long
            logits = encoder(phi_idx_b)                       # [B,K]
            q = F.softmax(logits / max(tau, 1e-6), dim=-1).cpu().numpy()
            key_rows = phi_idx_b.cpu().numpy()                # [[D,S,E,C], ...]
            for i in range(key_rows.shape[0]):
                key = tuple(int(x) for x in key_rows[i])
                acc[key] += q[i]

    out = {}
    for key, vec in acc.items():
        k_eff = min(max(1, topk), K)
        keep = np.argpartition(-vec, k_eff-1)[:k_eff]
        keep = keep[np.argsort(-vec[keep])]
        out[key] = keep
    return out

def bucket_support_policy_embed(
    candidates: Dict[Tuple[int,int,int,int], np.ndarray]
) -> SupportPolicy:
    def policy(logits: torch.Tensor, phi_idx_b: Optional[torch.Tensor]) -> torch.Tensor:
        device = logits.device
        B, K = logits.shape
        mask = torch.zeros((B, K), dtype=torch.bool, device=device)
        keys = phi_idx_b.tolist()  # list of [D,S,E,C]
        for i in range(B):
            key = tuple(int(x) for x in keys[i])
            keep = candidates.get(key)
            if keep is None or len(keep) == 0:
                j = int(torch.argmax(logits[i]))
                mask[i, j] = True
            else:
                idx = torch.as_tensor(keep, dtype=torch.long, device=device)
                mask[i, idx] = True
        return mask
    return policy

# ------------------------------------------------------------
# 4) Non-param decoder  \hat{P}(XY|H)  (masked)
# ------------------------------------------------------------

@torch.no_grad()
def build_nonparam_decoder_xy_given_h(
    encoder: nn.Module,
    hts_loader: DataLoader,
    n_xy: int,
    K: int,
    device: str = "cpu",
    dtype: torch.dtype = torch.float32,
    progress: bool = True,
    tau: float = 1.0,
    support_policy: Optional[SupportPolicy] = None
) -> torch.Tensor:
    """
    \hat P(XY=j | H=h) = E[ 1{XY=j} q(h|Phi) ] / E[ q(h|Phi) ], with q masked per Φ (indices).
    Returns CPU tensor [n_xy, K] with columns summing to 1 (over kept H).
    """
    encoder.eval()
    num = torch.zeros((n_xy, K), dtype=dtype, device="cpu")
    den = torch.zeros((K,), dtype=dtype, device="cpu")

    for bi, (phi_idx_b, xy_b) in enumerate(hts_loader):
        if progress and bi % 50 == 0:
            print(f"[decoder] pass chunk {bi}")
        phi_idx_b = phi_idx_b.to(device=device, dtype=torch.long)
        logits = encoder(phi_idx_b)
        if support_policy is None:
            q = softmax_tau(logits, tau)
        else:
            mask = support_policy(logits, phi_idx_b)
            q = masked_softmax(logits, mask, tau)
        q = q.to("cpu")
        xy_b = xy_b.to("cpu")

        num.index_add_(0, xy_b, q)
        den += q.sum(dim=0)

    P = torch.zeros_like(num)
    mask = den > 0
    if mask.any():
        P[:, mask] = num[:, mask] / den[mask]
    P = P / P.sum(dim=0, keepdim=True).clamp_min(1e-12)
    return P

# ------------------------------------------------------------
# 5) Loss components (masked q everywhere)
# ------------------------------------------------------------

def compute_hts_nll(
    encoder: nn.Module,
    hts_loader: DataLoader,
    P_xy_given_h: torch.Tensor,
    log_norm: float,
    device: str,
    dtype: torch.dtype,
    progress: bool = True,
    tau: float = 1.0,
    support_policy: Optional[SupportPolicy] = None,
):
    total_nll = torch.zeros((), device=device, dtype=dtype)
    totalN = 0
    P_xy_given_h = P_xy_given_h.to(device=device, dtype=dtype)

    for bi, (phi_idx_b, xy_b) in enumerate(hts_loader):
        if progress and bi % 50 == 0:
            print(f"[hts-nll] pass chunk {bi}")
        B = phi_idx_b.shape[0]
        phi_idx_b = phi_idx_b.to(device=device, dtype=torch.long)
        xy_b  = xy_b.to(device=device, dtype=torch.long)

        logits = encoder(phi_idx_b)
        if support_policy is None:
            q = softmax_tau(logits, tau)
        else:
            mask = support_policy(logits, phi_idx_b)
            q = masked_softmax(logits, mask, tau)

        P_rows = P_xy_given_h.index_select(0, xy_b)    # [B, K]
        mix = (q * P_rows).sum(dim=-1).clamp_min(1e-12)
        nll_sum = -torch.log(mix).sum()

        total_nll = total_nll + nll_sum
        totalN += B

    avg_nll = total_nll / max(totalN, 1)
    if log_norm > 0:
        avg_nll = avg_nll / log_norm
    return avg_nll, totalN

def compute_latent_marginals(
    encoder: nn.Module,
    loader: DataLoader,
    K: int,
    device: str,
    dtype: torch.dtype,
    progress: bool = True,
    tau: float = 1.0,
    support_policy: Optional[SupportPolicy] = None,
):
    acc = torch.zeros(K, device=device, dtype=dtype)
    wsum = torch.zeros((), device=device, dtype=dtype)

    for bi, batch in enumerate(loader):
        if progress and bi % 50 == 0:
            print(f"[latent-marg] pass chunk {bi}")
        if len(batch) == 2:
            phi_idx_b, w_b = batch
            # HTS loader provides xy_b here; PCM loader provides COUNT; we handle both
            if w_b.dtype == torch.long or w_b.dtype == torch.int64:
                w_b = None
            else:
                w_b = w_b.to(device=device, dtype=dtype)
        else:
            phi_idx_b = batch[0]
            w_b = None

        phi_idx_b = phi_idx_b.to(device=device, dtype=torch.long)
        logits = encoder(phi_idx_b)
        if support_policy is None:
            q = softmax_tau(logits, tau)
        else:
            mask = support_policy(logits, phi_idx_b)
            q = masked_softmax(logits, mask, tau)

        if w_b is None:
            acc  = acc  + q.sum(dim=0)
            wsum = wsum + torch.tensor(q.shape[0], device=device, dtype=dtype)
        else:
            acc  = acc  + (q * w_b.unsqueeze(-1)).sum(dim=0)
            wsum = wsum + w_b.sum()

    p = acc / wsum.clamp_min(1e-12)
    p = p / p.sum().clamp_min(1e-12)
    return p

def compute_fusion_js(
    encoder: nn.Module,
    hts_loader: DataLoader,
    pcm_loader: DataLoader,
    K: int,
    device: str,
    dtype: torch.dtype,
    use_normalized: bool = True,
    tau: float = 1.0,
    support_policy: Optional[SupportPolicy] = None,
):
    p_hts = compute_latent_marginals(encoder, hts_loader, K, device, dtype, progress=False, tau=tau, support_policy=support_policy)
    p_pcm = compute_latent_marginals(encoder, pcm_loader, K, device, dtype, progress=False, tau=tau, support_policy=support_policy)
    m = 0.5 * (p_hts + p_pcm)
    js = 0.5 * (
        (p_hts * (torch.log(p_hts.clamp_min(1e-12)) - torch.log(m.clamp_min(1e-12)))).sum()
      + (p_pcm * (torch.log(p_pcm.clamp_min(1e-12)) - torch.log(m.clamp_min(1e-12)))).sum()
    )
    return js / math.log(2.0) if use_normalized else js

# ------------------------------------------------------------
# 6) Config
# ------------------------------------------------------------

@dataclass
class TrainConfig:
    K: int = 64
    enc_hidden: int = 256
    enc_layers: int = 2
    enc_dropout: float = 0.0
    epochs: int = 10
    batch_size_hts: int = 8192
    batch_size_pcm: int = 16384
    lr: float = 2e-3
    wd: float = 1e-4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    use_normalized_objective: bool = True
    lambda_fus: float = 1.0
    dtype: torch.dtype = torch.float32
    rebuild_every: int = 1
    progress: bool = True

    # softmax temperature annealing
    tau_start: float = 1.0
    tau_end:   float = 0.3
    tau_schedule: str = "cosine"

    # support policy controls
    topk_h: int = 8                   # per-Φ top-k for q(H|Φ)
    use_bucket_candidates: bool = False
    bucket_topk: int = 8
    refresh_bucket_every: int = 2     # epochs; 0 disables refresh

# ------------------------------------------------------------
# 7) Training loops (masked)
# ------------------------------------------------------------

def _prepare_xy_and_loaders(df_hts: pd.DataFrame,
                            df_pcm: pd.DataFrame,
                            cfg: TrainConfig):
    att_xy_series = (df_hts["ATT_X"].astype(str) + "|" + df_hts["ATT_Y"].astype(str))
    xy_vals = att_xy_series.to_numpy()
    xy_unique = pd.unique(xy_vals)
    xy_map = {k: i for i, k in enumerate(xy_unique)}
    xy_ids = np.vectorize(xy_map.__getitem__)(xy_vals)
    xy_classes = list(xy_unique)
    n_xy = len(xy_classes)

    hts_ds = HTSDataset(df_hts, xy_ids)
    pcm_ds = PCMDataset(df_pcm)
    hts_loader_full = DataLoader(hts_ds, batch_size=cfg.batch_size_hts, shuffle=False, num_workers=0, pin_memory=False)
    pcm_loader_full = DataLoader(pcm_ds, batch_size=cfg.batch_size_pcm, shuffle=False, num_workers=0, pin_memory=False)
    phi_dim = 4  # indices length, not used by embedding encoder
    return n_xy, xy_map, xy_classes, hts_loader_full, pcm_loader_full, phi_dim

def _make_support_policy(enc: nn.Module,
                         hts_loader_full: DataLoader,
                         tau: float,
                         cfg: TrainConfig) -> SupportPolicy:
    if cfg.use_bucket_candidates:
        cands = build_phi_bucket_candidates_embed(
            enc, hts_loader_full, cfg.K, tau,
            topk=cfg.bucket_topk, device=cfg.device
        )
        return bucket_support_policy_embed(cands)
    else:
        return topk_support_policy(cfg.topk_h)

def train_encoder_only_fusion(
    df_hts: pd.DataFrame,
    df_pcm: pd.DataFrame,                 # kept for signature parity
    cfg: TrainConfig,
    phi_segments_meta: Dict[str,int]      # pass fitted["phi_segments"]
) -> Dict:
    device = cfg.device
    dtype = cfg.dtype

    n_xy, xy_map, xy_classes, hts_loader_full, _pcm_loader_full, phi_dim = _prepare_xy_and_loaders(df_hts, df_pcm, cfg)

    # Embedding encoder
    cardinals = (phi_segments_meta["n_dist"], phi_segments_meta["n_start"],
                 phi_segments_meta["n_end"],  phi_segments_meta["n_cluster"])
    enc = EncoderEmbed(
        cardinals=cardinals, K=cfg.K,
        emb_dims=(16,16,16,16),
        hidden=cfg.enc_hidden, num_layers=cfg.enc_layers, dropout=cfg.enc_dropout
    ).to(device=device, dtype=torch.float32)
    opt = torch.optim.AdamW(enc.parameters(), lr=cfg.lr, weight_decay=cfg.wd)

    logXY = math.log(max(n_xy, 2))

    # Initial tau and support policy
    tau = get_tau(1, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
    support_policy = _make_support_policy(enc, hts_loader_full, tau, cfg)

    # Initial non-param decoder
    P_xy_h = build_nonparam_decoder_xy_given_h(
        encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
        device=device, dtype=dtype, progress=cfg.progress, tau=tau,
        support_policy=support_policy
    )  # CPU [n_xy, K]

    history = []
    for ep in range(1, cfg.epochs + 1):
        tau = get_tau(ep, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
        enc.train(); opt.zero_grad()

        # Optionally refresh bucket candidates (EM-style)
        if cfg.use_bucket_candidates and cfg.refresh_bucket_every > 0 and (ep == 1 or (ep % cfg.refresh_bucket_every) == 0):
            support_policy = _make_support_policy(enc, hts_loader_full, tau, cfg)

        # (A) HTS NLL only
        L_hts, _ = compute_hts_nll(
            encoder=enc, hts_loader=hts_loader_full,
            P_xy_given_h=P_xy_h,
            log_norm=(logXY if cfg.use_normalized_objective else 1.0),
            device=device, dtype=dtype, progress=cfg.progress, tau=tau,
            support_policy=support_policy
        )

        total = L_hts
        total.backward()
        opt.step()

        # (B) Rebuild decoder periodically with current tau + support
        if (ep % cfg.rebuild_every) == 0:
            P_xy_h = build_nonparam_decoder_xy_given_h(
                encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
                device=device, dtype=dtype, progress=cfg.progress, tau=tau,
                support_policy=support_policy
            )

        metrics = {
            "epoch": ep,
            "tau": float(tau),
            "L_hts": float(L_hts.detach().cpu().item()),
            "J_total": float(total.detach().cpu().item()),
        }
        history.append(metrics)
        print(f"[epoch {ep:03d}] tau={tau:.4f}  L_hts={metrics['L_hts']:.6f}  total={metrics['J_total']:.6f}")

    # ===== Final evaluation with final tau + fresh support =====
    tau_final = get_tau(cfg.epochs, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
    support_policy = _make_support_policy(enc, hts_loader_full, tau_final, cfg)

    P_xy_h_final = build_nonparam_decoder_xy_given_h(
        encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
        device=device, dtype=dtype, progress=cfg.progress, tau=tau_final,
        support_policy=support_policy
    )

    L_hts_final, _ = compute_hts_nll(
        encoder=enc, hts_loader=hts_loader_full, P_xy_given_h=P_xy_h_final,
        log_norm=(logXY if cfg.use_normalized_objective else 1.0),
        device=device, dtype=dtype, progress=False, tau=tau_final,
        support_policy=support_policy
    )

    J_final = float(L_hts_final.detach().cpu().item())
    print(f"[FINAL] K={cfg.K}  tau={tau_final:.4f}  L_hts={J_final:.6f}")

    pack = {
        "encoder_state_dict": enc.state_dict(),
        "encoder_cfg": {
            "cardinals": list(cardinals), "K": cfg.K, "hidden": cfg.enc_hidden,
            "layers": cfg.enc_layers, "dropout": cfg.enc_dropout, "dtype": str(dtype)
        },
        "xy_map": xy_map,
        "xy_classes": xy_classes,
        "P_xy_given_h": P_xy_h_final,         # CPU tensor [n_xy, K]
        "history": history,
        "config": cfg.__dict__,
        "tau_final": float(tau_final),
        "J_final": J_final,
        "L_hts_final": J_final,
        "phi_segments": dict(phi_segments_meta),
    }
    return {"model": enc, "pack": pack}

def train_encoder_only_fusion_finding_K(
    df_hts: pd.DataFrame,
    df_pcm: pd.DataFrame,
    cfg: TrainConfig,
    phi_segments_meta: Dict[str,int]
) -> Dict:
    device = cfg.device
    dtype = cfg.dtype

    # Prepare data/loaders
    n_xy, xy_map, xy_classes, hts_loader_full, pcm_loader_full, phi_dim = _prepare_xy_and_loaders(df_hts, df_pcm, cfg)

    # Build embedding encoder
    cardinals = (phi_segments_meta["n_dist"], phi_segments_meta["n_start"],
                 phi_segments_meta["n_end"],  phi_segments_meta["n_cluster"])
    enc = EncoderEmbed(
        cardinals=cardinals, K=cfg.K,
        emb_dims=(16,16,16,16),
        hidden=cfg.enc_hidden, num_layers=cfg.enc_layers, dropout=cfg.enc_dropout
    ).to(device=device, dtype=torch.float32)
    opt = torch.optim.AdamW(enc.parameters(), lr=cfg.lr, weight_decay=cfg.wd)

    logXY = math.log(max(n_xy, 2))

    # Init temperature + support policy
    tau = get_tau(1, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
    support_policy = _make_support_policy(enc, hts_loader_full, tau, cfg)

    # Initial non-parametric decoder P(XY|H)
    P_xy_h = build_nonparam_decoder_xy_given_h(
        encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
        device=device, dtype=dtype, progress=cfg.progress, tau=tau,
        support_policy=support_policy
    )

    history = []
    for ep in range(1, cfg.epochs + 1):
        tau = get_tau(ep, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
        enc.train(); opt.zero_grad()

        # Optional refresh of Φ-bucket candidates
        if cfg.use_bucket_candidates and cfg.refresh_bucket_every > 0 and (ep == 1 or (ep % cfg.refresh_bucket_every) == 0):
            support_policy = _make_support_policy(enc, hts_loader_full, tau, cfg)

        # ---- Likelihood-only objective ----
        L_hts, _ = compute_hts_nll(
            encoder=enc, hts_loader=hts_loader_full,
            P_xy_given_h=P_xy_h,
            log_norm=(logXY if cfg.use_normalized_objective else 1.0),
            device=device, dtype=dtype, progress=cfg.progress, tau=tau,
            support_policy=support_policy
        )

        total = L_hts  # <-- only likelihood drives training
        total.backward()
        opt.step()

        # Periodically rebuild decoder with current encoder + tau + support
        if (ep % cfg.rebuild_every) == 0:
            P_xy_h = build_nonparam_decoder_xy_given_h(
                encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
                device=device, dtype=dtype, progress=cfg.progress, tau=tau,
                support_policy=support_policy
            )

        metrics = {
            "epoch": ep,
            "tau": float(tau),
            "L_hts": float(L_hts.detach().cpu().item()),
            "J_total": float(total.detach().cpu().item()),  # equals L_hts
        }
        history.append(metrics)
        print(f"[epoch {ep:03d}] tau={tau:.4f}  L_hts={metrics['L_hts']:.6f}")

    # ===== Final evaluation =====
    tau_final = get_tau(cfg.epochs, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
    support_policy = _make_support_policy(enc, hts_loader_full, tau_final, cfg)

    # Final decoder
    P_xy_h_final = build_nonparam_decoder_xy_given_h(
        encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
        device=device, dtype=dtype, progress=cfg.progress, tau=tau_final,
        support_policy=support_policy
    )

    # Final likelihood (objective)
    L_hts_final, _ = compute_hts_nll(
        encoder=enc, hts_loader=hts_loader_full, P_xy_given_h=P_xy_h_final,
        log_norm=(math.log(max(n_xy,2)) if cfg.use_normalized_objective else 1.0),
        device=device, dtype=dtype, progress=False, tau=tau_final,
        support_policy=support_policy
    )

    # Fusion consistency (post-hoc) = JS(p_HTS(H), p_PCM(H))
    L_fus_final = compute_fusion_js(
        encoder=enc, hts_loader=hts_loader_full, pcm_loader=pcm_loader_full,
        K=cfg.K, device=device, dtype=dtype,
        use_normalized=cfg.use_normalized_objective, tau=tau_final,
        support_policy=support_policy
    )

    # Print requested summaries
    print(f"[FINAL] K={cfg.K}  tau={tau_final:.4f}")
    print(f"        Final likelihood (HTS NLL): {float(L_hts_final):.6f}")
    print(f"        Fusion consistency (JS):    {float(L_fus_final):.6f}")

    pack = {
        "encoder_state_dict": enc.state_dict(),
        "encoder_cfg": {"cardinals": list(cardinals), "K": cfg.K, "hidden": cfg.enc_hidden,
                        "layers": cfg.enc_layers, "dropout": cfg.enc_dropout, "dtype": str(dtype)},
        "xy_map": xy_map,
        "xy_classes": xy_classes,
        "P_xy_given_h": P_xy_h_final,
        "history": history,
        "config": cfg.__dict__,
        "tau_final": float(tau_final),
        # store both metrics explicitly
        "L_hts_final": float(L_hts_final.detach().cpu().item()),
        "fusion_consistency_js": float(L_fus_final.detach().cpu().item()),
        "phi_segments": dict(phi_segments_meta),
    }
    return {"model": enc, "pack": pack}


# ------------------------------------------------------------
# 8) Decoder table -> DataFrame
# ------------------------------------------------------------

def pack_to_dataframe(pack: dict) -> pd.DataFrame:
    P_xy_h = pack["P_xy_given_h"]         # [n_xy, K] CPU torch.Tensor
    xy_classes = pack["xy_classes"]       # list of "ATT_X|ATT_Y"
    K = int(pack["encoder_cfg"]["K"])

    P = P_xy_h.detach().cpu().numpy()     # (n_xy, K)
    df = pd.DataFrame({"ATT_XY": xy_classes})
    h_cols = [f"H_{h}" for h in range(K)]
    df_probs = pd.DataFrame(P, columns=h_cols)
    df = pd.concat([df, df_probs], axis=1)
    df = df.melt(id_vars=["ATT_XY"], value_vars=h_cols,
                 var_name="H", value_name="PROB")
    df["H"] = df["H"].apply(lambda s: int(re.sub(r"^H_", "", s)))

    def split_att(x):
        parts = str(x).split("|", 1)
        if len(parts) == 2: return parts[0], parts[1]
        else: return parts[0], ""
    att = df["ATT_XY"].apply(split_att)
    df["ATT_X"] = att.apply(lambda t: t[0])
    df["ATT_Y"] = att.apply(lambda t: t[1])

    return df[["ATT_X", "ATT_Y", "H", "PROB"]].reset_index(drop=True)

# ------------------------------------------------------------
# 9) Latent diagnostics from q (supports sparsity via policies)
# ------------------------------------------------------------

def _latent_diagnostics_from_q(
    q_latent: np.ndarray,
    weights: Optional[np.ndarray],
    eps: float = 1e-12
) -> Dict[str, float]:
    N, K = q_latent.shape
    w = np.ones(N, dtype=np.float64) if weights is None else np.asarray(weights, dtype=np.float64)
    w = np.clip(w, 0.0, None)
    wsum = w.sum() + eps

    H = -(q_latent * np.log(np.clip(q_latent, eps, 1.0))).sum(axis=1)
    H_mean = float((H * w).sum() / wsum)
    H_norm = H_mean / math.log(K)

    p = (q_latent * w[:, None]).sum(axis=0) / wsum
    p = p / p.sum()
    H_marg = float(-(p * np.log(np.clip(p, eps, 1.0))).sum())
    I_est  = float(max(0.0, H_marg - H_mean))

    qmax = q_latent.max(axis=1)
    q2   = np.partition(q_latent, -2, axis=1)[:, -2]
    gap  = qmax - q2
    qmax_mean = float((qmax * w).sum() / wsum)
    gap_mean  = float((gap  * w).sum() / wsum)

    KL_qU_mean = float(math.log(K) - H_mean)
    KL_qU_norm = KL_qU_mean / math.log(K)

    winners = q_latent.argmax(axis=1)
    win_counts = np.bincount(winners, minlength=K).astype(np.float64)
    win_ratio_max = float(win_counts.max() / max(N, 1))

    return {
        "E[H(q)]": H_mean,
        "E[H(q)]/logK": H_norm,
        "H_marg": H_marg,
        "I_est = H(P) - E[H(q)]": I_est,
        "E[max q]": qmax_mean,
        "E[max-min2 gap]": gap_mean,
        "E[KL(q||U)]/logK": KL_qU_norm,
        "argmax_dominance_ratio": win_ratio_max,
        "K": float(K),
        "N": float(N),
    }

@torch.no_grad()
def _encode_q_from_df(
    df: 'pd.DataFrame',
    encoder: 'nn.Module',
    K: int,
    device: str,
    batch_size: int = 8192,
    tau: float = 1.0,
    weights_col: Optional[str] = None,
    support_policy: Optional[SupportPolicy] = None
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    weights = None
    if (weights_col is not None) and (weights_col in df.columns):
        weights = df[weights_col].to_numpy(dtype=np.float64)

    phi_np = np.vstack(df["PHI_IDX"].to_numpy()).astype(np.int64)
    N = phi_np.shape[0]
    q_blocks = []

    encoder.eval()
    for start in range(0, N, batch_size):
        end = min(N, start + batch_size)
        phi_b = torch.from_numpy(phi_np[start:end]).to(device=device, dtype=torch.long)
        logits = encoder(phi_b)
        if support_policy is None:
            q = softmax_tau(logits, tau)
        else:
            mask = support_policy(logits, phi_b)
            q = masked_softmax(logits, mask, tau)
        q_blocks.append(q.cpu().numpy())

    q_latent = np.vstack(q_blocks)
    return q_latent, weights

def uniformity_report_from_df(
    df_hts: 'pd.DataFrame',
    df_pcm: 'pd.DataFrame',
    encoder: 'nn.Module',
    pack: Dict,
    device: str,
    batch_size: int = 8192,
    tau: Optional[float] = None,
    support_policy: Optional[SupportPolicy] = None,
) -> Dict[str, Dict[str, float]]:
    if tau is None:
        tau = float(pack.get("tau_final", 1.0))
    K = int(pack["encoder_cfg"]["K"])

    q_hts, _ = _encode_q_from_df(df_hts, encoder, K, device, batch_size, tau, weights_col=None, support_policy=support_policy)
    rep_hts = _latent_diagnostics_from_q(q_hts, weights=None)

    wcol = "COUNT" if "COUNT" in df_pcm.columns else None
    q_pcm, w_pcm = _encode_q_from_df(df_pcm, encoder, K, device, batch_size, tau, weights_col=wcol, support_policy=support_policy)
    rep_pcm = _latent_diagnostics_from_q(q_pcm, weights=w_pcm)

    return {"HTS": rep_hts, "PCM": rep_pcm}

# ------------------------------------------------------------
# 10) Merge-based latent encoding for PCM (masked export)
# ------------------------------------------------------------

@torch.no_grad()
def encode_pcm_to_latent_df(
    df_pcm: pd.DataFrame,
    model: nn.Module,
    pack: Dict,
    device: str = "cpu",
    batch_size: int = 8192,
    tau: Optional[float] = None,
    topk_h: Optional[int] = None,
    h_prob_threshold: Optional[float] = None,
    support_policy: Optional[SupportPolicy] = None
) -> Tuple[pd.DataFrame, np.ndarray]:
    """
    Returns q_df columns: ["PCM_ROW","H","PROB"] where PROB = masked q_phi(H|Phi_row).
    If topk_h or threshold provided, additionally sparsifies the exported rows.
    """
    if tau is None:
        tau = float(pack.get("tau_final", 1.0))
    K = int(pack["encoder_cfg"]["K"])

    all_rows = []
    q_blocks = []

    N = len(df_pcm)
    model.eval()

    for start in range(0, N, batch_size):
        end = min(N, start + batch_size)
        phi_b = np.vstack(df_pcm["PHI_IDX"].iloc[start:end].to_numpy()).astype(np.int64)
        phi_b = torch.from_numpy(phi_b).to(device=device, dtype=torch.long)

        logits = model(phi_b)
        if support_policy is None:
            q = softmax_tau(logits, tau)
        else:
            mask = support_policy(logits, phi_b)
            q = masked_softmax(logits, mask, tau)

        q_cpu = q.to("cpu").numpy()
        q_blocks.append(q_cpu)

        B = q_cpu.shape[0]
        if (topk_h is None) and (h_prob_threshold is None):
            pcm_idx = np.repeat(np.arange(start, end), K)
            h_idx   = np.tile(np.arange(K), B)
            prob    = q_cpu.reshape(-1)
            block = np.column_stack([pcm_idx, h_idx, prob])
            all_rows.append(block)
        else:
            for i in range(B):
                probs = q_cpu[i]
                if h_prob_threshold is not None:
                    keep = np.where(probs >= h_prob_threshold)[0]
                    vals = probs[keep]
                else:
                    k = min(int(topk_h), K)
                    keep = np.argpartition(-probs, k-1)[:k]
                    keep = keep[np.argsort(-probs[keep])]
                    vals = probs[keep]
                pcm_row = start + i
                if len(keep) > 0:
                    block = np.column_stack([np.full_like(keep, pcm_row), keep, vals])
                    all_rows.append(block)

    q_latent = np.vstack(q_blocks) if len(q_blocks) > 0 else np.empty((0, K))
    if len(all_rows) == 0:
        q_df = pd.DataFrame(columns=["PCM_ROW", "H", "PROB"])
    else:
        arr = np.vstack(all_rows).astype(np.float64)
        q_df = pd.DataFrame(arr, columns=["PCM_ROW", "H", "PROB"]).astype(
            {"PCM_ROW": int, "H": int, "PROB": float}
        )
    return q_df, q_latent

# ------------------------------------------------------------
# 11) Optional: conditional filtering helper
# ------------------------------------------------------------

def filter_by_conditioned_probability(
    df: pd.DataFrame,
    att_cols: List[str],
    prob_col: str = "Prob_XYZ_fus",
    *,
    abs_threshold: float = 1e-4,
    min_group_sum: float = 1e-15,
    add_column_name: str = "P_cond",
    return_splits: bool = True,
    preserve: str = "group_then_global"  # {"group_then_global", "global_only", "none"}
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    """
    Filter by per-group conditional probability with an absolute threshold.

    Steps:
      1) Compute group sums and conditional prob: P_cond = prob / group_sum
      2) Keep rows with P_cond >= abs_threshold
      3) Renormalize P_cond within each group to sum to 1
      4) Preserve probability mass:
         - "group_then_global": restore original group mass, then global normalize
         - "global_only": only global normalize
         - "none": leave raw group mass as-is (after step 3)
    Returns:
      kept_df, dropped_df (or None if return_splits=False)
    """
    if preserve not in {"group_then_global", "global_only", "none"}:
        raise ValueError("preserve must be one of {'group_then_global','global_only','none'}")

    df = df.copy()

    # 1) Group totals and conditional probability
    group_sum_orig = (
        df.groupby(att_cols, dropna=False)[prob_col]
          .transform("sum")
          .clip(lower=min_group_sum)
    )
    df[add_column_name] = df[prob_col] / group_sum_orig

    # 2) Keep rows above absolute conditional threshold
    keep_mask = df[add_column_name] >= float(abs_threshold)
    kept = df[keep_mask].reset_index(drop=True)
    dropped = df[~keep_mask].reset_index(drop=True) if return_splits else None

    if kept.empty:
        return kept, dropped

    # 3) Renormalize conditional probabilities within each group
    group_sum_kept_cond = (
        kept.groupby(att_cols, dropna=False)[add_column_name]
            .transform("sum")
            .clip(lower=min_group_sum)
    )
    kept[add_column_name] = kept[add_column_name] / group_sum_kept_cond

    # 4) Preserve probability mass as requested
    if preserve == "group_then_global":
        # Restore each group's original mass
        group_mass = (
            df[att_cols + [prob_col]]
            .groupby(att_cols, dropna=False)[prob_col]
            .sum()
            .rename("_GROUP_MASS_ORIG")
            .reset_index()
        )
        kept = kept.merge(group_mass, on=att_cols, how="left")
        kept["_GROUP_MASS_ORIG"] = kept["_GROUP_MASS_ORIG"].fillna(0.0)
        kept[prob_col] = kept[add_column_name] * kept["_GROUP_MASS_ORIG"]
        kept.drop(columns=["_GROUP_MASS_ORIG"], inplace=True)

        # Global normalization (optional but common in probability tables)
        total = float(kept[prob_col].sum())
        if total > 0:
            kept[prob_col] = kept[prob_col] / total

    elif preserve == "global_only":
        total = float(kept[prob_col].sum())
        if total > 0:
            kept[prob_col] = kept[prob_col] / total

    else:
        # "none": leave as-is; prob_col unchanged except for earlier steps
        pass

    return kept, dropped

def filter_by_conditioned_probability2(
    df: pd.DataFrame,
    att_cols: List[str],
    prob_col: str = "Prob_XYZ_fus",
    *,
    abs_threshold: Optional[float] = 1e-4,
    quantile_q: Optional[float] = None,
    topk_per_group: Optional[int] = None,
    min_group_sum: float = 1e-15,
    add_column_name: str = "P_cond",
    return_splits: bool = True,
    preserve: str = "group_then_global"   # {"group_then_global", "global_only", "none"}
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    modes = [abs_threshold is not None, quantile_q is not None, topk_per_group is not None]
    if sum(modes) != 1:
        raise ValueError("Pick exactly ONE of abs_threshold, quantile_q, or topk_per_group.")
    if preserve not in {"group_then_global", "global_only", "none"}:
        raise ValueError("preserve must be one of {'group_then_global','global_only','none'}")

    df = df.copy()
    group_sum_orig = df.groupby(att_cols, dropna=False)[prob_col].transform("sum").clip(lower=min_group_sum)
    df[add_column_name] = df[prob_col] / group_sum_orig

    if abs_threshold is not None:
        keep_mask = df[add_column_name] >= float(abs_threshold)
    elif quantile_q is not None:
        if not (0.0 <= quantile_q <= 1.0):
            raise ValueError("quantile_q must be in [0, 1].")
        q_thr = df.groupby(att_cols, dropna=False)[add_column_name].transform(lambda s: s.quantile(quantile_q))
        keep_mask = df[add_column_name] >= q_thr
    else:
        if topk_per_group <= 0:
            raise ValueError("topk_per_group must be > 0.")
        df["_rank_in_group"] = (
            df.groupby(att_cols, dropna=False)[add_column_name]
              .rank(method="first", ascending=False)
        )
        keep_mask = df["_rank_in_group"] <= float(topk_per_group)
        df.drop(columns=["_rank_in_group"], inplace=True)

    kept    = df[keep_mask].reset_index(drop=True)
    dropped = df[~keep_mask].reset_index(drop=True) if return_splits else None

    if kept.empty:
        return kept, dropped

    group_sum_kept_cond = kept.groupby(att_cols, dropna=False)[add_column_name].transform("sum").clip(lower=min_group_sum)
    kept[add_column_name] = kept[add_column_name] / group_sum_kept_cond

    if preserve == "group_then_global":
        kept = kept.merge(
            df[att_cols + [prob_col]].groupby(att_cols, dropna=False).sum().rename(columns={prob_col: "_GROUP_MASS_ORIG"}).reset_index(),
            on=att_cols, how="left"
        )
        kept["_GROUP_MASS_ORIG"] = kept["_GROUP_MASS_ORIG"].fillna(0.0)
        kept[prob_col] = kept[add_column_name] * kept["_GROUP_MASS_ORIG"]
        kept.drop(columns=["_GROUP_MASS_ORIG"], inplace=True)
        total = float(kept[prob_col].sum())
        if total > 0:
            kept[prob_col] = kept[prob_col] / total
    elif preserve == "global_only":
        total = float(kept[prob_col].sum())
        if total > 0:
            kept[prob_col] = kept[prob_col] / total
    else:
        pass

    return kept, dropped




# Controlled experiment: Stage 1 Data fusion

In [ ]:
# ------------------------------------------------------------
# __main__: Example end-to-end run (adjust paths)
# ------------------------------------------------------------

import pandas as pd
from typing import List, Optional, Tuple

def filter_by_conditioned_probability(
    df: pd.DataFrame,
    att_cols: List[str],
    prob_col: str = "Prob_XYZ_fus",
    *,
    abs_threshold: float = 1e-4,
    add_column_name: str = "P_cond",
    return_splits: bool = True,
    preserve: str = "group_then_global"  # {"group_then_global", "global_only", "none"}
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    """
    Filter by per-group conditional probability with an absolute threshold.

    Steps:
      1) Compute group sums and conditional prob: P_cond = prob / group_sum
      2) Keep rows with P_cond >= abs_threshold
      3) Renormalize P_cond within each group to sum to 1
      4) Preserve probability mass:
         - "group_then_global": restore original group mass, then global normalize
         - "global_only": only global normalize
         - "none": leave raw group mass as-is (after step 3)
    Returns:
      kept_df, dropped_df (or None if return_splits=False)
    """
    if preserve not in {"group_then_global", "global_only", "none"}:
        raise ValueError("preserve must be one of {'group_then_global','global_only','none'}")

    df = df.copy()

    # 1) Group totals and conditional probability
    group_sum_orig = (
        df.groupby(att_cols, dropna=False)[prob_col]
          .transform("sum")
          .clip(lower=1e-15)
    )
    df[add_column_name] = df[prob_col] / group_sum_orig

    # 2) Keep rows above absolute conditional threshold
    keep_mask = df[add_column_name] >= float(abs_threshold)
    kept = df[keep_mask].reset_index(drop=True)
    dropped = df[~keep_mask].reset_index(drop=True) if return_splits else None

    if kept.empty:
        return kept, dropped

    # 3) Renormalize conditional probabilities within each group
    group_sum_kept_cond = (
        kept.groupby(att_cols, dropna=False)[add_column_name]
            .transform("sum")
            .clip(lower=1e-15)
    )
    kept[add_column_name] = kept[add_column_name] / group_sum_kept_cond

    # 4) Preserve probability mass as requested
    if preserve == "group_then_global":
        # Restore each group's original mass
        group_mass = (
            df[att_cols + [prob_col]]
            .groupby(att_cols, dropna=False)[prob_col]
            .sum()
            .rename("_GROUP_MASS_ORIG")
            .reset_index()
        )
        kept = kept.merge(group_mass, on=att_cols, how="left")
        kept["_GROUP_MASS_ORIG"] = kept["_GROUP_MASS_ORIG"].fillna(0.0)
        kept[prob_col] = kept[add_column_name] * kept["_GROUP_MASS_ORIG"]
        kept.drop(columns=["_GROUP_MASS_ORIG"], inplace=True)

        # Global normalization (optional but common in probability tables)
        total = float(kept[prob_col].sum())
        if total > 0:
            kept[prob_col] = kept[prob_col] / total

    elif preserve == "global_only":
        total = float(kept[prob_col].sum())
        if total > 0:
            kept[prob_col] = kept[prob_col] / total

    else:
        # "none": leave as-is; prob_col unchanged except for earlier steps
        pass

    return kept, dropped



# ------------------------------------------------------------
# 11) Validation: Marginal Distribution Comparison
# ------------------------------------------------------------

def validate_z_marginals(df_fused: pd.DataFrame, df_pcm_orig: pd.DataFrame, att_z_cols: list):
    print("\n" + "="*50)
    print("VALIDATION: Marginal Distribution of Z (PCM)")
    print("="*50)

    # 1. Prepare Ground Truth from original PCM
    # We use the 'COUNT' column to represent the true distribution
    gt_z = df_pcm_orig.groupby(att_z_cols)['COUNT'].sum().reset_index()
    gt_z['P_Z_true'] = gt_z['COUNT'] / gt_z['COUNT'].sum()

    # 2. Prepare Fused Marginal
    fus_z = df_fused.groupby(att_z_cols)['Prob_XYZ_fus'].sum().reset_index()
    fus_z.rename(columns={'Prob_XYZ_fus': 'P_Z_fused'}, inplace=True)

    # 3. Merge for comparison
    comparison = pd.merge(gt_z, fus_z, on=att_z_cols, how='outer').fillna(0)

    # 4. Calculate Metrics
    # Total Variation Distance: 0.5 * sum|p - q|
    tvd = 0.5 * np.abs(comparison['P_Z_true'] - comparison['P_Z_fused']).sum()

    # Pearson Correlation
    corr = np.corrcoef(comparison['P_Z_true'], comparison['P_Z_fused'])[0, 1]

    # Print Results
    print(f"Total Unique Z-bins (Trips): {len(comparison)}")
    print(f"Total Variation Distance (TVD): {tvd:.6f}  (Ideal: 0.0)")
    print(f"Pearson Correlation:           {corr:.6f}  (Ideal: 1.0)")

    # Summary of Drift
    max_drift = (comparison['P_Z_true'] - comparison['P_Z_fused']).abs().max()
    print(f"Max Probability Drift:         {max_drift:.6e}")

    if tvd < 0.01:
        print("RESULT: SUCCESS - P(att_Z) is highly preserved.")
    else:
        print("RESULT: WARNING - Significant drift detected in Z-marginal.")



if __name__ == "__main__":

    for ff in [0.1]:#[0.2, 0.4, 0.6]:
        print(f'fraction {ff}...')
        # --- Load data ---
        df_hts = pd.read_csv(os.path.join(path_data, f"val_data_hts_trip_{ff}.csv"))
        df_pcm = pd.read_csv(os.path.join(path_data, "val_data_pcm_trip.csv"))

        # 1) Aggregate PCM counts per (O,D,start,end)
        if "COUNT" not in df_pcm.columns:
            df_pcm["COUNT"] = 1.0
            df_pcm["COUNT"] = df_pcm.groupby(
                ["ORIGIN_SUBZONE","DESTINATION_SUBZONE","TRIP_STARTTIME","TRIP_ENDTIME"]
            )["COUNT"].transform("sum")
            df_pcm = df_pcm.drop_duplicates()

        # 2) Base features for discretization
        df_hts["TRIP_DISTANCE"] = distance(df_hts, ['ORIGIN_SUBZONE_X','DESTINATION_SUBZONE_X','ORIGIN_SUBZONE_Y','DESTINATION_SUBZONE_Y'])
        df_pcm["TRIP_DISTANCE"] = distance(df_pcm, ['ORIGIN_SUBZONE_X','DESTINATION_SUBZONE_X','ORIGIN_SUBZONE_Y','DESTINATION_SUBZONE_Y'])
        df_land_use = land_use(df_hts, df_pcm) #check
        df_mode = mode_share(df_hts, df_pcm) #check
        df_hts = df_hts.merge(df_land_use, on='DESTINATION_SUBZONE').merge(df_mode, on='DESTINATION_SUBZONE')
        df_pcm = df_pcm.merge(df_land_use, on='DESTINATION_SUBZONE').merge(df_mode, on='DESTINATION_SUBZONE')
        # Uniform
        #df_pcm.fillna(0.1, inplace=True)

        # 3) Φ (DISCRETE): fit on HTS and apply to HTS/PCM, then store indices
        nb = 10
        spec = DiscreteSpec(n_dist=nb, n_start=nb, n_end=nb, n_lu_mode_clusters=nb,
                            dist_binning="uniform", time_binning="uniform")

        fitted = fit_semantic_spec_on_hts(df_hts, spec)
        df_hts = apply_semantic_spec(df_hts, fitted)
        df_pcm = apply_semantic_spec(df_pcm, fitted)

        # Use the stored segment sizes so bucket policies work later
        phi_segments_meta = fitted["phi_segments"]
        n_d = phi_segments_meta["n_dist"]
        n_s = phi_segments_meta["n_start"]
        n_e = phi_segments_meta["n_end"]
        n_c = phi_segments_meta["n_cluster"]

        df_hts, phi_dim = build_phi_indices_from_bins(df_hts)
        df_pcm, _       = build_phi_indices_from_bins(df_pcm)

        # 4) Attributes (X,Y,Z) and stringified ATT_X, ATT_Y (for XY classes)
        att_X = ['AGE','GENDER','INCOME']  #
        att_Y = ['TRIP_CNT','TRIP_MAX','TRIP_PURPOSE','TRAVEL_MODE']
        att_Z = ['ORIGIN_SUBZONE','DESTINATION_SUBZONE','TRIP_STARTTIME','TRIP_ENDTIME',
                'ORIGIN_SUBZONE_X','DESTINATION_SUBZONE_X','ORIGIN_SUBZONE_Y','DESTINATION_SUBZONE_Y']

        df_hts = df_hts[att_X + att_Y + att_Z + ['PHI_IDX']]
        df_pcm = df_pcm[att_Z + ['PHI_IDX','COUNT']]

        df_hts['ATT_X'] = df_hts[att_X].astype(str).apply('_'.join, axis=1)
        df_hts['ATT_Y'] = df_hts[att_Y].astype(str).apply('_'.join, axis=1)
        df_hts['ATT_Z'] = df_hts[att_Z].astype(str).apply('_'.join, axis=1)
        df_pcm['ATT_Z'] = df_pcm[att_Z].astype(str).apply('_'.join, axis=1)

        df_hts = reduce_mem_usage(df_hts)
        df_pcm = reduce_mem_usage(df_pcm)

        # 5) Train with sparse per-Φ support (embedding encoder)
        cfg = TrainConfig(
            K=4000,                       # large K is fine with embeddings 2000
            enc_hidden=256,
            enc_layers=2,
            enc_dropout=0.0,
            epochs=5,
            batch_size_hts=8192,
            batch_size_pcm=16384,
            lr=2e-3,
            wd=1e-4,
            device=("cuda" if torch.cuda.is_available() else "cpu"),
            use_normalized_objective=True,
            rebuild_every=1,
            progress=True,
            tau_start=0.01,
            tau_end=0.01,
            tau_schedule="cosine",
            topk_h=10,
            use_bucket_candidates=False,
            # bucket_topk=8,
            # refresh_bucket_every=2
        )

        out = train_encoder_only_fusion(df_hts, df_pcm, cfg, phi_segments_meta)
        model, pack = out["model"], out["pack"]

        # 6) Build the decoder table once after training
        decoder_df = pack_to_dataframe(pack).copy()   # ["ATT_X","ATT_Y","H","PROB"]
        # Threshold + renormalize within latent state H
        decoder_df = decoder_df[decoder_df['PROB'] > 1e-4] #1e-3
        sum_h = decoder_df.groupby('H', observed=True)['PROB'].transform('sum')
        decoder_df = decoder_df[sum_h > 0]
        decoder_df['PROB'] = decoder_df['PROB'] / sum_h[sum_h > 0]
        decoder_df.rename(columns={'PROB': 'PROB_XY|H'}, inplace=True)

        # 7) Prepare the SAME support policy for inference/merging
        tau_eval = float(pack.get("tau_final", 1.0))

        if cfg.use_bucket_candidates:
            phi_hts = torch.from_numpy(np.vstack(df_hts["PHI_IDX"].to_numpy()).astype(np.int64))
            dummy_y = torch.zeros((phi_hts.shape[0],), dtype=torch.long)
            hts_phi_loader = DataLoader(TensorDataset(phi_hts, dummy_y),
                                        batch_size=cfg.batch_size_hts, shuffle=False, num_workers=0)
            cand_map = build_phi_bucket_candidates_embed(
                model, hts_phi_loader, cfg.K, tau_eval,
                topk=cfg.bucket_topk, device=cfg.device
            )
            support_policy_infer = bucket_support_policy_embed(cand_map)
        else:
            support_policy_infer = topk_support_policy(cfg.topk_h)

        # 8) Encode PCM to latent with the same support policy
        q_df, q_latent = encode_pcm_to_latent_df(
            df_pcm=df_pcm, model=model, pack=pack, device=cfg.device, batch_size=8192,
            tau=tau_eval,
            topk_h=2,                      # optional export sparsification
            h_prob_threshold=None,
            support_policy=support_policy_infer
        )
        q_df.rename(columns={'PROB': 'PROB_H'}, inplace=True)

        # 9) Merge for fused probabilities

        # 1. Prepare PCM and HTS data components
        df_fus_pcm = df_pcm.copy()
        df_fus_pcm['P_Z'] = df_fus_pcm['COUNT'] / df_fus_pcm['COUNT'].sum()
        df_fus_pcm['PCM_ROW'] = df_fus_pcm.index
        df_fus_pcm = df_fus_pcm.merge(q_df, on='PCM_ROW') # [ATT_Z, P_Z, H, PROB_H]

        df_hts_attrs = df_hts[att_X + att_Y + ['ATT_X', 'ATT_Y']].drop_duplicates()
        df_fus_hts = decoder_df.merge(df_hts_attrs, on=['ATT_X', 'ATT_Y']) # [ATT_X, ATT_Y, H, PROB_XY|H]

        # 2. Identify Cohorts
        att_cols = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX', 'TRIP_CNT']
        df_cohorts = df_fus_hts[att_cols].drop_duplicates().sort_values(by=att_cols)

        fus_parts = []

        # 3. Cohort Iterator Loop
        for age, gender, income, trip_max, trip_cnt in df_cohorts.itertuples(index=False):
            # Filter Person-side
            mask = (
                (df_fus_hts['AGE'] == age) & (df_fus_hts['GENDER'] == gender) & (df_fus_hts['INCOME'] == income) &
                (df_fus_hts['TRIP_MAX'] == trip_max) & (df_fus_hts['TRIP_CNT'] == trip_cnt)
            )
            df_hts_part = df_fus_hts.loc[mask].copy()
            if df_hts_part.empty: continue

            # Merge with PCM-side via Bridge H
            # This result is P(XY, Z, H)
            df_part = df_hts_part.merge(df_fus_pcm, on='H')

            # Raw Probability: P(XY|H) * P(H|Z) * P(Z)
            df_part['Prob_XYZ_fus'] = df_part['PROB_XY|H'] * df_part['PROB_H'] * df_part['P_Z']

            # Immediate Aggregation to (X, Y, Z) to save RAM
            # We MUST keep att_Z here to perform the global correction later
            df_part = df_part.groupby(att_X + att_Y + att_Z)['Prob_XYZ_fus'].sum().reset_index()

            # Light Filtering (Conditional threshold)
            # Note: We can't fully re-normalize yet because we don't have the other cohorts
            df_part = df_part[df_part['Prob_XYZ_fus'] > 1e-9]

            fus_parts.append(df_part)
            print(f"AGE={age}, GENDER={gender}, INCOME={income}, TRIP_MAX={trip_max}, TRIP_CNT={trip_cnt}")

            # Cleanup
            del df_hts_part, df_part
            # gc.collect() # Uncomment if RAM is extremely tight

        # 4. Global Z-Preservation Correction
        print("Merging cohorts and applying Z-preservation...")
        df_fusion = pd.concat(fus_parts, ignore_index=True)
        del fus_parts

        # Collapse any duplicates that appeared in different cohorts
        df_fusion = df_fusion.groupby(att_X + att_Y + att_Z)['Prob_XYZ_fus'].sum().reset_index()

        # --- THE Z-PRESERVATION KEY ---
        # Calculate the current marginal mass for each trip type Z
        z_mass_current = df_fusion.groupby(att_Z)['Prob_XYZ_fus'].transform('sum')

        # Calculate the target marginal mass from original PCM
        # We use the original df_pcm to get the 'COUNT' distribution
        df_pcm_targets = df_pcm.groupby(att_Z)['COUNT'].sum().reset_index()
        df_pcm_targets['target_P_Z'] = df_pcm_targets['COUNT'] / df_pcm_targets['COUNT'].sum()

        # Merge target mass onto our fused data
        df_fusion = df_fusion.merge(df_pcm_targets[att_Z + ['target_P_Z']], on=att_Z, how='left')

        # Scale probabilities so that sum(Prob) for each Z equals target_P_Z
        df_fusion['Prob_XYZ_fus'] *= (df_fusion['target_P_Z'] / z_mass_current.replace(0, 1))

        # Final clean up
        df_fusion.drop(columns=['target_P_Z'], inplace=True)
        df_fusion['Prob_XYZ_fus'] /= df_fusion['Prob_XYZ_fus'].sum()

        print(df_fusion['Prob_XYZ_fus'].sum())
        print(df_fusion.info())
        df_fusion.to_csv(os.path.join(path_result, f'val_sim_trip_proposed_{ff}.csv'), index=False)

        # Run the check
        validate_z_marginals(df_fusion, df_pcm, att_Z)



fraction 0.1...
[decoder] pass chunk 0
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 001] tau=0.0100  L_hts=0.514477  total=0.514477
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 002] tau=0.0100  L_hts=0.439405  total=0.439405
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 003] tau=0.0100  L_hts=0.415047  total=0.415047
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 004] tau=0.0100  L_hts=0.389750  total=0.389750
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 005] tau=0.0100  L_hts=0.377195  total=0.377195
[decoder] pass chunk 0
[FINAL] K=4000  tau=0.0100  L_hts=0.370722
AGE=0, GENDER=0, INCOME=3, TRIP_MAX=2, TRIP_CNT=1
AGE=0, GENDER=0, INCOME=3, TRIP_MAX=2, TRIP_CNT=2
AGE=0, GENDER=0, INCOME=3, TRIP_MAX=3, TRIP_CNT=1
AGE=0, GENDER=0, INCOME=3, TRIP_MAX=3, TRIP_CNT=2
AGE=0, GENDER=0, INCOME=3, TRIP_MAX=3, TRIP_CNT=3
AGE=0, GENDER=1, INCOME=3, TRIP_MAX=2, TRIP_CNT=1
AGE=0, GENDER=1, INCOME=3, TRIP_MAX=2, TRIP_CNT=2
AGE=0, GENDER=1, INCOME=3, TRIP_MAX=3, T

#Validation: Stage 2 Tour generation

In [ ]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.cluster import Birch
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.special import rel_entr, kl_div
import math
import time
import warnings
warnings.filterwarnings('ignore')

TRIP_MAX_ = 7

# ============================================================
# Memory helpers
# ============================================================

def _downcast_int(s: pd.Series):
    if s.isnull().any():
        return s
    i32 = s.astype(np.int64)
    if i32.min() >= np.iinfo(np.int8).min and i32.max() <= np.iinfo(np.int8).max:
        return i32.astype(np.int8)
    if i32.min() >= np.iinfo(np.int16).min and i32.max() <= np.iinfo(np.int16).max:
        return i32.astype(np.int16)
    if i32.min() >= np.iinfo(np.int32).min and i32.max() <= np.iinfo(np.int32).max:
        return i32.astype(np.int32)
    return i32

def _downcast_float(s: pd.Series):
    return s.astype(np.float32)

def reduce_mem_usage(df: pd.DataFrame, use_cats: bool = True):
    for col in df.columns:
        if pd.api.types.is_integer_dtype(df[col]):
            df[col] = _downcast_int(df[col])
        elif pd.api.types.is_float_dtype(df[col]):
            df[col] = _downcast_float(df[col])
        elif use_cats and (df[col].nunique(dropna=False) / max(1, len(df[col])) < 0.5):
            df[col] = df[col].astype('category')
    return df

def as_codes(df: pd.DataFrame, cols):
    for c in cols:
        if not pd.api.types.is_categorical_dtype(df[c]):
            df[c] = df[c].astype('category')
        df[c] = df[c].cat.codes.astype(np.int32)
    return df

# ============================================================
# Math helpers
# ============================================================

def js_div(p, q):
    h = 0.5 * np.add(p, q)
    return 0.5 * rel_entr(p, h) + 0.5 * rel_entr(q, h)

# ============================================================
# Column ops
# ============================================================

def _renmap(i: int):
    return {
        'TRIP_PURPOSE':          f'TRIP_PURPOSE_{i}',
        'TRAVEL_MODE':           f'TRAVEL_MODE_{i}',
        'ORIGIN_SUBZONE':        f'ORIGIN_SUBZONE_{i}',
        'DESTINATION_SUBZONE':   f'DESTINATION_SUBZONE_{i}',
        'TRIP_STARTTIME':        f'TRIP_STARTTIME_{i}',
        'TRIP_ENDTIME':          f'TRIP_ENDTIME_{i}',
        'ORIGIN_SUBZONE_X':      f'ORIGIN_SUBZONE_X_{i}',
        'ORIGIN_SUBZONE_Y':      f'ORIGIN_SUBZONE_Y_{i}',
        'DESTINATION_SUBZONE_X': f'DESTINATION_SUBZONE_X_{i}',
        'DESTINATION_SUBZONE_Y': f'DESTINATION_SUBZONE_Y_{i}',
        'Prob_XYZ_fus':          f'Prob_{i}',
    }

def rename_columns(df: pd.DataFrame, i: int):
    df.rename(columns=_renmap(i), inplace=True)

# ============================================================
# Sampling utils
# ============================================================

def _prob_sample_indices(prob: np.ndarray, N: int) -> np.ndarray:
    prob = np.asarray(prob, dtype=np.float64)
    prob_sum = prob.sum()
    if prob_sum <= 0:
        return np.array([], dtype=np.int64)
    prob = prob / prob_sum
    return np.random.choice(len(prob), size=N, replace=True, p=prob)

# ============================================================
# Core pipeline
# ============================================================

def simulate_trips(df_fus: pd.DataFrame, i: int, N: int) -> pd.DataFrame:
    att_X = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX']
    att_Z = [f'TRIP_PURPOSE_{i}', f'TRAVEL_MODE_{i}',
             f'ORIGIN_SUBZONE_{i}', f'ORIGIN_SUBZONE_X_{i}', f'ORIGIN_SUBZONE_Y_{i}',
             f'DESTINATION_SUBZONE_{i}', f'DESTINATION_SUBZONE_X_{i}', f'DESTINATION_SUBZONE_Y_{i}',
             f'TRIP_STARTTIME_{i}', f'TRIP_ENDTIME_{i}']

    df_sim = df_fus[df_fus['TRIP_CNT'] == i].drop(columns=['TRIP_CNT'])
    rename_columns(df_sim, i)
    df_sim.rename(columns={f'Prob_{i}': 'Prob'}, inplace=True)
    df_sim = df_sim[att_X + att_Z + ['Prob']].copy()
    reduce_mem_usage(df_sim)

    idx = _prob_sample_indices(df_sim['Prob'].to_numpy(), N)
    if len(idx) == 0:
        return df_sim.iloc[0:0][att_X + att_Z]

    out = df_sim.iloc[idx][att_X + att_Z].reset_index(drop=True)
    return reduce_mem_usage(out)


def simulate_tours(df_tours: pd.DataFrame, order: int, N: int) -> pd.DataFrame:
    att_X = ['ID', 'AGE', 'GENDER', 'INCOME', 'TRIP_MAX']
    att_Z = []
    for i in range(1, order + 1):
        att_Z.extend([f'TRIP_PURPOSE_{i}', f'TRAVEL_MODE_{i}',
                      f'ORIGIN_SUBZONE_{i}', f'ORIGIN_SUBZONE_X_{i}', f'ORIGIN_SUBZONE_Y_{i}',
                      f'DESTINATION_SUBZONE_{i}', f'DESTINATION_SUBZONE_X_{i}', f'DESTINATION_SUBZONE_Y_{i}',
                      f'TRIP_STARTTIME_{i}', f'TRIP_ENDTIME_{i}'])

    probs = df_tours['Prob'].to_numpy(dtype=np.float64)
    probs = probs / probs.sum() if probs.sum() > 0 else probs
    idx = _prob_sample_indices(probs, N)
    if len(idx) == 0:
        return df_tours.iloc[0:0][att_X + att_Z]

    out = df_tours.iloc[idx][att_X + att_Z].reset_index(drop=True)
    return reduce_mem_usage(out)


def next_start_times(df_hts_: pd.DataFrame) -> pd.DataFrame:
    """
    Compute P(TRIP_STARTTIME | TRIP_ENDTIME, TRIP_PURPOSE) from HTS activity records.
    """
    t = df_hts_[['ACTIVITY_STARTTIME', 'TRIP_PURPOSE', 'ACTIVITY_DURATION']].copy()
    t['TRIP_STARTTIME'] = t['ACTIVITY_STARTTIME'] + t['ACTIVITY_DURATION']
    t = t.rename(columns={'ACTIVITY_STARTTIME': 'TRIP_ENDTIME'}).drop(columns=['ACTIVITY_DURATION'])

    t['Numbers'] = 1.0
    grp_keys = ['TRIP_ENDTIME', 'TRIP_PURPOSE', 'TRIP_STARTTIME']
    agg = t.groupby(grp_keys, observed=True)['Numbers'].sum().astype(np.float64).reset_index()
    agg['Prob'] = agg['Numbers'] / agg['Numbers'].sum()
    agg.drop(columns=['Numbers'], inplace=True)

    base_keys = ['TRIP_ENDTIME', 'TRIP_PURPOSE']
    agg['Prob_base'] = agg.groupby(base_keys, observed=True)['Prob'].transform('sum')
    agg['Prob_cond'] = (agg['Prob'] / agg['Prob_base']).astype(np.float32)
    agg = agg.drop(columns=['Prob_base', 'Prob'])

    reduce_mem_usage(agg)
    return agg.dropna()


def _prep_ith_trips(df_fus: pd.DataFrame, i: int) -> pd.DataFrame:
    """
    Compute P(dest, endtime, mode, purpose | age, gender, income, trip_max, origin, starttime)
    from the fused OD matrix. Pre-building this table once per trip index avoids
    recomputing it inside the chunk loop.
    """
    cols_keep = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX', 'TRIP_CNT',
                 'TRIP_PURPOSE', 'TRAVEL_MODE',
                 'ORIGIN_SUBZONE', 'ORIGIN_SUBZONE_X', 'ORIGIN_SUBZONE_Y',
                 'DESTINATION_SUBZONE', 'DESTINATION_SUBZONE_X', 'DESTINATION_SUBZONE_Y',
                 'TRIP_STARTTIME', 'TRIP_ENDTIME', 'Prob_XYZ_fus']
    ith = df_fus.loc[df_fus['TRIP_CNT'] == i, cols_keep].copy()
    ith.drop(columns=['TRIP_CNT'], inplace=True)
    rename_columns(ith, i)

    den_keys = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX',
                f'ORIGIN_SUBZONE_{i}', f'TRIP_STARTTIME_{i}']
    ith['Prob_den'] = ith.groupby(den_keys, observed=True)[f'Prob_{i}'].transform('sum')
    # Guard against zero denominator
    ith[f'Prob_{i}'] = np.where(
        ith['Prob_den'] > 0,
        (ith[f'Prob_{i}'] / ith['Prob_den']).astype(np.float32),
        0.0
    )
    ith = ith[ith[f'Prob_{i}'] > 0].copy()
    ith.drop(columns=['Prob_den'], inplace=True)

    return reduce_mem_usage(ith)


# ============================================================
# Fallback: OD-preserving draw when the strict join finds no match
# ============================================================

def _fallback_trip_sample(df_unmatched: pd.DataFrame,
                           df_fus: pd.DataFrame,
                           i: int) -> pd.DataFrame:
    """
    For individuals that could not be matched via the strict
    (AGE, GENDER, INCOME, TRIP_MAX, ORIGIN, STARTTIME) join, draw
    trip i from the OD matrix using a cascading relaxation strategy
    that preserves the OD distribution at each level.

    Relaxation ladder
    -----------------
    Level 1  Full demographic + spatial match, time relaxed
             Condition on (AGE, GENDER, INCOME, TRIP_MAX, ORIGIN_SUBZONE)
    Level 2  Spatial match only, demographics relaxed
             Condition on (ORIGIN_SUBZONE) only
    Level 3  Unconditional draw from full OD matrix for this trip index
             Last resort — preserves global OD marginals

    In all levels the draw is weighted by Prob_XYZ_fus so the OD
    pair distribution from the source matrix is respected.

    Parameters
    ----------
    df_unmatched : rows from df_tours_ that found no match in concat_trips
    df_fus       : original fused OD table (pre-rename, with Prob_XYZ_fus)
    i            : current trip index

    Returns
    -------
    DataFrame with the same schema as df_unmatched plus trip-i columns appended
    """
    trip_cols = [
        f'TRIP_PURPOSE_{i}', f'TRAVEL_MODE_{i}',
        f'ORIGIN_SUBZONE_{i}', f'ORIGIN_SUBZONE_X_{i}', f'ORIGIN_SUBZONE_Y_{i}',
        f'DESTINATION_SUBZONE_{i}', f'DESTINATION_SUBZONE_X_{i}', f'DESTINATION_SUBZONE_Y_{i}',
        f'TRIP_STARTTIME_{i}', f'TRIP_ENDTIME_{i}',
    ]

    # Prepare a renamed view of df_fus for trip i (do NOT rename in-place)
    fus_i = df_fus.loc[df_fus['TRIP_CNT'] == i].copy()
    fus_i.drop(columns=['TRIP_CNT'], inplace=True)
    rename_columns(fus_i, i)
    fus_i.rename(columns={f'Prob_{i}': '_prob'}, inplace=True)
    fus_i['_prob'] = fus_i['_prob'].astype(np.float64)

    origin_col = f'ORIGIN_SUBZONE_{i}'
    prev_dest  = f'DESTINATION_SUBZONE_{i - 1}'
    prev_age   = 'AGE'
    prev_gnd   = 'GENDER'
    prev_inc   = 'INCOME'
    prev_tmax  = 'TRIP_MAX'

    rows_out = []
    fallback_levels = {1: 0, 2: 0, 3: 0}

    for _, row in df_unmatched.iterrows():
        prev_zone = row[prev_dest]
        age       = row[prev_age]
        gender    = row[prev_gnd]
        income    = row[prev_inc]
        tmax      = row[prev_tmax]

        # --- Level 1: full demographics + origin, relax time ---------------
        pool = fus_i[
            (fus_i[origin_col]  == prev_zone) &
            (fus_i[prev_age]    == age)        &
            (fus_i[prev_gnd]    == gender)     &
            (fus_i[prev_inc]    == income)     &
            (fus_i[prev_tmax]   == tmax)
        ]

        if len(pool) == 0:
            # --- Level 2: spatial only, relax demographics -----------------
            pool = fus_i[fus_i[origin_col] == prev_zone]

        if len(pool) == 0:
            # --- Level 3: unconditional draw from full OD matrix -----------
            pool = fus_i
            fallback_levels[3] += 1
        elif pool is fus_i:
            fallback_levels[2] += 1
        else:
            fallback_levels[1] += 1

        pool = pool.copy()
        pool['_prob'] = pool['_prob'] / pool['_prob'].sum()
        idx = np.random.choice(len(pool), p=pool['_prob'].to_numpy())
        drawn = pool.iloc[idx][trip_cols].to_dict()

        combined = row.to_dict()
        combined.update(drawn)
        rows_out.append(combined)

    if rows_out:
        df_filled = pd.DataFrame(rows_out)
    else:
        df_filled = df_unmatched.copy()
        for c in trip_cols:
            df_filled[c] = np.nan

    return df_filled, fallback_levels


# ============================================================
# concat_trips — strict join + OD-preserving fallback
# ============================================================

def concat_trips(df_tours: pd.DataFrame,
                 df_ith_trips: pd.DataFrame,
                 df_next_start_times: pd.DataFrame,
                 i: int,
                 df_fus: pd.DataFrame = None):
    """
    Attach trip i to partial tours.

    Primary path  : strict join on (AGE, GENDER, INCOME, TRIP_MAX, ORIGIN, STARTTIME)
                    after conditioning on P(next_start | prev_end, prev_purpose).
    Fallback path : OD-preserving draw via _fallback_trip_sample() for any
                    individual that found no match in the primary join.

    Returns
    -------
    df_tours_complete   : rows where TRIP_MAX == i
    df_tours_incomplete : rows where TRIP_MAX >  i
    stats               : dict with matched / fallback counts per level
    """
    tours = df_tours.copy()
    tours['PRE_SUBZONE'] = tours[f'DESTINATION_SUBZONE_{i-1}']
    tours['PRE_PURPOSE'] = tours[f'TRIP_PURPOSE_{i-1}']
    tours['PRE_ENDTIME'] = tours[f'TRIP_ENDTIME_{i-1}']

    nst = df_next_start_times.rename(columns={
        'TRIP_PURPOSE':  'PRE_PURPOSE',
        'TRIP_ENDTIME':  'PRE_ENDTIME',
        'TRIP_STARTTIME': 'PRE_STARTTIME',
    }, errors='ignore')

    ith = df_ith_trips.copy()
    ith['PRE_SUBZONE']   = ith[f'ORIGIN_SUBZONE_{i}']
    ith['PRE_STARTTIME'] = ith[f'TRIP_STARTTIME_{i}']

    # ── Primary join ──────────────────────────────────────────────────────
    tours_joined = tours.merge(
        nst[['PRE_PURPOSE', 'PRE_ENDTIME', 'PRE_STARTTIME', 'Prob_cond']],
        on=['PRE_PURPOSE', 'PRE_ENDTIME'],
        how='left',
        copy=False,
    )

    join_keys = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX', 'PRE_SUBZONE', 'PRE_STARTTIME']
    tours_joined = tours_joined.merge(
        ith[join_keys + [
            f'TRIP_PURPOSE_{i}', f'TRAVEL_MODE_{i}',
            f'ORIGIN_SUBZONE_{i}', f'ORIGIN_SUBZONE_X_{i}', f'ORIGIN_SUBZONE_Y_{i}',
            f'DESTINATION_SUBZONE_{i}', f'DESTINATION_SUBZONE_X_{i}', f'DESTINATION_SUBZONE_Y_{i}',
            f'TRIP_STARTTIME_{i}', f'TRIP_ENDTIME_{i}', f'Prob_{i}']],
        on=join_keys,
        how='inner',
        copy=False,
    )

    matched_ids  = set(tours_joined['ID'].unique()) if len(tours_joined) > 0 else set()
    all_ids      = set(df_tours['ID'].unique())
    unmatched_ids = all_ids - matched_ids

    stats = {'matched': len(matched_ids), 'fallback_l1': 0,
             'fallback_l2': 0, 'fallback_l3': 0}

    # ── Fallback for unmatched individuals ───────────────────────────────
    if unmatched_ids and df_fus is not None:
        df_unmatched = tours[tours['ID'].isin(unmatched_ids)].copy()
        df_unmatched = df_unmatched.drop(
            columns=['PRE_SUBZONE', 'PRE_PURPOSE', 'PRE_ENDTIME'], errors='ignore'
        )

        df_filled, fb_levels = _fallback_trip_sample(df_unmatched, df_fus, i)
        stats['fallback_l1'] += fb_levels[1]
        stats['fallback_l2'] += fb_levels[2]
        stats['fallback_l3'] += fb_levels[3]

        # Assign nominal probability for filled rows so Prob column exists
        if len(tours_joined) > 0 and f'Prob_{i}' in tours_joined.columns:
            nominal_prob = float(tours_joined[f'Prob_{i}'].median())
        else:
            nominal_prob = 1e-6
        df_filled['Prob'] = nominal_prob

        # Finalise primary matched rows
        tours_joined['Prob'] = (
            tours_joined[f'Prob_{i}'] * tours_joined['Prob_cond']
        ).astype(np.float32)
        tours_joined.drop(
            columns=[f'Prob_{i}', 'Prob_cond',
                     'PRE_SUBZONE', 'PRE_PURPOSE', 'PRE_ENDTIME', 'PRE_STARTTIME'],
            inplace=True, errors='ignore'
        )

        # Align columns before concat (filled rows won't have Prob_cond etc.)
        shared_cols = [c for c in tours_joined.columns if c in df_filled.columns]
        tours_all = pd.concat(
            [tours_joined[shared_cols], df_filled[shared_cols]],
            ignore_index=True
        )
    else:
        # Normal path — no fallback needed
        tours_joined['Prob'] = (
            tours_joined[f'Prob_{i}'] * tours_joined['Prob_cond']
        ).astype(np.float32)
        tours_joined.drop(
            columns=[f'Prob_{i}', 'Prob_cond',
                     'PRE_SUBZONE', 'PRE_PURPOSE', 'PRE_ENDTIME', 'PRE_STARTTIME'],
            inplace=True, errors='ignore'
        )
        tours_all = tours_joined

    # ── Split complete vs incomplete ──────────────────────────────────────
    mask_done   = tours_all['TRIP_MAX'] == i
    mask_home   = mask_done & (tours_all[f'TRIP_PURPOSE_{i}'] == 0)

    complete_home = tours_all.loc[mask_home].copy()
    complete_home[f'DESTINATION_SUBZONE_{i}']   = complete_home['ORIGIN_SUBZONE_1']
    complete_home[f'DESTINATION_SUBZONE_X_{i}'] = complete_home['ORIGIN_SUBZONE_X_1']
    complete_home[f'DESTINATION_SUBZONE_Y_{i}'] = complete_home['ORIGIN_SUBZONE_Y_1']

    complete_nohome = tours_all.loc[mask_done & ~mask_home]
    df_tours_complete   = pd.concat([complete_home, complete_nohome], ignore_index=True)
    df_tours_incomplete = tours_all.loc[~mask_done].copy()

    return (reduce_mem_usage(df_tours_complete),
            reduce_mem_usage(df_tours_incomplete),
            stats)


# ============================================================
# Main generation function
# ============================================================

def generate_tours(df_fus: pd.DataFrame,
                   df_next_start_times: pd.DataFrame,
                   N: int,
                   verbose: bool = True) -> pd.DataFrame:
    """
    Generate N synthetic tour records.

    Changes vs original
    -------------------
    * concat_trips now returns (complete, incomplete, stats) — call sites updated.
    * Unmatched individuals are filled via OD-preserving fallback rather than dropped.
    * _prep_ith_trips is pre-computed once before the chunk loop (not rebuilt per chunk).
    * verbose flag prints per-trip match / fallback statistics.

    Notes on scale
    --------------
    scale > 1 serves two distinct purposes and should be kept:
      1. Seed diversity  : scale*N individuals are seeded so the chunk pool
                           starts with enough variety across cohorts.
      2. Pool diversity  : simulate_tours re-samples scale*N rows after each
                           inner join step so the incomplete pool does not
                           collapse to a handful of rows and produce near-clone tours.
    The fallback fixes zero-row failures; scale fixes low-diversity. They are
    complementary and both are needed.
    """
    scale = 2 #5

    # ── Seed tours from TRIP_CNT == 1 ────────────────────────────────────
    df_tours_seed = df_fus[df_fus['TRIP_CNT'] == 1]
    df_tours = simulate_trips(df_tours_seed, 1, int(scale * N)).reset_index(drop=True)
    df_tours = df_tours.reset_index()
    df_tours['ID'] = df_tours['index'].astype(np.int64)
    df_tours.drop(columns=['index'], inplace=True)

    # Trip-max proportions per (AGE, GENDER, INCOME, TRIP_MAX)
    df_tm = df_tours[['AGE', 'GENDER', 'INCOME', 'TRIP_MAX']].copy()
    df_tm['Numbers'] = 1.0
    df_tm = (df_tm.groupby(['AGE', 'GENDER', 'INCOME', 'TRIP_MAX'], observed=True)
             ['Numbers'].sum().reset_index())
    df_tm['Pop'] = (df_tm['Numbers'] / df_tm['Numbers'].sum()).astype(np.float32)
    df_tm.drop(columns=['Numbers'], inplace=True)
    trip_max = {(r.AGE, r.GENDER, r.INCOME, r.TRIP_MAX): r.Pop
                for r in df_tm.itertuples(index=False)}

    # ── Pre-build conditional trip tables once (avoids re-compute per chunk) ──
    if verbose:
        print("Pre-building conditional trip tables for trips 2 …", TRIP_MAX_)
    cond_tables = {i: _prep_ith_trips(df_fus, i) for i in range(2, TRIP_MAX_ + 1)}
    df_next_start_times = reduce_mem_usage(df_next_start_times.copy())

    # ── Chunk processing ──────────────────────────────────────────────────
    chunk_size  = 10_000
    Z           = (len(df_tours) + chunk_size - 1) // chunk_size
    out_chunks  = []
    total_stats = {'matched': 0, 'fallback_l1': 0, 'fallback_l2': 0, 'fallback_l3': 0}

    if verbose:
        print(f"Processing {Z} chunks of ≤{chunk_size} rows …")

    for z in range(Z):
        df_tours_ = df_tours.iloc[z * chunk_size:(z + 1) * chunk_size].copy()
        df_tours_['Prob'] = 1.0
        df_result = []

        for i in range(2, TRIP_MAX_ + 1):
            df_ith_trips = cond_tables[i]

            df_tours_complete, df_tours_incomplete, stats = concat_trips(
                df_tours_, df_ith_trips, df_next_start_times, i, df_fus=df_fus
            )

            for k in total_stats:
                total_stats[k] += stats.get(k, 0)

            if verbose:
                print(f"  chunk {z:3d} | trip {i}: "
                      f"matched={stats['matched']}, "
                      f"fb_l1={stats['fallback_l1']}, "
                      f"fb_l2={stats['fallback_l2']}, "
                      f"fb_l3={stats['fallback_l3']}, "
                      f"complete={len(df_tours_complete)}, "
                      f"incomplete={len(df_tours_incomplete)}")

            if len(df_tours_complete) > 0:
                df_result.append(
                    simulate_tours(df_tours_complete, i, int(scale * N))
                )

            if len(df_tours_incomplete) > 0:
                df_tours_ = simulate_tours(df_tours_incomplete, i, int(scale * N))
            else:
                break

        if df_result:
            res = pd.concat(df_result, ignore_index=True)
            res = res.drop_duplicates(subset=['ID'], keep='first')
            out_chunks.append(res)

        if verbose:
            print(f"  chunk {z} done  ({len(out_chunks[-1]) if out_chunks else 0} unique tours)")

    if not out_chunks:
        return pd.DataFrame(columns=['ID', 'AGE', 'GENDER', 'INCOME', 'TRIP_MAX'])

    df_result_ = pd.concat(out_chunks, ignore_index=True)
    df_result_ = df_result_.drop_duplicates(subset=['ID'], keep='first')

    # ── Final cohort sampling to match TRIP_MAX proportions ──────────────
    final_chunks = []
    for (age, gender, income, tmax), pop in trip_max.items():
        df_tmp = df_result_[
            (df_result_['TRIP_MAX'] == tmax)   &
            (df_result_['AGE']      == age)    &
            (df_result_['INCOME']   == income) &
            (df_result_['GENDER']   == gender)
        ]
        target = int(N * pop)
        n2 = len(df_tmp)
        if n2 == 0:
            if verbose:
                print(f"  WARNING: no tours for cohort "
                      f"(AGE={age}, GENDER={gender}, INCOME={income}, TRIP_MAX={tmax})")
            continue
        if n2 > target and target > 0:
            df_tmp = df_tmp.sample(n=target, replace=False, random_state=None)
        final_chunks.append(df_tmp)

    df_result_final = (pd.concat(final_chunks, ignore_index=True)
                       if final_chunks else df_result_.iloc[0:0])

    # Assign globally unique sequential ID
    df_result_final = df_result_final.reset_index(drop=True)
    df_result_final['ID'] = df_result_final.index

    if verbose:
        total_fb = total_stats['fallback_l1'] + total_stats['fallback_l2'] + total_stats['fallback_l3']
        total_all = total_stats['matched'] + total_fb
        pct = 100 * total_fb / max(total_all, 1)
        print(f"\n── Generation complete ──────────────────────────────────")
        print(f"   Output rows : {len(df_result_final)}")
        print(f"   Matched     : {total_stats['matched']}")
        print(f"   Fallback L1 : {total_stats['fallback_l1']}  "
              f"(demo+spatial, time relaxed)")
        print(f"   Fallback L2 : {total_stats['fallback_l2']}  "
              f"(spatial only)")
        print(f"   Fallback L3 : {total_stats['fallback_l3']}  "
              f"(unconditional OD draw)")
        print(f"   Total fallback: {total_fb} / {total_all}  ({pct:.1f}%)")

    return reduce_mem_usage(df_result_final)


# ============================================================
# Entry point
# ============================================================

if __name__ == '__main__':
    #ff = 0.6
    for ff in [0.1]:#[0.2, 0.4, 0.6]:
        df_hts = pd.read_csv(path_data + f'val_data_hts_trip_{ff}.csv')

        start_time = time.time()

        df_fus = pd.read_csv(path_result + f'val_sim_trip_proposed_{ff}.csv')
        df_fus = reduce_mem_usage(df_fus)
        print('df_fus length:', len(df_fus))

        df_next_start_times = next_start_times(df_hts)

        NN = 16_000 #50_000 15358
        df_tours = generate_tours(df_fus, df_next_start_times, NN, verbose=True)
        print(df_tours.info())
        df_tours.to_csv(path_result + f'val_sim_tour_proposed_{ff}.csv', index=False)

        print('Time taken: %s seconds' % (time.time() - start_time))
        print('Done!')

df_fus length: 1049964
Pre-building conditional trip tables for trips 2 … 7
Processing 4 chunks of ≤10000 rows …
  chunk   0 | trip 2: matched=9507, fb_l1=493, fb_l2=0, fb_l3=0, complete=535603, incomplete=34176
  chunk   0 | trip 3: matched=2282, fb_l1=1014, fb_l2=0, fb_l3=0, complete=306130, incomplete=123304
  chunk   0 | trip 4: matched=1005, fb_l1=616, fb_l2=0, fb_l3=0, complete=232264, incomplete=67020
  chunk   0 | trip 5: matched=402, fb_l1=469, fb_l2=0, fb_l3=0, complete=124730, incomplete=101379
  chunk   0 | trip 6: matched=169, fb_l1=256, fb_l2=0, fb_l3=0, complete=100660, incomplete=8519
  chunk   0 | trip 7: matched=39, fb_l1=249, fb_l2=0, fb_l3=0, complete=93716, incomplete=0
  chunk 0 done  (8970 unique tours)
  chunk   1 | trip 2: matched=9502, fb_l1=498, fb_l2=0, fb_l3=0, complete=534117, incomplete=34396
  chunk   1 | trip 3: matched=2303, fb_l1=892, fb_l2=0, fb_l3=0, complete=314442, incomplete=133021
  chunk   1 | trip 4: matched=1047, fb_l1=266, fb_l2=0, fb_l3=0, 

In [ ]:
#With income
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.cluster import Birch
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.special import rel_entr, kl_div
import math
import time

import warnings
warnings.filterwarnings('ignore')

TRIP_MAX_ = 7

# --------------- Memory helpers ---------------

def _downcast_int(s: pd.Series):
    if s.isnull().any():
        return s
    i32 = s.astype(np.int64)
    if i32.min() >= np.iinfo(np.int8).min and i32.max() <= np.iinfo(np.int8).max:
        return i32.astype(np.int8)
    if i32.min() >= np.iinfo(np.int16).min and i32.max() <= np.iinfo(np.int16).max:
        return i32.astype(np.int16)
    if i32.min() >= np.iinfo(np.int32).min and i32.max() <= np.iinfo(np.int32).max:
        return i32.astype(np.int32)
    return i32

def _downcast_float(s: pd.Series):
    return s.astype(np.float32)

def reduce_mem_usage(df: pd.DataFrame, use_cats: bool = True):
    for col in df.columns:
        if pd.api.types.is_integer_dtype(df[col]):
            df[col] = _downcast_int(df[col])
        elif pd.api.types.is_float_dtype(df[col]):
            df[col] = _downcast_float(df[col])
        elif use_cats and (df[col].nunique(dropna=False) / max(1, len(df[col])) < 0.5):
            df[col] = df[col].astype('category')
    return df

def as_codes(df: pd.DataFrame, cols):
    """Convert columns to categorical codes (int32) consistently, preserving mapping via .cat.categories."""
    for c in cols:
        if not pd.api.types.is_categorical_dtype(df[c]):
            df[c] = df[c].astype('category')
        df[c] = df[c].cat.codes.astype(np.int32)
    return df

# --------------- Math helpers ---------------

def js_div(p, q):
    h = 0.5*np.add(p, q)
    return 0.5*rel_entr(p, h) + 0.5*rel_entr(q, h)

# --------------- Column ops (zero-copy where possible) ---------------

def _renmap(i: int):
    return {
        'TRIP_PURPOSE': f'TRIP_PURPOSE_{i}',
        'TRAVEL_MODE': f'TRAVEL_MODE_{i}',
        'ORIGIN_SUBZONE': f'ORIGIN_SUBZONE_{i}',
        'DESTINATION_SUBZONE': f'DESTINATION_SUBZONE_{i}',
        'TRIP_STARTTIME': f'TRIP_STARTTIME_{i}',
        'TRIP_ENDTIME': f'TRIP_ENDTIME_{i}',
        'ORIGIN_SUBZONE_X': f'ORIGIN_SUBZONE_X_{i}',
        'ORIGIN_SUBZONE_Y': f'ORIGIN_SUBZONE_Y_{i}',
        'DESTINATION_SUBZONE_X': f'DESTINATION_SUBZONE_X_{i}',
        'DESTINATION_SUBZONE_Y': f'DESTINATION_SUBZONE_Y_{i}',
        'Prob_XYZ_fus': f'Prob_{i}',
    }

def rename_columns(df: pd.DataFrame, i: int):
    df.rename(columns=_renmap(i), inplace=True)

# --------------- Sampling utils (no-merge) ---------------

def _prob_sample_indices(prob: np.ndarray, N: int) -> np.ndarray:
    prob = np.asarray(prob, dtype=np.float64)
    prob_sum = prob.sum()
    if prob_sum <= 0:
        # Avoid divide-by-zero; return empty
        return np.array([], dtype=np.int64)
    prob = prob / prob_sum
    # Use cumulative trick to avoid materializing large intermediate df
    return np.random.choice(len(prob), size=N, replace=True, p=prob)

def _select_columns(df: pd.DataFrame, cols):
    # Avoid copying where possible; pandas will create a view if safe.
    return df.loc[:, cols]

# --------------- Core pipeline ---------------

def simulate_trips(df_fus: pd.DataFrame, i: int, N: int) -> pd.DataFrame:
    att_X = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX']
    att_Z = [f'TRIP_PURPOSE_{i}', f'TRAVEL_MODE_{i}',
             f'ORIGIN_SUBZONE_{i}', f'ORIGIN_SUBZONE_X_{i}', f'ORIGIN_SUBZONE_Y_{i}',
             f'DESTINATION_SUBZONE_{i}', f'DESTINATION_SUBZONE_X_{i}', f'DESTINATION_SUBZONE_Y_{i}',
             f'TRIP_STARTTIME_{i}', f'TRIP_ENDTIME_{i}']

    df_sim = df_fus[df_fus['TRIP_CNT'] == i].drop(columns=['TRIP_CNT'])
    rename_columns(df_sim, i)
    df_sim.rename(columns={f'Prob_{i}': 'Prob'}, inplace=True)

    df_sim = _select_columns(df_sim, att_X + att_Z + ['Prob'])
    reduce_mem_usage(df_sim)

    idx = _prob_sample_indices(df_sim['Prob'].to_numpy(), N)
    if len(idx) == 0:
        return df_sim.iloc[0:0][att_X + att_Z]

    # Use iloc/take instead of merging on an artificial index
    out = df_sim.iloc[idx][att_X + att_Z].reset_index(drop=True)
    return reduce_mem_usage(out)

def simulate_tours(df_tours: pd.DataFrame, order: int, N: int) -> pd.DataFrame:
    att_X = ['ID', 'AGE', 'GENDER', 'INCOME', 'TRIP_MAX']
    att_Z = []
    for i in range(1, order + 1):
        att_Z.extend([f'TRIP_PURPOSE_{i}', f'TRAVEL_MODE_{i}',
                      f'ORIGIN_SUBZONE_{i}', f'ORIGIN_SUBZONE_X_{i}', f'ORIGIN_SUBZONE_Y_{i}',
                      f'DESTINATION_SUBZONE_{i}', f'DESTINATION_SUBZONE_X_{i}', f'DESTINATION_SUBZONE_Y_{i}',
                      f'TRIP_STARTTIME_{i}', f'TRIP_ENDTIME_{i}'])

    # Prob normalization without extra copy
    probs = df_tours['Prob'].to_numpy(dtype=np.float64)
    probs = probs / probs.sum() if probs.sum() > 0 else probs
    idx = _prob_sample_indices(probs, N)
    if len(idx) == 0:
        return df_tours.iloc[0:0][att_X + att_Z]

    out = df_tours.iloc[idx][att_X + att_Z].reset_index(drop=True)
    return reduce_mem_usage(out)

def next_start_times(df_hts_: pd.DataFrame) -> pd.DataFrame:
    # Build minimal table, count combos, then convert to conditional P(start_next | end_prev, purpose_prev)
    t = df_hts_[['ACTIVITY_STARTTIME', 'TRIP_PURPOSE', 'ACTIVITY_DURATION']].copy()
    t['TRIP_STARTTIME'] = t['ACTIVITY_STARTTIME'] + t['ACTIVITY_DURATION']
    t = t.rename(columns={'ACTIVITY_STARTTIME': 'TRIP_ENDTIME'}).drop(columns=['ACTIVITY_DURATION'])

    # Use value_counts -> Prob -> conditional
    t['Numbers'] = 1.0
    grp_keys = ['TRIP_ENDTIME', 'TRIP_PURPOSE', 'TRIP_STARTTIME']
    agg = t.groupby(grp_keys, observed=True)['Numbers'].sum().astype(np.float64).reset_index()
    agg['Prob'] = agg['Numbers'] / agg['Numbers'].sum()
    agg.drop(columns=['Numbers'], inplace=True)

    base_keys = ['TRIP_ENDTIME', 'TRIP_PURPOSE']
    agg['Prob_base'] = agg.groupby(base_keys, observed=True)['Prob'].transform('sum')
    agg['Prob_cond'] = (agg['Prob'] / agg['Prob_base']).astype(np.float32)
    agg = agg.drop(columns=['Prob_base', 'Prob'])

    # Downcast & codes for join keys
    reduce_mem_usage(agg)
    return agg.dropna()

def _prep_ith_trips(df_fus: pd.DataFrame, i: int) -> pd.DataFrame:
    # Slice, rename, then build conditional Prob over (X, Y, O, S)
    cols_keep = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX', 'TRIP_CNT',
                 'TRIP_PURPOSE', 'TRAVEL_MODE',
                 'ORIGIN_SUBZONE', 'ORIGIN_SUBZONE_X', 'ORIGIN_SUBZONE_Y',
                 'DESTINATION_SUBZONE', 'DESTINATION_SUBZONE_X', 'DESTINATION_SUBZONE_Y',
                 'TRIP_STARTTIME', 'TRIP_ENDTIME', 'Prob_XYZ_fus']
    ith = df_fus.loc[df_fus['TRIP_CNT'] == i, cols_keep].copy()
    ith.drop(columns=['TRIP_CNT'], inplace=True)
    rename_columns(ith, i)

    # Group denominator: sum Prob over X,Y,O,S
    den_keys = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX', f'ORIGIN_SUBZONE_{i}', f'TRIP_STARTTIME_{i}']
    ith['Prob_den'] = ith.groupby(den_keys, observed=True)[f'Prob_{i}'].transform('sum')
    ith[f'Prob_{i}'] = (ith[f'Prob_{i}'] / ith['Prob_den']).astype(np.float32)
    ith.drop(columns=['Prob_den'], inplace=True)

    return reduce_mem_usage(ith)

def concat_trips(df_tours: pd.DataFrame,
                 df_ith_trips: pd.DataFrame,
                 df_next_start_times: pd.DataFrame,
                 i: int):
    # Build minimal PRE_* keys to join
    tours = df_tours.copy()
    tours['PRE_SUBZONE'] = tours[f'DESTINATION_SUBZONE_{i-1}']
    tours['PRE_PURPOSE'] = tours[f'TRIP_PURPOSE_{i-1}']
    tours['PRE_ENDTIME'] = tours[f'TRIP_ENDTIME_{i-1}']

    nst = df_next_start_times.rename(columns={
        'TRIP_PURPOSE': 'PRE_PURPOSE',
        'TRIP_ENDTIME': 'PRE_ENDTIME',
        'TRIP_STARTTIME': 'PRE_STARTTIME'
    }, errors='ignore')

    ith = df_ith_trips.copy()
    ith['PRE_SUBZONE'] = ith[f'ORIGIN_SUBZONE_{i}']
    ith['PRE_STARTTIME'] = ith[f'TRIP_STARTTIME_{i}']

    # Left-join tours with cond P(next start | prev end, purpose) to get Prob_cond
    tours = tours.merge(
        nst[[#'AGE', 'GENDER', 'TRIP_MAX',
            'PRE_PURPOSE', 'PRE_ENDTIME', 'PRE_STARTTIME', 'Prob_cond']],
        on=[#'AGE', 'GENDER', 'TRIP_MAX',
            'PRE_PURPOSE', 'PRE_ENDTIME'],
        how='left',
        copy=False
    )

    # Keyed join with i-th trips on (X, Y, PRE_SUBZONE, PRE_STARTTIME)
    join_keys = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX', 'PRE_SUBZONE', 'PRE_STARTTIME']
    tours = tours.merge(
        ith[join_keys + [f'TRIP_PURPOSE_{i}', f'TRAVEL_MODE_{i}',
                         f'ORIGIN_SUBZONE_{i}', f'ORIGIN_SUBZONE_X_{i}', f'ORIGIN_SUBZONE_Y_{i}',
                         f'DESTINATION_SUBZONE_{i}', f'DESTINATION_SUBZONE_X_{i}', f'DESTINATION_SUBZONE_Y_{i}',
                         f'TRIP_STARTTIME_{i}', f'TRIP_ENDTIME_{i}', f'Prob_{i}']],
        on=join_keys,
        how='inner',
        copy=False
    )

    # Combine probabilities
    tours['Prob'] = (tours[f'Prob_{i}'] * tours['Prob_cond']).astype(np.float32)
    tours.drop(columns=[f'Prob_{i}', 'Prob_cond', 'PRE_SUBZONE', 'PRE_PURPOSE', 'PRE_ENDTIME', 'PRE_STARTTIME'], inplace=True)

    # Split complete vs incomplete
    mask_done = (tours['TRIP_MAX'] == i)
    mask_home = mask_done & (tours[f'TRIP_PURPOSE_{i}'] == 0)

    complete_home = tours.loc[mask_home].copy()
    # Enforce return to home
    complete_home[f'DESTINATION_SUBZONE_{i}']   = complete_home['ORIGIN_SUBZONE_1']
    complete_home[f'DESTINATION_SUBZONE_X_{i}'] = complete_home['ORIGIN_SUBZONE_X_1']
    complete_home[f'DESTINATION_SUBZONE_Y_{i}'] = complete_home['ORIGIN_SUBZONE_Y_1']

    complete_nohome = tours.loc[mask_done & ~mask_home]
    df_tours_complete = pd.concat([complete_home, complete_nohome], ignore_index=True)

    df_tours_incomplete = tours.loc[~mask_done]

    return reduce_mem_usage(df_tours_complete), reduce_mem_usage(df_tours_incomplete)

def generate_tours(df_fus: pd.DataFrame, df_next_start_times: pd.DataFrame, N: int) -> pd.DataFrame:
    scale = 5

    # Seed tours from TRIP_CNT==1
    df_tours_seed = df_fus[df_fus['TRIP_CNT'] == 1]
    df_tours = simulate_trips(df_tours_seed, 1, int(scale * N)).reset_index(drop=True)
    df_tours = df_tours.reset_index()
    df_tours['ID'] = df_tours['index'].astype(np.int64)
    df_tours.drop(columns=['index'], inplace=True)

    # Trip-max proportions per (AGE, GENDER, TRIP_MAX)
    df_tm = df_tours[['AGE', 'GENDER', 'INCOME', 'TRIP_MAX']].copy()
    df_tm['Numbers'] = 1.0
    df_tm = df_tm.groupby(['AGE', 'GENDER', 'INCOME', 'TRIP_MAX'], observed=True)['Numbers'].sum().reset_index()
    df_tm['Pop'] = (df_tm['Numbers'] / df_tm['Numbers'].sum()).astype(np.float32)
    df_tm.drop(columns=['Numbers'], inplace=True)
    trip_max = {(r.AGE, r.GENDER, r.INCOME, r.TRIP_MAX): r.Pop for r in df_tm.itertuples(index=False)}

    # Chunk process to cap peak RAM
    out_chunks = []
    chunk_size = 10000
    Z = (len(df_tours) + chunk_size - 1) // chunk_size
    print('Number of chunks ', Z)

    # Precompute next-start table as int codes to reduce join memory
    # (keep original too; merge handles both)
    df_next_start_times = reduce_mem_usage(df_next_start_times.copy())

    for z in range(Z):
        df_tours_ = df_tours.iloc[z*chunk_size:(z+1)*chunk_size].copy()
        df_tours_['Prob'] = 1.0  # starts uniform before conditionals
        df_result = []

        for i in range(2, TRIP_MAX_ + 1):
            df_ith_trips = _prep_ith_trips(df_fus, i)
            df_tours_complete, df_tours_incomplete = concat_trips(df_tours_, df_ith_trips, df_next_start_times, i)

            if len(df_tours_complete) > 0:
                df_result.append(simulate_tours(df_tours_complete, i, int(scale * N)))
            if len(df_tours_incomplete) > 0:
                # Advance partial tours for next step; reweight by sampling
                df_tours_ = simulate_tours(df_tours_incomplete, i, int(scale * N))
            else:
                break

        if len(df_result):
            res = pd.concat(df_result, ignore_index=True)
            # Keep first completion per ID
            res = res.drop_duplicates(subset=['ID'], keep='first')
            out_chunks.append(res)
        print('Finish chunk ', str(z))

    if not out_chunks:
        return pd.DataFrame(columns=['ID', 'AGE', 'GENDER', 'INCOME', 'TRIP_MAX'])

    df_result_ = pd.concat(out_chunks, ignore_index=True)
    df_result_ = df_result_.drop_duplicates(subset=['ID'], keep='first')

    # Final cohort-downsampling to match TRIP_MAX proportions
    final_chunks = []
    for (age, gender, income, tmax), pop in trip_max.items():
        df_tmp = df_result_[(df_result_['TRIP_MAX'] == tmax) &
                            (df_result_['AGE'] == age) &
                            (df_result_['INCOME'] == income) &
                            (df_result_['GENDER'] == gender)]
        target = int(N * pop)
        n2 = len(df_tmp)
        if n2 > target and target > 0:
            df_tmp = df_tmp.sample(n=target, replace=False, random_state=None)
        final_chunks.append(df_tmp)

    df_result_final = pd.concat(final_chunks, ignore_index=True) if final_chunks else df_result_.iloc[0:0]
    return reduce_mem_usage(df_result_final)

# ------------------------------------------------------------
# __main__: Example end-to-end run (adjust paths)
# ------------------------------------------------------------

ff = 0.1

df_hts = pd.read_csv(path_data + f'val_data_hts_trip_{ff}.csv')
#df_true_trip = pd.read_csv(path_data + 'data_sgp_hts_trip.csv')
#df_pcm = pd.read_csv(path_data + 'val_data_pcm_trip.csv')

start_time = time.time()

df_fus = pd.read_csv(path_result + f'val_sim_trip_proposed_{ff}.csv')
df_fus = reduce_mem_usage(df_fus)
print('df_fus lenght', len(df_fus))

df_next_start_times = next_start_times(df_hts)

#
NN = 50000 #len(df_true_trip[['ID']].drop_duplicates())
df_tours = generate_tours(df_fus, df_next_start_times, NN)
print(df_tours.info())
df_tours.to_csv(path_result + f'val_sim_tour_proposed_{ff}.csv', index = False)

print('Time take %s seconds' % (time.time() - start_time))
print('Done!')

df_fus lenght 3055883
Number of chunks  25
Finish chunk  0
Finish chunk  1
Finish chunk  2
Finish chunk  3
Finish chunk  4
Finish chunk  5
Finish chunk  6
Finish chunk  7
Finish chunk  8
Finish chunk  9
Finish chunk  10
Finish chunk  11
Finish chunk  12
Finish chunk  13
Finish chunk  14
Finish chunk  15
Finish chunk  16
Finish chunk  17
Finish chunk  18
Finish chunk  19
Finish chunk  20
Finish chunk  21
Finish chunk  22
Finish chunk  23
Finish chunk  24
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49789 entries, 0 to 49788
Data columns (total 75 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   ID                       49789 non-null  int32   
 1   AGE                      49789 non-null  int8    
 2   GENDER                   49789 non-null  int8    
 3   INCOME                   49789 non-null  int8    
 4   TRIP_MAX                 49789 non-null  int8    
 5   TRIP_PURPOSE_1           49789 non-null  in

#Validation: Stage 1 Finding M

In [ ]:
import os, math, gc, torch, numpy as np, pandas as pd
from dataclasses import replace

# ---------------------------
# Minimal, RAM-friendly train
# ---------------------------
def train_encoder_only_fusion_finding_K_MINRAM(
    df_hts: pd.DataFrame,
    df_pcm: pd.DataFrame,
    cfg: "TrainConfig",
    phi_segments_meta: Dict[str,int],
    use_mixed_precision: bool = True,   # safe no-op on CPU; helpful on CUDA
) -> Dict[str, float]:
    """
    RAM-efficient variant:
      - No 'pack' and no model object returned.
      - No per-epoch history stored.
      - Temporary tensors are freed aggressively.
    Returns tiny dict of scalars: {"K", "L_hts_final", "fusion_consistency_js"}.
    """
    device = cfg.device
    dtype = cfg.dtype

    # 1) Prepare data and loaders (do NOT keep extra returns around)
    n_xy, _, _, hts_loader_full, pcm_loader_full, _phi_dim = _prepare_xy_and_loaders(df_hts, df_pcm, cfg)

    # 2) Build encoder (embedding version)
    cardinals = (
        phi_segments_meta["n_dist"],
        phi_segments_meta["n_start"],
        phi_segments_meta["n_end"],
        phi_segments_meta["n_cluster"],
    )
    enc = EncoderEmbed(
        cardinals=cardinals,
        K=cfg.K,
        emb_dims=(16, 16, 16, 16),
        hidden=cfg.enc_hidden,
        num_layers=cfg.enc_layers,
        dropout=cfg.enc_dropout,
    ).to(device=device, dtype=torch.float32)  # keep params in fp32 for stability

    opt = torch.optim.AdamW(enc.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
    logXY = math.log(max(n_xy, 2))

    # 3) Temperature & support policy
    tau = get_tau(1, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
    support_policy = _make_support_policy(enc, hts_loader_full, tau, cfg)

    # Initial non-parametric decoder; compute under no_grad and on CPU
    with torch.no_grad():
        P_xy_h = build_nonparam_decoder_xy_given_h(
            encoder=enc,
            hts_loader=hts_loader_full,
            n_xy=n_xy,
            K=cfg.K,
            device=device,
            dtype=dtype,
            progress=cfg.progress,
            tau=tau,
            support_policy=support_policy,
        )

    # 4) Train loop — no history stored
    scaler = torch.cuda.amp.GradScaler(enabled=(use_mixed_precision and torch.cuda.is_available()))
    for ep in range(1, cfg.epochs + 1):
        tau = get_tau(ep, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
        enc.train()
        opt.zero_grad(set_to_none=True)

        if (
            cfg.use_bucket_candidates
            and cfg.refresh_bucket_every > 0
            and (ep == 1 or (ep % cfg.refresh_bucket_every) == 0)
        ):
            support_policy = _make_support_policy(enc, hts_loader_full, tau, cfg)

        # Mixed precision autocast (safe no-op on CPU)
        autocast_enabled = (use_mixed_precision and torch.cuda.is_available())
        with torch.cuda.amp.autocast(enabled=autocast_enabled):
            L_hts, _ = compute_hts_nll(
                encoder=enc,
                hts_loader=hts_loader_full,
                P_xy_given_h=P_xy_h,
                log_norm=(logXY if cfg.use_normalized_objective else 1.0),
                device=device,
                dtype=dtype,
                progress=cfg.progress,
                tau=tau,
                support_policy=support_policy,
            )
            total = L_hts

        # Backprop + step
        if scaler.is_enabled():
            scaler.scale(total).backward()
            scaler.step(opt)
            scaler.update()
        else:
            total.backward()
            opt.step()

        # Rebuild decoder periodically without storing old copies
        if (ep % cfg.rebuild_every) == 0:
            with torch.no_grad():
                P_xy_h = build_nonparam_decoder_xy_given_h(
                    encoder=enc,
                    hts_loader=hts_loader_full,
                    n_xy=n_xy,
                    K=cfg.K,
                    device=device,
                    dtype=dtype,
                    progress=cfg.progress,
                    tau=tau,
                    support_policy=support_policy,
                )

        print(f"[epoch {ep:03d}] tau={tau:.4f}  L_hts={float(L_hts.detach().cpu()):.6f}")

        # Proactively free ephemeral graph/tensors
        del L_hts, total
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # 5) Final evaluation — compute scalars only
    tau_final = get_tau(cfg.epochs, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
    support_policy = _make_support_policy(enc, hts_loader_full, tau_final, cfg)

    with torch.no_grad():
        P_xy_h_final = build_nonparam_decoder_xy_given_h(
            encoder=enc,
            hts_loader=hts_loader_full,
            n_xy=n_xy,
            K=cfg.K,
            device=device,
            dtype=dtype,
            progress=False,
            tau=tau_final,
            support_policy=support_policy,
        )

        L_hts_final, _ = compute_hts_nll(
            encoder=enc,
            hts_loader=hts_loader_full,
            P_xy_given_h=P_xy_h_final,
            log_norm=(math.log(max(n_xy, 2)) if cfg.use_normalized_objective else 1.0),
            device=device,
            dtype=dtype,
            progress=False,
            tau=tau_final,
            support_policy=support_policy,
        )

        L_fus_final = compute_fusion_js(
            encoder=enc,
            hts_loader=hts_loader_full,
            pcm_loader=pcm_loader_full,
            K=cfg.K,
            device=device,
            dtype=dtype,
            use_normalized=cfg.use_normalized_objective,
            tau=tau_final,
            support_policy=support_policy,
        )

    print(f"[FINAL] K={cfg.K}  tau={tau_final:.4f}")
    print(f"        Final likelihood (HTS NLL): {float(L_hts_final):.6f}")
    print(f"        Fusion consistency (JS):    {float(L_fus_final):.6f}")

    # Pull out tiny scalars and immediately free everything heavy
    K_val  = int(cfg.K)
    L_val  = float(L_hts_final.detach().cpu().item())
    JS_val = float(L_fus_final.detach().cpu().item())

    # Drop references to big objects
    del P_xy_h, P_xy_h_final, enc, opt, support_policy
    del hts_loader_full, pcm_loader_full
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return {"K": K_val, "L_hts_final": L_val, "fusion_consistency_js": JS_val}


# ---------------------------
# Normalization & elbow (as-is)
# ---------------------------
def _minmax_norm(arr: np.ndarray) -> np.ndarray:
    return arr

def find_elbow_by_max_distance(K_vals: np.ndarray, scores: np.ndarray) -> int:
    assert len(K_vals) == len(scores) and len(K_vals) >= 2
    x = _minmax_norm(np.asarray(K_vals, dtype=float))
    y = np.asarray(scores, dtype=float)
    x1, y1 = x[0], y[0]; x2, y2 = x[-1], y[-1]
    denom = np.hypot(y2 - y1, x2 - x1)
    if denom < 1e-12:
        return int(np.argmin(y))
    num = np.abs((y2 - y1) * x - (x2 - x1) * y + (x2 * y1 - y2 * x1))
    return int(np.argmax(num))


# ---------------------------
# K sweep — store metrics only
# ---------------------------
if __name__ == "__main__":
    # --- Load data (unchanged) ---
    frac = '0.2'
    df_hts = pd.read_csv(os.path.join(path_data, f"val_data_hts_trip_{frac}.csv"))
    df_pcm = pd.read_csv(os.path.join(path_data, "val_data_pcm_trip.csv"))
    df_true_trip = pd.read_csv(os.path.join(path_data, "data_sgp_hts_trip.csv"))

    if "COUNT" not in df_pcm.columns:
        df_pcm["COUNT"] = 1.0
        df_pcm["COUNT"] = df_pcm.groupby(
            ["ORIGIN_SUBZONE","DESTINATION_SUBZONE","TRIP_STARTTIME","TRIP_ENDTIME"]
        )["COUNT"].transform("sum")
        df_pcm = df_pcm.drop_duplicates()

    # --- Build Φ and slim tables (unchanged structure) ---
    df_hts["TRIP_DISTANCE"] = distance(df_hts, ['ORIGIN_SUBZONE_X','DESTINATION_SUBZONE_X','ORIGIN_SUBZONE_Y','DESTINATION_SUBZONE_Y'])
    df_pcm["TRIP_DISTANCE"] = distance(df_pcm, ['ORIGIN_SUBZONE_X','DESTINATION_SUBZONE_X','ORIGIN_SUBZONE_Y','DESTINATION_SUBZONE_Y'])
    df_land_use = land_use(df_true_trip, df_pcm)
    df_mode = mode_share(df_true_trip, df_pcm)
    df_hts = df_hts.merge(df_land_use, on='DESTINATION_SUBZONE').merge(df_mode, on='DESTINATION_SUBZONE')
    df_pcm = df_pcm.merge(df_land_use, on='DESTINATION_SUBZONE').merge(df_mode, on='DESTINATION_SUBZONE')

    nb = 10
    spec = DiscreteSpec(n_dist=nb, n_start=nb, n_end=nb, n_lu_mode_clusters=nb,
                        dist_binning="uniform", time_binning="uniform")
    fitted = fit_semantic_spec_on_hts(df_hts, spec)
    df_hts = apply_semantic_spec(df_hts, fitted)
    df_pcm = apply_semantic_spec(df_pcm, fitted)
    phi_segments_meta = fitted["phi_segments"]

    df_hts, phi_dim = build_phi_indices_from_bins(df_hts)
    df_pcm, _       = build_phi_indices_from_bins(df_pcm)

    att_X = ['AGE','GENDER', 'INCOME']
    att_Y = ['TRIP_CNT','TRIP_MAX','TRIP_PURPOSE','TRAVEL_MODE']
    att_Z = ['ORIGIN_SUBZONE','DESTINATION_SUBZONE','TRIP_STARTTIME','TRIP_ENDTIME',
             'ORIGIN_SUBZONE_X','DESTINATION_SUBZONE_X','ORIGIN_SUBZONE_Y','DESTINATION_SUBZONE_Y']
    df_hts = df_hts[att_X + att_Y + att_Z + ['PHI_IDX']]
    df_pcm = df_pcm[att_Z + ['PHI_IDX','COUNT']]
    df_hts['ATT_X'] = df_hts[att_X].astype(str).apply('_'.join, axis=1)
    df_hts['ATT_Y'] = df_hts[att_Y].astype(str).apply('_'.join, axis=1)
    df_hts['ATT_Z'] = df_hts[att_Z].astype(str).apply('_'.join, axis=1)
    df_pcm['ATT_Z'] = df_pcm[att_Z].astype(str).apply('_'.join, axis=1)
    df_hts = reduce_mem_usage(df_hts)
    df_pcm = reduce_mem_usage(df_pcm)

    # --- Config baseline (unchanged) ---
    cfg_base = TrainConfig(
        K=2500, enc_hidden=256, enc_layers=2, enc_dropout=0.0,
        epochs=5, batch_size_hts=8192, batch_size_pcm=16384,
        lr=2e-3, wd=1e-4, device=("cuda" if torch.cuda.is_available() else "cpu"),
        use_normalized_objective=True, rebuild_every=1, progress=True,
        tau_start=0.01, tau_end=0.01, tau_schedule="cosine",
        topk_h=10, use_bucket_candidates=False,
    )

    # --- Sweep K WITHOUT storing models or packs ---
    K_grid = [i*500 for i in range(1,20)]
    rows = []
    for K in K_grid:
        print(f"\n=== Running K={K} ===")
        cfgK = replace(cfg_base, K=K)
        out = train_encoder_only_fusion_finding_K_MINRAM(
            df_hts=df_hts, df_pcm=df_pcm, cfg=cfgK,
            phi_segments_meta=phi_segments_meta, use_mixed_precision=True,
        )
        rows.append(out)
        # hard cleanup between runs
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    dfK = pd.DataFrame(rows).sort_values("K").reset_index(drop=True)

    # Normalize (you kept identity; leave as-is)
    L_norm  = _minmax_norm(dfK["L_hts_final"].to_numpy())
    JS_norm = _minmax_norm(dfK["fusion_consistency_js"].to_numpy())
    score_norm = L_norm + JS_norm

    K_vals = dfK["K"].to_numpy(dtype=int)
    elbow_idx = find_elbow_by_max_distance(K_vals, score_norm)
    best_K = int(K_vals[elbow_idx])

    dfK["L_hts_final_norm"] = L_norm
    dfK["fusion_js_norm"]   = JS_norm
    dfK["score_norm"]       = score_norm
    dfK["is_elbow"]         = False
    dfK.loc[elbow_idx, "is_elbow"] = True

    print("\n=== K sweep results (elbow on normalized sum) ===")
    print(dfK[["K", "L_hts_final", "fusion_consistency_js", "L_hts_final_norm", "fusion_js_norm", "score_norm", "is_elbow"]])
    print(f"\n>>> Selected elbow K = {best_K}")




=== Running K=500 ===
[decoder] pass chunk 0
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 001] tau=0.0100  L_hts=0.616676
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 002] tau=0.0100  L_hts=0.552764
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 003] tau=0.0100  L_hts=0.516114
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 004] tau=0.0100  L_hts=0.505671
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 005] tau=0.0100  L_hts=0.495832
[FINAL] K=500  tau=0.0100
        Final likelihood (HTS NLL): 0.487612
        Fusion consistency (JS):    0.005555

=== Running K=1000 ===
[decoder] pass chunk 0
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 001] tau=0.0100  L_hts=0.589339
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 002] tau=0.0100  L_hts=0.527310
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 003] tau=0.0100  L_hts=0.486486
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 004] tau=0.0100  L_hts=0.474798
[hts-nll] pass chunk 0
[de

# Controlled experiment: Construct activity schedules

In [ ]:
from __future__ import annotations

import os
import time
from typing import Dict, Literal

import numpy as np
import pandas as pd

# ============================================================
# 1. Configuration
# ============================================================

MAX_TRIPS = 7


AGE_GROUP_MAPPING = {
    0: "age_0",
    1: "age_1",
    2: "age_2",
    3: "age_3",
}

AGE_GROUPS = list(AGE_GROUP_MAPPING.values())


GENDER_GROUP_MAPPING = {
    0: "male",
    1: "female",
}

GENDER_GROUPS = list(GENDER_GROUP_MAPPING.values())


INCOME_GROUP_MAPPING = {
    0: "income_0",
    1: "income_1",
    2: "income_2",
}

INCOME_GROUPS = list(INCOME_GROUP_MAPPING.values())


############################################
# 2. Reshape wide schedules into long trips
############################################
# For seoul
def reshape_trip(
    df: pd.DataFrame,
    max_trips: int = MAX_TRIPS,
) -> pd.DataFrame:
    """
    Convert a wide person-level schedule dataframe into long trip format.

    Each output row represents one trip.

    Required person-level columns
    -----------------------------
    ID, AGE, GENDER, INCOME, TRIP_MAX

    Required trip columns
    ---------------------
    TRIP_PURPOSE_i
    TRAVEL_MODE_i
    TRIP_STARTTIME_i
    TRIP_ENDTIME_i
    ORIGIN_SUBZONE_i
    DESTINATION_SUBZONE_i
    """

    person_columns = [
        "ID",
        "AGE",
        "GENDER",
        "INCOME",
        "TRIP_MAX",
    ]

    missing_person_columns = [column for column in person_columns if column not in df.columns]

    if missing_person_columns:
        raise KeyError("Missing person-level columns: " f"{missing_person_columns}")

    parts = []

    for i in range(1, max_trips + 1):

        trip_columns = [
            f"TRIP_PURPOSE_{i}",
            f"TRAVEL_MODE_{i}",
            f"TRIP_STARTTIME_{i}",
            f"TRIP_ENDTIME_{i}",
            f"ORIGIN_SUBZONE_{i}",
            f"DESTINATION_SUBZONE_{i}",
        ]

        # Skip unavailable trip slots.
        if not all(column in df.columns for column in trip_columns):
            continue

        dff = df[person_columns + trip_columns].copy()

        dff = dff.rename(
            columns={
                f"TRIP_PURPOSE_{i}": "TRIP_PURPOSE",
                f"TRAVEL_MODE_{i}": "TRAVEL_MODE",
                f"TRIP_STARTTIME_{i}": "TRIP_STARTTIME",
                f"TRIP_ENDTIME_{i}": "TRIP_ENDTIME",
                f"ORIGIN_SUBZONE_{i}": "ORIGIN_SUBZONE",
                f"DESTINATION_SUBZONE_{i}": "DESTINATION_SUBZONE",
            }
        )

        dff["TRIP_NUMBER"] = i

        # Retain trip i only if the person reports at least i trips.
        trip_max_numeric = pd.to_numeric(
            dff["TRIP_MAX"],
            errors="coerce",
        )

        dff = dff.loc[trip_max_numeric >= i].copy()

        parts.append(dff)

    if not parts:
        raise ValueError("No valid trip-column sets were found.")

    trips = pd.concat(
        parts,
        ignore_index=True,
    )

    numeric_columns = [
        "ID",
        "AGE",
        "GENDER",
        "INCOME",
        "TRIP_MAX",
        "TRIP_NUMBER",
        "TRIP_PURPOSE",
        "TRIP_STARTTIME",
        "TRIP_ENDTIME",
    ]

    for column in numeric_columns:
        trips[column] = pd.to_numeric(
            trips[column],
            errors="coerce",
        )

    # Subzone identifiers may be alphanumeric and must remain strings.
    # Strip surrounding whitespace and standardize textual missing values.
    subzone_columns = [
        "ORIGIN_SUBZONE",
        "DESTINATION_SUBZONE",
    ]

    for column in subzone_columns:
        trips[column] = (
            trips[column]
            .astype("string")
            .str.strip()
            .replace(
                {
                    "": pd.NA,
                    "nan": pd.NA,
                    "None": pd.NA,
                    "<NA>": pd.NA,
                }
            )
        )

    # Activity reconstruction requires valid origin/destination subzones
    # and a valid destination-purpose code.
    # TRAVEL_MODE may remain missing because it is not used below.
    trips = trips.dropna(
        subset=[
            "ID",
            "AGE",
            "GENDER",
            "INCOME",
            "TRIP_MAX",
            "TRIP_NUMBER",
            "TRIP_PURPOSE",
            "TRIP_STARTTIME",
            "TRIP_ENDTIME",
            "ORIGIN_SUBZONE",
            "DESTINATION_SUBZONE",
        ]
    ).copy()

    # Retain valid demographic codes.
    trips = trips.loc[
        trips["AGE"].isin(AGE_GROUP_MAPPING)
        & trips["GENDER"].isin(GENDER_GROUP_MAPPING)
        & trips["INCOME"].isin(INCOME_GROUP_MAPPING)
    ].copy()

    # Retain valid activity-purpose and hourly codes.
    trips = trips.loc[
        trips["TRIP_PURPOSE"].isin([0, 1, 2, 3, 4, 5])
        & trips["TRIP_STARTTIME"].between(0, 23)
        & trips["TRIP_ENDTIME"].between(0, 23)
    ].copy()

    # Each trip arrival should not precede its departure.
    trips = trips.loc[trips["TRIP_ENDTIME"] >= trips["TRIP_STARTTIME"]].copy()

    integer_columns = [
        "ID",
        "AGE",
        "GENDER",
        "INCOME",
        "TRIP_MAX",
        "TRIP_NUMBER",
        "TRIP_PURPOSE",
    ]

    trips[integer_columns] = trips[integer_columns].astype(int)

    trips["ORIGIN_SUBZONE"] = trips["ORIGIN_SUBZONE"].astype("string")

    trips["DESTINATION_SUBZONE"] = trips["DESTINATION_SUBZONE"].astype("string")

    trips["AGE_GROUP"] = trips["AGE"].map(AGE_GROUP_MAPPING)

    trips["GENDER_GROUP"] = trips["GENDER"].map(GENDER_GROUP_MAPPING)

    trips["INCOME_GROUP"] = trips["INCOME"].map(INCOME_GROUP_MAPPING)

    duplicated = trips.duplicated(
        subset=["ID", "TRIP_NUMBER"],
        keep=False,
    )

    if duplicated.any():
        examples = trips.loc[
            duplicated,
            ["ID", "TRIP_NUMBER"],
        ].head(10)

        raise ValueError("Duplicate trip numbers were found within individuals.\n" f"{examples}")

    return trips.sort_values(["ID", "TRIP_NUMBER"]).reset_index(drop=True)


############################################
# 3. Reconstruct complete daily activities
############################################


def calculate_destination_activities(
    trips: pd.DataFrame,
    day_start_hour: float = 0.0,
    day_end_hour: float = 23.0,
    invalid_timing: Literal["drop", "raise", "keep"] = "drop",
) -> pd.DataFrame:
    """
    Reconstruct the complete sequence of daily activities.

    Parameters
    ----------
    trips:
        Output from reshape_trip().

    day_start_hour:
        Beginning of the observation day.

    day_end_hour:
        End of the observation day.

    invalid_timing:
        Treatment of invalid activity intervals where end time is earlier
        than start time:

        "drop":
            Remove invalid activity rows.

        "raise":
            Raise an error.

        "keep":
            Retain invalid rows.
    """

    required_columns = [
        "ID",
        "AGE",
        "AGE_GROUP",
        "GENDER",
        "GENDER_GROUP",
        "INCOME",
        "INCOME_GROUP",
        "TRIP_NUMBER",
        "TRIP_PURPOSE",
        "TRIP_STARTTIME",
        "TRIP_ENDTIME",
        "ORIGIN_SUBZONE",
        "DESTINATION_SUBZONE",
    ]

    missing_columns = [column for column in required_columns if column not in trips.columns]

    if missing_columns:
        raise KeyError(
            "Missing columns required for activity reconstruction: " f"{missing_columns}"
        )

    data = trips.copy().sort_values(["ID", "TRIP_NUMBER"]).reset_index(drop=True)

    if data.empty:
        raise ValueError("The trip dataframe is empty.")

    demographic_columns = [
        "ID",
        "AGE",
        "AGE_GROUP",
        "GENDER",
        "GENDER_GROUP",
        "INCOME",
        "INCOME_GROUP",
    ]

    # Identify the first and last trip of each person.
    data["_ROW_ORDER"] = data.groupby("ID").cumcount()

    data["_REVERSE_ROW_ORDER"] = data.groupby("ID").cumcount(ascending=False)

    first_trips = data.loc[data["_ROW_ORDER"] == 0].copy()

    last_trips = data.loc[data["_REVERSE_ROW_ORDER"] == 0].copy()

    activity_parts = []

    # ========================================================
    # First activity
    # ========================================================

    first_activities = first_trips[demographic_columns].copy()

    first_activities["ACTIVITY_ORDER"] = 1

    # The day begins at home by definition.
    first_activities["ACTIVITY_PURPOSE"] = 0

    first_activities["ACTIVITY_SUBZONE"] = first_trips["ORIGIN_SUBZONE"].values

    first_activities["ACTIVITY_STARTTIME"] = day_start_hour

    first_activities["ACTIVITY_ENDTIME"] = first_trips["TRIP_STARTTIME"].values

    first_activities["ACTIVITY_TYPE"] = "first_activity"

    activity_parts.append(first_activities)

    # ========================================================
    # Intermediate activities
    # ========================================================

    data["NEXT_TRIP_STARTTIME"] = data.groupby("ID")["TRIP_STARTTIME"].shift(-1)

    intermediate_trips = data.loc[data["_REVERSE_ROW_ORDER"] > 0].copy()

    if not intermediate_trips.empty:

        intermediate_activities = intermediate_trips[demographic_columns].copy()

        # Trip 1 creates activity 2, trip 2 creates activity 3.
        intermediate_activities["ACTIVITY_ORDER"] = (
            intermediate_trips["TRIP_NUMBER"].astype(int).values + 1
        )

        intermediate_activities["ACTIVITY_PURPOSE"] = intermediate_trips["TRIP_PURPOSE"].values

        intermediate_activities["ACTIVITY_SUBZONE"] = intermediate_trips[
            "DESTINATION_SUBZONE"
        ].values

        intermediate_activities["ACTIVITY_STARTTIME"] = intermediate_trips["TRIP_ENDTIME"].values

        intermediate_activities["ACTIVITY_ENDTIME"] = intermediate_trips[
            "NEXT_TRIP_STARTTIME"
        ].values

        intermediate_activities["ACTIVITY_TYPE"] = "intermediate_activity"

        activity_parts.append(intermediate_activities)

    # ========================================================
    # Last activity
    # ========================================================

    last_activities = last_trips[demographic_columns].copy()

    last_activities["ACTIVITY_ORDER"] = last_trips["TRIP_NUMBER"].astype(int).values + 1

    last_activities["ACTIVITY_PURPOSE"] = last_trips["TRIP_PURPOSE"].values

    last_activities["ACTIVITY_SUBZONE"] = last_trips["DESTINATION_SUBZONE"].values

    last_activities["ACTIVITY_STARTTIME"] = last_trips["TRIP_ENDTIME"].values

    last_activities["ACTIVITY_ENDTIME"] = day_end_hour

    last_activities["ACTIVITY_TYPE"] = "last_activity"

    activity_parts.append(last_activities)

    # ========================================================
    # Combine activity records
    # ========================================================

    activities = pd.concat(
        activity_parts,
        ignore_index=True,
    )

    activities["ACTIVITY_STARTTIME"] = pd.to_numeric(
        activities["ACTIVITY_STARTTIME"],
        errors="coerce",
    )

    activities["ACTIVITY_ENDTIME"] = pd.to_numeric(
        activities["ACTIVITY_ENDTIME"],
        errors="coerce",
    )

    activities = activities.dropna(
        subset=[
            "ID",
            "ACTIVITY_ORDER",
            "ACTIVITY_SUBZONE",
            "ACTIVITY_STARTTIME",
            "ACTIVITY_ENDTIME",
        ]
    ).copy()

    # Inclusive duration requested by the study design:
    # duration = end time - start time + 1.
    #
    # Examples:
    # start=8, end=8  -> duration=1 hour
    # start=8, end=10 -> duration=3 hours
    activities["ACTIVITY_DURATION_HOURS"] = (
        activities["ACTIVITY_ENDTIME"] - activities["ACTIVITY_STARTTIME"] + 1.0
    )

    # Check the underlying interval directly. With inclusive duration,
    # end < start corresponds to a non-positive/invalid interval.
    activities["INVALID_ACTIVITY_TIMING"] = (
        activities["ACTIVITY_ENDTIME"] < activities["ACTIVITY_STARTTIME"]
    )

    invalid_rows = activities.loc[activities["INVALID_ACTIVITY_TIMING"]]

    if not invalid_rows.empty:

        invalid_examples = invalid_rows[
            [
                "ID",
                "ACTIVITY_ORDER",
                "ACTIVITY_STARTTIME",
                "ACTIVITY_ENDTIME",
                "ACTIVITY_DURATION_HOURS",
            ]
        ].head(10)

        if invalid_timing == "raise":
            raise ValueError("Invalid activity intervals were found:\n" f"{invalid_examples}")

        if invalid_timing == "drop":
            activities = activities.loc[~activities["INVALID_ACTIVITY_TIMING"]].copy()

        elif invalid_timing != "keep":
            raise ValueError("invalid_timing must be " "'drop', 'raise', or 'keep'.")

    # Convert the inclusive duration directly to minutes.
    activities["ACTIVITY_DURATION_MINUTES"] = activities["ACTIVITY_DURATION_HOURS"] * 60.0

    activities["ACTIVITY_ORDER"] = activities["ACTIVITY_ORDER"].astype(int)

    activities["ACTIVITY_SUBZONE"] = activities["ACTIVITY_SUBZONE"].astype("string").str.strip()

    activities = activities.sort_values(["ID", "ACTIVITY_ORDER"]).reset_index(drop=True)

    output_columns = [
        "ID",
        "AGE",
        "AGE_GROUP",
        "GENDER",
        "GENDER_GROUP",
        "INCOME",
        "INCOME_GROUP",
        "ACTIVITY_ORDER",
        "ACTIVITY_PURPOSE",
        "ACTIVITY_SUBZONE",
        "ACTIVITY_STARTTIME",
        "ACTIVITY_ENDTIME",
        "ACTIVITY_DURATION_HOURS",
        "ACTIVITY_DURATION_MINUTES",
    ]

    return activities[output_columns]


#######################################
# 4. Validate the activity reconstruction
#######################################


def validate_activity_reconstruction(
    trips: pd.DataFrame,
    activities: pd.DataFrame,
) -> pd.DataFrame:
    """
    Check that a person with N trips has N + 1 activities.
    """

    trip_counts = trips.groupby("ID").size().rename("N_TRIPS")

    activity_counts = activities.groupby("ID").size().rename("N_ACTIVITIES")

    validation = pd.concat(
        [
            trip_counts,
            activity_counts,
        ],
        axis=1,
    ).fillna(0)

    validation["N_TRIPS"] = validation["N_TRIPS"].astype(int)

    validation["N_ACTIVITIES"] = validation["N_ACTIVITIES"].astype(int)

    validation["EXPECTED_N_ACTIVITIES"] = validation["N_TRIPS"] + 1

    validation["VALID_ACTIVITY_COUNT"] = (
        validation["N_ACTIVITIES"] == validation["EXPECTED_N_ACTIVITIES"]
    )

    return validation.reset_index()


###############################################
# 5. Check first and last activity consistency
###############################################


def validate_first_last_activities(
    trips: pd.DataFrame,
    activities: pd.DataFrame,
) -> pd.DataFrame:
    """
    Validate the locations and purposes of first and last activities.

    Expected:
    - first activity location = first trip origin;
    - last activity location = last trip destination;
    - first activity purpose = 0 (home);
    - last activity purpose = last trip purpose.
    """

    ordered_trips = trips.sort_values(["ID", "TRIP_NUMBER"])

    first_trips = ordered_trips.groupby("ID", as_index=False).first()

    last_trips = ordered_trips.groupby("ID", as_index=False).last()

    ordered_activities = activities.sort_values(["ID", "ACTIVITY_ORDER"])

    first_activities = ordered_activities.groupby(
        "ID",
        as_index=False,
    ).first()

    last_activities = ordered_activities.groupby(
        "ID",
        as_index=False,
    ).last()

    validation = (
        first_trips[
            [
                "ID",
                "ORIGIN_SUBZONE",
                "TRIP_STARTTIME",
            ]
        ]
        .rename(
            columns={
                "ORIGIN_SUBZONE": ("EXPECTED_FIRST_SUBZONE"),
                "TRIP_STARTTIME": ("EXPECTED_FIRST_ENDTIME"),
            }
        )
        .merge(
            last_trips[
                [
                    "ID",
                    "DESTINATION_SUBZONE",
                    "TRIP_PURPOSE",
                    "TRIP_ENDTIME",
                ]
            ].rename(
                columns={
                    "DESTINATION_SUBZONE": ("EXPECTED_LAST_SUBZONE"),
                    "TRIP_PURPOSE": ("EXPECTED_LAST_PURPOSE"),
                    "TRIP_ENDTIME": ("EXPECTED_LAST_STARTTIME"),
                }
            ),
            on="ID",
            how="inner",
            validate="one_to_one",
        )
        .merge(
            first_activities[
                [
                    "ID",
                    "ACTIVITY_PURPOSE",
                    "ACTIVITY_SUBZONE",
                    "ACTIVITY_ENDTIME",
                ]
            ].rename(
                columns={
                    "ACTIVITY_PURPOSE": ("FIRST_ACTIVITY_PURPOSE"),
                    "ACTIVITY_SUBZONE": ("FIRST_ACTIVITY_SUBZONE"),
                    "ACTIVITY_ENDTIME": ("FIRST_ACTIVITY_ENDTIME"),
                }
            ),
            on="ID",
            how="inner",
            validate="one_to_one",
        )
        .merge(
            last_activities[
                [
                    "ID",
                    "ACTIVITY_PURPOSE",
                    "ACTIVITY_SUBZONE",
                    "ACTIVITY_STARTTIME",
                    "ACTIVITY_ENDTIME",
                ]
            ].rename(
                columns={
                    "ACTIVITY_PURPOSE": ("LAST_ACTIVITY_PURPOSE"),
                    "ACTIVITY_SUBZONE": ("LAST_ACTIVITY_SUBZONE"),
                    "ACTIVITY_STARTTIME": ("LAST_ACTIVITY_STARTTIME"),
                    "ACTIVITY_ENDTIME": ("LAST_ACTIVITY_ENDTIME"),
                }
            ),
            on="ID",
            how="inner",
            validate="one_to_one",
        )
    )

    validation["VALID_FIRST_LOCATION"] = (
        validation["FIRST_ACTIVITY_SUBZONE"] == validation["EXPECTED_FIRST_SUBZONE"]
    )

    validation["VALID_LAST_LOCATION"] = (
        validation["LAST_ACTIVITY_SUBZONE"] == validation["EXPECTED_LAST_SUBZONE"]
    )

    validation["VALID_FIRST_PURPOSE"] = validation["FIRST_ACTIVITY_PURPOSE"] == 0

    validation["VALID_LAST_PURPOSE"] = (
        validation["LAST_ACTIVITY_PURPOSE"] == validation["EXPECTED_LAST_PURPOSE"]
    )

    validation["VALID_FIRST_ENDTIME"] = (
        validation["FIRST_ACTIVITY_ENDTIME"] == validation["EXPECTED_FIRST_ENDTIME"]
    )

    validation["VALID_LAST_STARTTIME"] = (
        validation["LAST_ACTIVITY_STARTTIME"] == validation["EXPECTED_LAST_STARTTIME"]
    )

    validation["VALID_LAST_ENDTIME"] = validation["LAST_ACTIVITY_ENDTIME"] == 24

    return validation


###################################
# 6. Full trip-to-activity wrapper
###################################


def build_activity_schedule_pipeline(
    df: pd.DataFrame,
    max_trips: int = MAX_TRIPS,
    day_start_hour: float = 0.0,
    day_end_hour: float = 23.0,
    invalid_timing: Literal[
        "drop",
        "raise",
        "keep",
    ] = "drop",
) -> Dict[str, pd.DataFrame]:
    """
    Complete pipeline from wide trip schedules to activity schedules.
    """

    trips = reshape_trip(
        df=df,
        max_trips=max_trips,
    )

    activities = calculate_destination_activities(
        trips=trips,
        day_start_hour=day_start_hour,
        day_end_hour=day_end_hour,
        invalid_timing=invalid_timing,
    )

    activity_count_validation = validate_activity_reconstruction(
        trips=trips,
        activities=activities,
    )

    first_last_validation = validate_first_last_activities(
        trips=trips,
        activities=activities,
    )

    return {
        "trips": trips,
        "activities": activities,
        "activity_count_validation": (activity_count_validation),
        "first_last_validation": (first_last_validation),
    }


PURPOSE_LABELS = {
    0: "Returning Home",
    1: "Work/Commute",
    2: "Dining",
    3: "Shopping",
    4: "Education",
    5: "Other",
}


def summarize_earliest_activity_start(activities: pd.DataFrame) -> pd.DataFrame:
    """Return the earliest activity start time for every purpose code."""
    summary = (
        activities.assign(
            ACTIVITY_PURPOSE=pd.to_numeric(activities["ACTIVITY_PURPOSE"], errors="coerce")
        )
        .dropna(subset=["ACTIVITY_PURPOSE", "ACTIVITY_STARTTIME"])
        .assign(ACTIVITY_PURPOSE=lambda x: x["ACTIVITY_PURPOSE"].astype(int))
        .groupby("ACTIVITY_PURPOSE", as_index=False)
        .agg(EARLIEST_ACTIVITY_STARTTIME=("ACTIVITY_STARTTIME", "min"))
        .set_index("ACTIVITY_PURPOSE")
        .reindex(PURPOSE_LABELS)
        .rename_axis("ACTIVITY_PURPOSE")
        .reset_index()
    )
    summary["PURPOSE_LABEL"] = summary["ACTIVITY_PURPOSE"].map(PURPOSE_LABELS)
    return summary[["ACTIVITY_PURPOSE", "PURPOSE_LABEL", "EARLIEST_ACTIVITY_STARTTIME"]]


def main() -> None:
    """Load a wide schedule, reconstruct activities, validate, and save output."""


    for fz in ['0.1']:#['0.2', '0.4', '0.6']:
        print('**********************************')
        print(f'For fraction {fz}...')

        total_start_time = time.time()
        input_file = os.path.join(
            path_result,
            f'val_sim_tour_proposed_{fz}.csv'
            #path_data,
            #'val_data_true_tour.csv'
        )
        output_file = os.path.join(
            path_segregation,
            f"val_segregation_activities_{fz}.csv",
        )

        df_fus_tour_raw = pd.read_csv(input_file)
        # commend out if not hts
        #age_mapping = {0: 0, 1: 0, 2: 1, 3: 1, 4: 2,  5: 2, 6: 3, 7: 3}
        #df_fus_tour_raw["AGE"] = df_fus_tour_raw["AGE"].replace(age_mapping)


        pipeline_start_time = time.time()
        results = build_activity_schedule_pipeline(
            df=df_fus_tour_raw,
            max_trips=MAX_TRIPS,
            day_start_hour=0,
            day_end_hour=23,
            invalid_timing="drop",
        )
        pipeline_duration = time.time() - pipeline_start_time

        activities = results["activities"]
        earliest_start = summarize_earliest_activity_start(activities)
        del df_fus_tour_raw, results

        #if reg == "seoul":
        #    activities["ACTIVITY_SUBZONE"] = activities["ACTIVITY_SUBZONE"].astype(float).astype(int)

        os.makedirs(path_segregation, exist_ok=True)
        activities.to_csv(output_file, index=False)

        print(activities)
        print("\n--- Earliest Activity Start Time by Purpose ---")
        print(earliest_start.to_string(index=False, na_rep="No activity"))
        print("\n--- Execution Time ---")
        print(f"Pipeline execution time: {pipeline_duration:.4f} seconds")
        print(f"Total execution time: {time.time() - total_start_time:.4f} seconds")
        print(f"Saved activities to: {output_file}")

if __name__ == "__main__":
    main()

**********************************
For fraction 0.1...
          ID  AGE AGE_GROUP  GENDER GENDER_GROUP  INCOME INCOME_GROUP  \
0        562    1     age_1       0         male       0     income_0   
1        562    1     age_1       0         male       0     income_0   
2        562    1     age_1       0         male       0     income_0   
3        563    1     age_1       0         male       0     income_0   
4        563    1     age_1       0         male       0     income_0   
...      ...  ...       ...     ...          ...     ...          ...   
37339  15271    3     age_3       1       female       2     income_2   
37340  15271    3     age_3       1       female       2     income_2   
37341  15272    3     age_3       1       female       2     income_2   
37342  15272    3     age_3       1       female       2     income_2   
37343  15272    3     age_3       1       female       2     income_2   

       ACTIVITY_ORDER  ACTIVITY_PURPOSE ACTIVITY_SUBZONE  ACTIVITY_S

# Controlled experiment: Place segregation

In [ ]:
from __future__ import annotations

import gc
import time
from typing import Iterable, Optional

import numpy as np
import pandas as pd

# ============================================================
# 1. CONFIGURATION
# ============================================================

AGE_GROUPS = ["age_0", "age_1", "age_2", "age_3"]

GENDER_GROUPS = ["male", "female"]

INCOME_GROUPS = ["income_0", "income_1", "income_2"]

ATTRIBUTE_CONFIGURATIONS = [
    ("AGE", "AGE_GROUP", AGE_GROUPS),
    ("GENDER", "GENDER_GROUP", GENDER_GROUPS),
    ("INCOME", "INCOME_GROUP", INCOME_GROUPS),
]


# ============================================================
# 2. VALIDATION
# ============================================================


def validate_required_columns(
    df: pd.DataFrame,
    required_columns: Iterable[str],
    dataframe_name: str,
) -> None:
    required_columns = list(required_columns)
    missing = [c for c in required_columns if c not in df.columns]

    if missing:
        raise KeyError(f"{dataframe_name} is missing columns: {missing}")

    if df.empty:
        raise ValueError(f"{dataframe_name} is empty.")


# ============================================================
# 3. COMMON SAMPLE PREPARATION
# ============================================================


def prepare_attribute_activities(
    activities: pd.DataFrame,
    group_column: str,
    group_categories: Iterable,
    person_column: str = "ID",
    location_column: str = "ACTIVITY_SUBZONE",
    purpose_column: str = "ACTIVITY_PURPOSE",
    start_column: str = "ACTIVITY_STARTTIME",
    end_column: str = "ACTIVITY_ENDTIME",
    duration_hours_column: str = "ACTIVITY_DURATION_HOURS",
    activity_purposes: Optional[Iterable] = None,
    minimum_duration_hours: float = 0.0,
) -> pd.DataFrame:
    """Prepare one common sample for all segregation resolutions."""
    if minimum_duration_hours < 0:
        raise ValueError("minimum_duration_hours must be non-negative.")

    required = [
        person_column,
        group_column,
        location_column,
        purpose_column,
        start_column,
        end_column,
    ]
    validate_required_columns(activities, required, "activities")

    columns = required.copy()
    if duration_hours_column in activities.columns:
        columns.append(duration_hours_column)

    data = activities[columns].copy()

    for column in [start_column, end_column]:
        data[column] = pd.to_numeric(data[column], errors="coerce")

    if duration_hours_column in data.columns:
        data[duration_hours_column] = pd.to_numeric(data[duration_hours_column], errors="coerce")

    data = data.dropna(subset=required).copy()

    data = data.loc[
        data[start_column].ge(0)
        & data[start_column].lt(24)
        & data[end_column].ge(0)
        & data[end_column].le(23)
    ].copy()

    # Apply the study's inclusive duration definition consistently.
    inclusive_duration = data[end_column] - data[start_column] + 1.0
    valid_timing = data[end_column].ge(data[start_column])

    if duration_hours_column in data.columns:
        stored_duration = data[duration_hours_column].astype("float64")
        mismatch = stored_duration.notna() & ~np.isclose(
            stored_duration, inclusive_duration, rtol=0.0, atol=1e-10
        )
        if mismatch.any():
            examples = data.loc[
                mismatch,
                [start_column, end_column, duration_hours_column],
            ].head(10)
            raise ValueError(
                f"{duration_hours_column} is inconsistent with the inclusive "
                f"formula end - start + 1.\nExamples:\n{examples}"
            )

    data["ACTIVITY_DURATION_HOURS_CLEAN"] = inclusive_duration.astype("float64")

    # Every valid inclusive interval has a duration of at least one hour.
    data = data.loc[
        valid_timing
        & data["ACTIVITY_DURATION_HOURS_CLEAN"].gt(0)
        & data["ACTIVITY_DURATION_HOURS_CLEAN"].ge(minimum_duration_hours)
    ].copy()

    categories = list(group_categories)
    data = data.loc[data[group_column].isin(categories)].copy()

    if activity_purposes is not None:
        purposes = list(activity_purposes)
        data = data.loc[data[purpose_column].isin(purposes)].copy()

    if data.empty:
        raise ValueError(f"No valid activities remain for {group_column}.")

    group_counts = (
        data[[person_column, group_column]]
        .drop_duplicates()
        .groupby(person_column, observed=True)[group_column]
        .nunique()
    )
    inconsistent = group_counts.loc[group_counts > 1]

    if not inconsistent.empty:
        raise ValueError(
            f"Some people have multiple {group_column} values.\n"
            f"Examples:\n{inconsistent.head(10)}"
        )

    return data.reset_index(drop=True)


# ============================================================
# 4. POPULATION SHARES
# ============================================================


def calculate_population_shares(
    activities: pd.DataFrame,
    group_column: str,
    group_categories: Iterable,
    person_column: str = "ID",
) -> pd.Series:
    categories = list(group_categories)

    people = (
        activities[[person_column, group_column]].dropna().drop_duplicates(subset=[person_column])
    )

    counts = people[group_column].value_counts().reindex(categories, fill_value=0).astype("float64")

    total = counts.sum()
    if total <= 0:
        raise ValueError("The total population count is zero.")

    shares = counts / total
    shares.name = "POPULATION_SHARE"
    return shares


# ============================================================
# 5. HIGHEST-RESOLUTION ATTENDANCE TENSOR
# ============================================================


def calculate_place_purpose_time_attendance(
    activities: pd.DataFrame,
    group_column: str,
    group_categories: Iterable,
    location_column: str = "ACTIVITY_SUBZONE",
    purpose_column: str = "ACTIVITY_PURPOSE",
    start_column: str = "ACTIVITY_STARTTIME",
    end_column: str = "ACTIVITY_ENDTIME",
    time_column: str = "ACTIVITY_TIME_SLOT",
) -> pd.DataFrame:
    """Construct attendance under the inclusive end-hour convention."""
    categories = list(group_categories)

    starts = pd.to_numeric(activities[start_column], errors="coerce").astype("float64")
    durations = pd.to_numeric(activities["ACTIVITY_DURATION_HOURS_CLEAN"], errors="coerce").astype(
        "float64"
    )
    recorded_ends = pd.to_numeric(activities[end_column], errors="coerce").astype("float64")
    # The inclusive interval [start, end] is represented as [start, end + 1)
    # for overlap calculations. Valid hourly slots are exactly 0 through 23.
    exclusive_ends = recorded_ends + 1.0
    expected_durations = recorded_ends - starts + 1.0

    invalid = (
        starts.isna()
        | recorded_ends.isna()
        | starts.lt(0)
        | starts.ge(24)
        | durations.isna()
        | durations.le(0)
        | recorded_ends.lt(starts)
        | recorded_ends.gt(23)
        | ~np.isclose(durations, expected_durations, rtol=0.0, atol=1e-10)
    )

    if invalid.any():
        examples = activities.loc[invalid, [start_column, end_column]].head(10)
        raise ValueError("Invalid intervals remain after preparation.\n" f"Examples:\n{examples}")

    base = activities[[location_column, purpose_column, group_column]].copy()
    start_values = starts.to_numpy(dtype="float64")
    end_values = exclusive_ends.to_numpy(dtype="float64")
    parts = []

    for hour in range(24):
        overlap = np.minimum(end_values, hour + 1.0) - np.maximum(start_values, float(hour))
        active_mask = overlap > 0

        if not active_mask.any():
            continue

        active = base.loc[
            active_mask,
            [location_column, purpose_column, group_column],
        ].copy()
        active[time_column] = hour
        active["_PARTICIPATION_HOURS"] = overlap[active_mask].astype("float64")

        parts.append(
            active.groupby(
                [
                    location_column,
                    purpose_column,
                    time_column,
                    group_column,
                ],
                observed=True,
                sort=False,
            )["_PARTICIPATION_HOURS"].sum()
        )
        del active

    if not parts:
        raise ValueError("No occupied hourly slots were generated.")

    tensor_long = pd.concat(parts).groupby(level=[0, 1, 2, 3], sort=False).sum()

    del parts
    gc.collect()

    tensor = (
        tensor_long.unstack(fill_value=0.0)
        .reindex(columns=categories, fill_value=0.0)
        .astype("float64")
    )
    tensor.index.names = [location_column, purpose_column, time_column]

    del tensor_long
    gc.collect()
    return tensor


# ============================================================
# 6. LOWER-RESOLUTION MARGINALS
# ============================================================


def derive_attendance_marginals(
    place_purpose_time_attendance: pd.DataFrame,
    location_column: str = "ACTIVITY_SUBZONE",
    purpose_column: str = "ACTIVITY_PURPOSE",
    time_column: str = "ACTIVITY_TIME_SLOT",
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if place_purpose_time_attendance.empty:
        raise ValueError("place_purpose_time_attendance is empty.")

    place_purpose = (
        place_purpose_time_attendance.groupby(level=[location_column, purpose_column], sort=False)
        .sum()
        .astype("float64")
    )

    place_time = (
        place_purpose_time_attendance.groupby(level=[location_column, time_column], sort=False)
        .sum()
        .astype("float64")
    )

    place = (
        place_purpose_time_attendance.groupby(level=[location_column], sort=False)
        .sum()
        .astype("float64")
    )

    return place, place_purpose, place_time


# ============================================================
# 7. SEGREGATION
# ============================================================


def calculate_segregation_output(
    attendance: pd.DataFrame,
    population_shares: pd.Series,
    segregation_column: str,
) -> pd.DataFrame:
    if attendance.empty:
        raise ValueError("The attendance matrix is empty.")

    attendance = attendance.astype("float64")
    total_attendance = attendance.sum(axis=1)
    valid = total_attendance > 0

    attendance = attendance.loc[valid].copy()
    total_attendance = total_attendance.loc[valid]
    composition = attendance.div(total_attendance, axis=0)

    reference = population_shares.reindex(composition.columns).astype("float64")

    if reference.isna().any():
        raise ValueError("Population shares are missing for some groups.")

    if not np.isclose(reference.sum(), 1.0, rtol=0.0, atol=1e-12):
        raise ValueError("Population shares must sum to one.")

    denominator = 2.0 * (1.0 - reference.min())
    if denominator <= 0:
        raise ValueError("The segregation denominator is not positive.")

    segregation = (composition.sub(reference, axis=1).abs().sum(axis=1) / denominator).clip(
        lower=0.0, upper=1.0
    )

    output = composition.add_prefix("COMPOSITION_").copy()
    output[segregation_column] = segregation.astype("float64")
    output["TOTAL_ATTENDANCE_HOURS"] = total_attendance.astype("float64")
    output["N_GROUPS_PRESENT"] = (attendance > 0).sum(axis=1)

    return output.reset_index()


# ============================================================
# 8. FOUR MODELS FOR ONE ATTRIBUTE
# ============================================================


def calculate_four_models_for_attribute(
    activities: pd.DataFrame,
    group_column: str,
    group_categories: Iterable,
    activity_purposes: Optional[Iterable] = None,
    minimum_duration_hours: float = 0.0,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    data = prepare_attribute_activities(
        activities=activities,
        group_column=group_column,
        group_categories=group_categories,
        activity_purposes=activity_purposes,
        minimum_duration_hours=minimum_duration_hours,
    )

    population_shares = calculate_population_shares(
        activities=data,
        group_column=group_column,
        group_categories=group_categories,
    )

    place_purpose_time_attendance = calculate_place_purpose_time_attendance(
        activities=data,
        group_column=group_column,
        group_categories=group_categories,
    )

    place_attendance, place_purpose_attendance, place_time_attendance = derive_attendance_marginals(
        place_purpose_time_attendance
    )

    place = calculate_segregation_output(place_attendance, population_shares, "PLACE_SEGREGATION")
    place_purpose = calculate_segregation_output(
        place_purpose_attendance,
        population_shares,
        "PLACE_PURPOSE_SEGREGATION",
    )
    place_time = calculate_segregation_output(
        place_time_attendance,
        population_shares,
        "PLACE_TIME_SEGREGATION",
    )
    place_purpose_time = calculate_segregation_output(
        place_purpose_time_attendance,
        population_shares,
        "PLACE_PURPOSE_TIME_SEGREGATION",
    )

    del data, population_shares
    del place_attendance, place_purpose_attendance, place_time_attendance
    del place_purpose_time_attendance
    gc.collect()

    return place, place_purpose, place_time, place_purpose_time


# ============================================================
# 9. ALL ATTRIBUTES
# ============================================================


def calculate_all_segregation_models(
    activities: pd.DataFrame,
    activity_purposes: Optional[Iterable] = None,
    minimum_duration_hours: float = 0.0,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    place_results = []
    purpose_results = []
    time_results = []
    purpose_time_results = []

    for attribute_label, group_column, group_categories in ATTRIBUTE_CONFIGURATIONS:
        print(f"Calculating {attribute_label} segregation...")

        place, place_purpose, place_time, place_purpose_time = calculate_four_models_for_attribute(
            activities=activities,
            group_column=group_column,
            group_categories=group_categories,
            activity_purposes=activity_purposes,
            minimum_duration_hours=minimum_duration_hours,
        )

        place_results.append(place.assign(ATTRIBUTE=attribute_label))
        purpose_results.append(place_purpose.assign(ATTRIBUTE=attribute_label))
        time_results.append(place_time.assign(ATTRIBUTE=attribute_label))
        purpose_time_results.append(place_purpose_time.assign(ATTRIBUTE=attribute_label))

        del place, place_purpose, place_time, place_purpose_time
        gc.collect()

    outputs = (
        pd.concat(place_results, ignore_index=True, sort=False),
        pd.concat(purpose_results, ignore_index=True, sort=False),
        pd.concat(time_results, ignore_index=True, sort=False),
        pd.concat(purpose_time_results, ignore_index=True, sort=False),
    )

    del place_results, purpose_results, time_results, purpose_time_results
    gc.collect()
    return outputs


# ============================================================
# 10. WEIGHTED RESOLUTION COMPARISON
# ============================================================


def calculate_attendance_weighted_mean_score(
    dataframe: pd.DataFrame,
    score_column: str,
    output_score_column: str,
    place_column: str = "ACTIVITY_SUBZONE",
    attribute_column: str = "ATTRIBUTE",
    attendance_column: str = "TOTAL_ATTENDANCE_HOURS",
) -> pd.DataFrame:
    required = [place_column, attribute_column, score_column, attendance_column]
    validate_required_columns(dataframe, required, output_score_column)

    data = dataframe[required].copy()
    data[score_column] = pd.to_numeric(data[score_column], errors="coerce")
    data[attendance_column] = pd.to_numeric(data[attendance_column], errors="coerce")
    data = data.dropna(subset=required).copy()
    data = data.loc[data[attendance_column] > 0].copy()

    if data.empty:
        raise ValueError(f"No positive-attendance rows remain for {output_score_column}.")

    data["_WEIGHTED_SCORE"] = data[score_column] * data[attendance_column]

    grouped = (
        data.groupby([place_column, attribute_column], observed=True, sort=False)
        .agg(
            _WEIGHTED_SCORE_SUM=("_WEIGHTED_SCORE", "sum"),
            RESOLVED_TOTAL_ATTENDANCE_HOURS=(attendance_column, "sum"),
            N_RESOLVED_CONTEXTS=(score_column, "size"),
        )
        .reset_index()
    )

    grouped[output_score_column] = (
        grouped["_WEIGHTED_SCORE_SUM"] / grouped["RESOLVED_TOTAL_ATTENDANCE_HOURS"]
    )

    return grouped[
        [
            place_column,
            attribute_column,
            output_score_column,
            "RESOLVED_TOTAL_ATTENDANCE_HOURS",
            "N_RESOLVED_CONTEXTS",
        ]
    ]


def build_place_resolution_comparison(
    all_place_segregation: pd.DataFrame,
    all_place_purpose_segregation: pd.DataFrame,
    all_place_time_segregation: pd.DataFrame,
    all_place_purpose_time_segregation: pd.DataFrame,
    tolerance: float = 1e-10,
) -> pd.DataFrame:
    keys = ["ACTIVITY_SUBZONE", "ATTRIBUTE"]

    place = (
        all_place_segregation[keys + ["PLACE_SEGREGATION", "TOTAL_ATTENDANCE_HOURS"]]
        .rename(columns={"TOTAL_ATTENDANCE_HOURS": "PLACE_TOTAL_ATTENDANCE_HOURS"})
        .copy()
    )

    if place.duplicated(keys).any():
        raise ValueError("Duplicate place-attribute keys in place output.")

    purpose = calculate_attendance_weighted_mean_score(
        all_place_purpose_segregation,
        "PLACE_PURPOSE_SEGREGATION",
        "PURPOSE_RESOLVED_MEAN_SEGREGATION",
    ).rename(
        columns={
            "RESOLVED_TOTAL_ATTENDANCE_HOURS": "PURPOSE_TOTAL_ATTENDANCE_HOURS",
            "N_RESOLVED_CONTEXTS": "N_PURPOSE_CONTEXTS",
        }
    )

    time_resolved = calculate_attendance_weighted_mean_score(
        all_place_time_segregation,
        "PLACE_TIME_SEGREGATION",
        "TIME_RESOLVED_MEAN_SEGREGATION",
    ).rename(
        columns={
            "RESOLVED_TOTAL_ATTENDANCE_HOURS": "TIME_TOTAL_ATTENDANCE_HOURS",
            "N_RESOLVED_CONTEXTS": "N_TIME_CONTEXTS",
        }
    )

    purpose_time = calculate_attendance_weighted_mean_score(
        all_place_purpose_time_segregation,
        "PLACE_PURPOSE_TIME_SEGREGATION",
        "PURPOSE_TIME_RESOLVED_MEAN_SEGREGATION",
    ).rename(
        columns={
            "RESOLVED_TOTAL_ATTENDANCE_HOURS": ("PURPOSE_TIME_TOTAL_ATTENDANCE_HOURS"),
            "N_RESOLVED_CONTEXTS": "N_PURPOSE_TIME_CONTEXTS",
        }
    )

    comparison = (
        place.merge(purpose, on=keys, how="left", validate="one_to_one")
        .merge(time_resolved, on=keys, how="left", validate="one_to_one")
        .merge(purpose_time, on=keys, how="left", validate="one_to_one")
    )

    comparison["PURPOSE_RESOLUTION_GAP"] = (
        comparison["PURPOSE_RESOLVED_MEAN_SEGREGATION"] - comparison["PLACE_SEGREGATION"]
    )
    comparison["TIME_RESOLUTION_GAP"] = (
        comparison["TIME_RESOLVED_MEAN_SEGREGATION"] - comparison["PLACE_SEGREGATION"]
    )
    comparison["PURPOSE_TIME_RESOLUTION_GAP"] = (
        comparison["PURPOSE_TIME_RESOLVED_MEAN_SEGREGATION"] - comparison["PLACE_SEGREGATION"]
    )

    # The convexity result forms a partial order (a diamond), not a total chain:
    #
    #                         purpose-time
    #                         /          \
    #                    purpose        time
    #                         \          /
    #                              place
    #
    # Purpose-resolved and time-resolved segregation are not generally ordered.
    comparison["PURPOSE_TIME_MINUS_PURPOSE_GAP"] = (
        comparison["PURPOSE_TIME_RESOLVED_MEAN_SEGREGATION"]
        - comparison["PURPOSE_RESOLVED_MEAN_SEGREGATION"]
    )
    comparison["PURPOSE_TIME_MINUS_TIME_GAP"] = (
        comparison["PURPOSE_TIME_RESOLVED_MEAN_SEGREGATION"]
        - comparison["TIME_RESOLVED_MEAN_SEGREGATION"]
    )

    comparison["PLACE_LE_PURPOSE_RESOLVED"] = comparison["PURPOSE_RESOLUTION_GAP"] >= -tolerance
    comparison["PLACE_LE_TIME_RESOLVED"] = comparison["TIME_RESOLUTION_GAP"] >= -tolerance
    comparison["PLACE_LE_PURPOSE_TIME_RESOLVED"] = (
        comparison["PURPOSE_TIME_RESOLUTION_GAP"] >= -tolerance
    )
    comparison["PURPOSE_RESOLVED_LE_PURPOSE_TIME_RESOLVED"] = (
        comparison["PURPOSE_TIME_MINUS_PURPOSE_GAP"] >= -tolerance
    )
    comparison["TIME_RESOLVED_LE_PURPOSE_TIME_RESOLVED"] = (
        comparison["PURPOSE_TIME_MINUS_TIME_GAP"] >= -tolerance
    )

    comparison["PURPOSE_ATTENDANCE_CONSISTENT"] = np.isclose(
        comparison["PLACE_TOTAL_ATTENDANCE_HOURS"],
        comparison["PURPOSE_TOTAL_ATTENDANCE_HOURS"],
        rtol=1e-10,
        atol=1e-9,
    )
    comparison["TIME_ATTENDANCE_CONSISTENT"] = np.isclose(
        comparison["PLACE_TOTAL_ATTENDANCE_HOURS"],
        comparison["TIME_TOTAL_ATTENDANCE_HOURS"],
        rtol=1e-10,
        atol=1e-9,
    )
    comparison["PURPOSE_TIME_ATTENDANCE_CONSISTENT"] = np.isclose(
        comparison["PLACE_TOTAL_ATTENDANCE_HOURS"],
        comparison["PURPOSE_TIME_TOTAL_ATTENDANCE_HOURS"],
        rtol=1e-10,
        atol=1e-9,
    )

    return comparison.sort_values(keys).reset_index(drop=True)


def summarize_place_resolution_comparison(
    comparison: pd.DataFrame,
) -> pd.DataFrame:
    required = [
        "ATTRIBUTE", "ACTIVITY_SUBZONE", "PLACE_TOTAL_ATTENDANCE_HOURS",
        "PLACE_SEGREGATION", "PURPOSE_RESOLVED_MEAN_SEGREGATION",
        "TIME_RESOLVED_MEAN_SEGREGATION", "PURPOSE_TIME_RESOLVED_MEAN_SEGREGATION",
        "PURPOSE_RESOLUTION_GAP", "TIME_RESOLUTION_GAP", "PURPOSE_TIME_RESOLUTION_GAP",
        "PURPOSE_TIME_MINUS_PURPOSE_GAP", "PURPOSE_TIME_MINUS_TIME_GAP",
        "PLACE_LE_PURPOSE_RESOLVED", "PLACE_LE_TIME_RESOLVED",
        "PLACE_LE_PURPOSE_TIME_RESOLVED", "PURPOSE_RESOLVED_LE_PURPOSE_TIME_RESOLVED",
        "TIME_RESOLVED_LE_PURPOSE_TIME_RESOLVED",
    ]
    validate_required_columns(comparison, required, "comparison")
    data = comparison[required].dropna().copy()
    data = data.loc[data["PLACE_TOTAL_ATTENDANCE_HOURS"] > 0].copy()
    score_columns = [
        "PLACE_SEGREGATION", "PURPOSE_RESOLVED_MEAN_SEGREGATION",
        "TIME_RESOLVED_MEAN_SEGREGATION", "PURPOSE_TIME_RESOLVED_MEAN_SEGREGATION",
    ]
    for column in score_columns:
        data[f"_WEIGHTED_{column}"] = data[column] * data["PLACE_TOTAL_ATTENDANCE_HOURS"]

    summary = (
        data.groupby("ATTRIBUTE", observed=True, sort=False)
        .agg(
            N_PLACES=("ACTIVITY_SUBZONE", "nunique"),
            TOTAL_ATTENDANCE_HOURS=("PLACE_TOTAL_ATTENDANCE_HOURS", "sum"),
            MEAN_PLACE_SEGREGATION=("PLACE_SEGREGATION", "mean"),
            MEAN_PURPOSE_RESOLVED_SEGREGATION=("PURPOSE_RESOLVED_MEAN_SEGREGATION", "mean"),
            MEAN_TIME_RESOLVED_SEGREGATION=("TIME_RESOLVED_MEAN_SEGREGATION", "mean"),
            MEAN_PURPOSE_TIME_RESOLVED_SEGREGATION=(
                "PURPOSE_TIME_RESOLVED_MEAN_SEGREGATION",
                "mean",
            ),
            MEAN_PURPOSE_GAP=("PURPOSE_RESOLUTION_GAP", "mean"),
            MEAN_TIME_GAP=("TIME_RESOLUTION_GAP", "mean"),
            MEAN_PURPOSE_TIME_GAP=("PURPOSE_TIME_RESOLUTION_GAP", "mean"),
            MIN_PURPOSE_GAP=("PURPOSE_RESOLUTION_GAP", "min"),
            MIN_TIME_GAP=("TIME_RESOLUTION_GAP", "min"),
            MIN_PURPOSE_TIME_GAP=("PURPOSE_TIME_RESOLUTION_GAP", "min"),
            MIN_PURPOSE_TIME_MINUS_PURPOSE_GAP=("PURPOSE_TIME_MINUS_PURPOSE_GAP", "min"),
            MIN_PURPOSE_TIME_MINUS_TIME_GAP=("PURPOSE_TIME_MINUS_TIME_GAP", "min"),
            SHARE_PLACE_LE_PURPOSE=("PLACE_LE_PURPOSE_RESOLVED", "mean"),
            SHARE_PLACE_LE_TIME=("PLACE_LE_TIME_RESOLVED", "mean"),
            SHARE_PLACE_LE_PURPOSE_TIME=("PLACE_LE_PURPOSE_TIME_RESOLVED", "mean"),
            SHARE_PURPOSE_LE_PURPOSE_TIME=(
                "PURPOSE_RESOLVED_LE_PURPOSE_TIME_RESOLVED", "mean"
            ),
            SHARE_TIME_LE_PURPOSE_TIME=("TIME_RESOLVED_LE_PURPOSE_TIME_RESOLVED", "mean"),
            _W_PLACE=("_WEIGHTED_PLACE_SEGREGATION", "sum"),
            _W_PURPOSE=("_WEIGHTED_PURPOSE_RESOLVED_MEAN_SEGREGATION", "sum"),
            _W_TIME=("_WEIGHTED_TIME_RESOLVED_MEAN_SEGREGATION", "sum"),
            _W_PURPOSE_TIME=("_WEIGHTED_PURPOSE_TIME_RESOLVED_MEAN_SEGREGATION", "sum"),
        )
        .reset_index()
    )
    summary["ATTENDANCE_WEIGHTED_PLACE_SEGREGATION"] = (
        summary["_W_PLACE"] / summary["TOTAL_ATTENDANCE_HOURS"]
    )
    summary["ATTENDANCE_WEIGHTED_PURPOSE_SEGREGATION"] = (
        summary["_W_PURPOSE"] / summary["TOTAL_ATTENDANCE_HOURS"]
    )
    summary["ATTENDANCE_WEIGHTED_TIME_SEGREGATION"] = (
        summary["_W_TIME"] / summary["TOTAL_ATTENDANCE_HOURS"]
    )
    summary["ATTENDANCE_WEIGHTED_PURPOSE_TIME_SEGREGATION"] = (
        summary["_W_PURPOSE_TIME"] / summary["TOTAL_ATTENDANCE_HOURS"]
    )
    return summary.drop(columns=["_W_PLACE", "_W_PURPOSE", "_W_TIME", "_W_PURPOSE_TIME"])


# ============================================================
# 11. CLEANLINESS DIAGNOSTICS
# ============================================================


def diagnose_activity_interval_cleanliness(
    activities: pd.DataFrame,
    start_column: str = "ACTIVITY_STARTTIME",
    end_column: str = "ACTIVITY_ENDTIME",
    tolerance: float = 1e-10,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    validate_required_columns(activities, [start_column, end_column], "activities")

    data = activities.copy()
    data[start_column] = pd.to_numeric(data[start_column], errors="coerce")
    data[end_column] = pd.to_numeric(data[end_column], errors="coerce")

    start = data[start_column]
    end = data[end_column]

    data["_IS_MISSING_TIME"] = start.isna() | end.isna()
    data["_START_OUTSIDE_DAY"] = start.notna() & (start.lt(0) | start.ge(24))
    data["_END_OUTSIDE_DAY"] = end.notna() & (end.lt(0) | end.gt(23))
    data["_SAME_START_END"] = (
        start.notna() & end.notna() & np.isclose(start, end, rtol=0.0, atol=1e-12)
    )
    data["_NEGATIVE_DURATION"] = start.notna() & end.notna() & end.lt(start)

    # Inclusive duration: an activity with start == end lasts one hour.
    direct_duration = end - start + 1.0
    start_values = start.to_numpy(dtype="float64")
    end_values = end.to_numpy(dtype="float64")
    valid_numeric = np.isfinite(start_values) & np.isfinite(end_values)
    overlap_sum = np.zeros(len(data), dtype="float64")

    # Use the 24 valid inclusive hourly slots, labelled 0 through 23.
    exclusive_end_values = end_values + 1.0
    for hour in range(24):
        overlap = np.minimum(exclusive_end_values, hour + 1.0) - np.maximum(
            start_values, float(hour)
        )
        overlap = np.where(valid_numeric, np.maximum(overlap, 0.0), 0.0)
        overlap_sum += overlap

    data["_DIRECT_DURATION"] = direct_duration
    data["_HOURLY_OVERLAP_SUM"] = overlap_sum
    data["_OVERLAP_ERROR"] = overlap_sum - direct_duration

    valid_within_day = (
        ~data["_IS_MISSING_TIME"]
        & ~data["_START_OUTSIDE_DAY"]
        & ~data["_END_OUTSIDE_DAY"]
        & ~data["_NEGATIVE_DURATION"]
    )
    data["_OVERLAP_IDENTITY_FAILS"] = valid_within_day & data["_OVERLAP_ERROR"].abs().gt(tolerance)

    summary = pd.DataFrame(
        {
            "CHECK": [
                "Missing time",
                "Start outside [0,24)",
                "End outside [0,23]",
                "Same start and end (one inclusive hour)",
                "Negative duration",
                "Hourly-overlap identity failure",
            ],
            "N_ROWS": [
                int(data["_IS_MISSING_TIME"].sum()),
                int(data["_START_OUTSIDE_DAY"].sum()),
                int(data["_END_OUTSIDE_DAY"].sum()),
                int(data["_SAME_START_END"].sum()),
                int(data["_NEGATIVE_DURATION"].sum()),
                int(data["_OVERLAP_IDENTITY_FAILS"].sum()),
            ],
        }
    )

    issue_columns = [
        "_IS_MISSING_TIME",
        "_START_OUTSIDE_DAY",
        "_END_OUTSIDE_DAY",
        "_NEGATIVE_DURATION",
        "_OVERLAP_IDENTITY_FAILS",
    ]
    problematic = data.loc[data[issue_columns].any(axis=1)].copy()
    return summary, problematic


# ============================================================
# 12. OUTPUT VALIDATION
# ============================================================


def validate_final_outputs(
    all_place_segregation: pd.DataFrame,
    all_place_purpose_segregation: pd.DataFrame,
    all_place_time_segregation: pd.DataFrame,
    all_place_purpose_time_segregation: pd.DataFrame,
    tolerance: float = 1e-10,
) -> pd.DataFrame:
    specifications = [
        (
            "PLACE",
            all_place_segregation,
            "PLACE_SEGREGATION",
            ["ACTIVITY_SUBZONE", "ATTRIBUTE"],
        ),
        (
            "PLACE_PURPOSE",
            all_place_purpose_segregation,
            "PLACE_PURPOSE_SEGREGATION",
            ["ACTIVITY_SUBZONE", "ACTIVITY_PURPOSE", "ATTRIBUTE"],
        ),
        (
            "PLACE_TIME",
            all_place_time_segregation,
            "PLACE_TIME_SEGREGATION",
            ["ACTIVITY_SUBZONE", "ACTIVITY_TIME_SLOT", "ATTRIBUTE"],
        ),
        (
            "PLACE_PURPOSE_TIME",
            all_place_purpose_time_segregation,
            "PLACE_PURPOSE_TIME_SEGREGATION",
            [
                "ACTIVITY_SUBZONE",
                "ACTIVITY_PURPOSE",
                "ACTIVITY_TIME_SLOT",
                "ATTRIBUTE",
            ],
        ),
    ]

    rows = []
    for model, dataframe, score_column, key_columns in specifications:
        required = key_columns + [
            score_column,
            "TOTAL_ATTENDANCE_HOURS",
            "N_GROUPS_PRESENT",
        ]
        validate_required_columns(dataframe, required, model)

        scores = pd.to_numeric(dataframe[score_column], errors="coerce")
        attendance = pd.to_numeric(dataframe["TOTAL_ATTENDANCE_HOURS"], errors="coerce")

        rows.append(
            {
                "MODEL": model,
                "N_ROWS": len(dataframe),
                "N_DUPLICATE_KEYS": int(dataframe.duplicated(subset=key_columns).sum()),
                "SCORES_NONMISSING": bool(scores.notna().all()),
                "SCORES_IN_0_1": bool(scores.between(-tolerance, 1.0 + tolerance).all()),
                "POSITIVE_ATTENDANCE": bool(attendance.gt(0).all()),
            }
        )

    return pd.DataFrame(rows)


# ============================================================
# 13. RUN
# ============================================================

start_time = time.time()

for fz in ['0.1']:#['0.2', '0.4', '0.6', 'true']:

    print('**********************************')
    print(f'For fraction {fz}...')


    activities = pd.read_csv(
        path_segregation + f"val_segregation_activities_{fz}.csv"

    )

    activity_cleanliness_summary, problematic_activities = diagnose_activity_interval_cleanliness(
        activities
    )

    print("\nRaw activity-time cleanliness")
    print(activity_cleanliness_summary.to_string(index=False))

    (
        all_place_segregation,
        all_place_purpose_segregation,
        all_place_time_segregation,
        all_place_purpose_time_segregation,
    ) = calculate_all_segregation_models(
        activities=activities,
        activity_purposes=None,
        minimum_duration_hours=0.0,
    )

    all_place_resolution_comparison = build_place_resolution_comparison(
        all_place_segregation=all_place_segregation,
        all_place_purpose_segregation=all_place_purpose_segregation,
        all_place_time_segregation=all_place_time_segregation,
        all_place_purpose_time_segregation=all_place_purpose_time_segregation,
        tolerance=1e-10,
    )

    place_resolution_summary = summarize_place_resolution_comparison(all_place_resolution_comparison)

    validation = validate_final_outputs(
        all_place_segregation,
        all_place_purpose_segregation,
        all_place_time_segregation,
        all_place_purpose_time_segregation,
    )

    print("\nOutput validation")
    print(validation.to_string(index=False))

    print("\nPlace-resolution summary")
    print(
        place_resolution_summary.to_string(
            index=False,
            float_format=lambda value: f"{value:.12g}",
        )
    )

    inequality_checks = [
        "PLACE_LE_PURPOSE_RESOLVED",
        "PLACE_LE_TIME_RESOLVED",
        "PLACE_LE_PURPOSE_TIME_RESOLVED",
        "PURPOSE_RESOLVED_LE_PURPOSE_TIME_RESOLVED",
        "TIME_RESOLVED_LE_PURPOSE_TIME_RESOLVED",
    ]

    attendance_checks = [
        "PURPOSE_ATTENDANCE_CONSISTENT",
        "TIME_ATTENDANCE_CONSISTENT",
        "PURPOSE_TIME_ATTENDANCE_CONSISTENT",
    ]

    print("\nConvexity inequality checks")
    print(all_place_resolution_comparison[inequality_checks].all().to_string())

    print("\nAttendance conservation checks")
    print(all_place_resolution_comparison[attendance_checks].all().to_string())

    print("\nMinimum observed gaps")
    print(
        all_place_resolution_comparison[
            [
                "PURPOSE_RESOLUTION_GAP",
                "TIME_RESOLUTION_GAP",
                "PURPOSE_TIME_RESOLUTION_GAP",
                "PURPOSE_TIME_MINUS_PURPOSE_GAP",
                "PURPOSE_TIME_MINUS_TIME_GAP",
            ]
        ]
        .min()
        .to_string()
    )

    violations = all_place_resolution_comparison.loc[
        ~all_place_resolution_comparison[inequality_checks].all(axis=1)
    ].copy()

    if violations.empty:
        print("\nNo material convexity violations were detected.")
    else:
        print("\nRows with apparent convexity violations")
        print(
            violations[
                [
                    "ACTIVITY_SUBZONE",
                    "ATTRIBUTE",
                    "PLACE_SEGREGATION",
                    "PURPOSE_RESOLVED_MEAN_SEGREGATION",
                    "TIME_RESOLVED_MEAN_SEGREGATION",
                    "PURPOSE_TIME_RESOLVED_MEAN_SEGREGATION",
                    "PURPOSE_RESOLUTION_GAP",
                    "TIME_RESOLUTION_GAP",
                    "PURPOSE_TIME_RESOLUTION_GAP",
                    "PURPOSE_TIME_MINUS_PURPOSE_GAP",
                    "PURPOSE_TIME_MINUS_TIME_GAP",
                    "PLACE_TOTAL_ATTENDANCE_HOURS",
                    "PURPOSE_TOTAL_ATTENDANCE_HOURS",
                    "TIME_TOTAL_ATTENDANCE_HOURS",
                    "PURPOSE_TIME_TOTAL_ATTENDANCE_HOURS",
                ]
            ]
            .sort_values(
                [
                    "PURPOSE_RESOLUTION_GAP",
                    "TIME_RESOLUTION_GAP",
                    "PURPOSE_TIME_RESOLUTION_GAP",
                    "PURPOSE_TIME_MINUS_PURPOSE_GAP",
                    "PURPOSE_TIME_MINUS_TIME_GAP",
                ]
            )
            .head(50)
            .to_string(index=False)
        )

    all_place_segregation.to_csv(
        path_segregation + f"val_all_place_segregation_{fz}.csv",
        index=False
    )
    all_place_purpose_segregation.to_csv(
        path_segregation + f"val_all_place_purpose_segregation_{fz}.csv",
        index=False,
    )
    all_place_time_segregation.to_csv(
        path_segregation + f"val_all_place_time_segregation_{fz}.csv",
        index=False,
    )
    all_place_purpose_time_segregation.to_csv(
        path_segregation + f"val_all_place_purpose_time_segregation_{fz}.csv",
        index=False,
    )


    end_time = time.time()
    print(f"\nTotal runtime: {end_time - start_time:.4f} seconds")

# Retained analytical outputs:
# 1. all_place_segregation
# 2. all_place_purpose_segregation
# 3. all_place_time_segregation
# 4. all_place_purpose_time_segregation
# 5. all_place_resolution_comparison
# 6. place_resolution_summary
#
# Retained diagnostics:
# 7. activity_cleanliness_summary
# 8. problematic_activities
# 9. validation
# 10. violations

**********************************
For fraction 0.1...

Raw activity-time cleanliness
                                  CHECK  N_ROWS
                           Missing time       0
                   Start outside [0,24)       0
                     End outside [0,23]       0
Same start and end (one inclusive hour)    3502
                      Negative duration       0
        Hourly-overlap identity failure       0
Calculating AGE segregation...
Calculating GENDER segregation...
Calculating INCOME segregation...

Output validation
             MODEL  N_ROWS  N_DUPLICATE_KEYS  SCORES_NONMISSING  SCORES_IN_0_1  POSITIVE_ATTENDANCE
             PLACE     897                 0               True           True                 True
     PLACE_PURPOSE    4137                 0               True           True                 True
        PLACE_TIME   20982                 0               True           True                 True
PLACE_PURPOSE_TIME   52437                 0               T

# Controlled experiment: Individual segregation

In [ ]:
from __future__ import annotations

import gc
from typing import Iterable
import os
import time
import warnings

import numpy as np
import pandas as pd

AGE_GROUPS = ["age_0", "age_1", "age_2", "age_3"]

GENDER_GROUPS = ["male", "female"]

INCOME_GROUPS = ["income_0", "income_1", "income_2"]

ATTRIBUTE_CONFIGURATIONS = [
    ("AGE", "AGE_GROUP", AGE_GROUPS),
    ("GENDER", "GENDER_GROUP", GENDER_GROUPS),
    ("INCOME", "INCOME_GROUP", INCOME_GROUPS),
]


def canonicalize_subzone(values: pd.Series) -> pd.Series:
    """Create stable string keys while preserving alphanumeric subzones."""
    result = values.astype("string").str.strip()
    result = result.mask(result.str.lower().isin(["", "nan", "none", "null", "<na>"]))
    numeric = pd.to_numeric(result, errors="coerce")
    integer_like = (
        numeric.notna() & np.isfinite(numeric) & np.isclose(numeric % 1, 0.0, rtol=0.0, atol=1e-12)
    )
    result.loc[integer_like] = numeric.loc[integer_like].astype("int64").astype("string")
    return result


def validate_required_columns(df: pd.DataFrame, columns: Iterable[str], name: str) -> None:
    columns = list(columns)
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing columns: {missing}")
    if df.empty:
        raise ValueError(f"{name} is empty.")


def prepare_activity_sample(
    activities: pd.DataFrame,
    group_column: str,
    group_categories: Iterable,
    person_column: str = "ID",
) -> pd.DataFrame:
    """Validate and prepare all activity rows without filtering any row."""
    required = [
        person_column,
        group_column,
        "ACTIVITY_SUBZONE",
        "ACTIVITY_PURPOSE",
        "ACTIVITY_STARTTIME",
        "ACTIVITY_ENDTIME",
        "ACTIVITY_DURATION_HOURS",
    ]
    validate_required_columns(activities, required, "activities")

    data = activities[required].copy()
    data["ACTIVITY_SUBZONE"] = canonicalize_subzone(data["ACTIVITY_SUBZONE"])
    data["ACTIVITY_STARTTIME"] = pd.to_numeric(
        data["ACTIVITY_STARTTIME"], errors="coerce", downcast="float"
    )
    data["ACTIVITY_ENDTIME"] = pd.to_numeric(
        data["ACTIVITY_ENDTIME"], errors="coerce", downcast="float"
    )
    data["ACTIVITY_DURATION_HOURS"] = pd.to_numeric(
        data["ACTIVITY_DURATION_HOURS"], errors="coerce", downcast="float"
    )
    valid = data[required].notna().all(axis=1)
    inclusive_duration = data["ACTIVITY_ENDTIME"] - data["ACTIVITY_STARTTIME"] + 1.0
    duration_matches = np.isclose(
        data["ACTIVITY_DURATION_HOURS"], inclusive_duration, rtol=0.0, atol=1e-10
    )
    valid &= data["ACTIVITY_STARTTIME"].ge(0) & data["ACTIVITY_STARTTIME"].lt(24)
    valid &= data["ACTIVITY_ENDTIME"].ge(data["ACTIVITY_STARTTIME"])
    valid &= data["ACTIVITY_ENDTIME"].le(23)
    valid &= data["ACTIVITY_DURATION_HOURS"].gt(0) & duration_matches
    valid &= data[group_column].isin(list(group_categories))
    if not valid.all():
        invalid = data.loc[~valid, required].head(20)
        raise ValueError(
            f"Found {(~valid).sum():,} invalid {group_column} activity rows. "
            "No rows were discarded. ACTIVITY_DURATION_HOURS must equal "
            "ACTIVITY_ENDTIME - ACTIVITY_STARTTIME + 1. Examples:\n"
            f"{invalid.to_string(index=False)}"
        )

    person_group = data[[person_column, group_column]].drop_duplicates()
    bad = person_group.groupby(person_column, observed=True)[group_column].nunique()
    if (bad > 1).any():
        raise ValueError(f"Some people have multiple {group_column} values.")

    return data.reset_index(drop=True)


def prepare_common_activity_cohort(
    activities: pd.DataFrame,
    person_column: str = "ID",
) -> pd.DataFrame:
    """Validate and retain every schedule row and ID for every attribute."""
    group_rules = {
        group_column: list(categories) for _, group_column, categories in ATTRIBUTE_CONFIGURATIONS
    }
    activity_columns = [
        "ACTIVITY_SUBZONE",
        "ACTIVITY_PURPOSE",
        "ACTIVITY_STARTTIME",
        "ACTIVITY_ENDTIME",
        "ACTIVITY_DURATION_HOURS",
    ]
    required = [person_column, *group_rules, *activity_columns]
    validate_required_columns(activities, required, "activities")
    if activities[person_column].isna().any():
        raise ValueError("activities contains missing ID values.")

    data = activities[required].copy()
    data["ACTIVITY_SUBZONE"] = canonicalize_subzone(data["ACTIVITY_SUBZONE"])
    data["ACTIVITY_STARTTIME"] = pd.to_numeric(
        data["ACTIVITY_STARTTIME"], errors="coerce", downcast="float"
    )
    data["ACTIVITY_ENDTIME"] = pd.to_numeric(
        data["ACTIVITY_ENDTIME"], errors="coerce", downcast="float"
    )
    data["ACTIVITY_DURATION_HOURS"] = pd.to_numeric(
        data["ACTIVITY_DURATION_HOURS"], errors="coerce", downcast="float"
    )

    demographic_valid = data[[person_column, *group_rules]].notna().all(axis=1)
    for group_column, categories in group_rules.items():
        demographic_valid &= data[group_column].isin(categories)
    if not demographic_valid.all():
        invalid = data.loc[~demographic_valid, [person_column, *group_rules]]
        raise ValueError(
            f"Found {len(invalid):,} rows with invalid demographic groups. "
            "Check the configured group lists. Examples:\n"
            f"{invalid.head(20).to_string(index=False)}"
        )

    activity_valid = data[[person_column, *activity_columns]].notna().all(axis=1)
    inclusive_duration = data["ACTIVITY_ENDTIME"] - data["ACTIVITY_STARTTIME"] + 1.0
    duration_matches = np.isclose(
        data["ACTIVITY_DURATION_HOURS"], inclusive_duration, rtol=0.0, atol=1e-10
    )
    activity_valid &= data["ACTIVITY_STARTTIME"].ge(0)
    activity_valid &= data["ACTIVITY_STARTTIME"].lt(24)
    activity_valid &= data["ACTIVITY_ENDTIME"].ge(data["ACTIVITY_STARTTIME"])
    activity_valid &= data["ACTIVITY_ENDTIME"].le(23)
    activity_valid &= data["ACTIVITY_DURATION_HOURS"].gt(0) & duration_matches
    if not activity_valid.all():
        invalid_columns = [person_column, *activity_columns]
        invalid = data.loc[~activity_valid, invalid_columns].head(20)
        raise ValueError(
            f"Found {(~activity_valid).sum():,} invalid activity rows. No rows or IDs "
            "were discarded. ACTIVITY_DURATION_HOURS must be positive and equal "
            "ACTIVITY_ENDTIME - ACTIVITY_STARTTIME + 1. "
            "Examples:\n"
            f"{invalid.to_string(index=False)}"
        )

    retained_ids = pd.Index(data[person_column].unique())

    for group_column in group_rules:
        counts = data.groupby(person_column, observed=True)[group_column].nunique()
        if (counts > 1).any():
            bad_ids = counts.index[counts > 1][:20].tolist()
            raise ValueError(
                f"Some people have multiple {group_column} values. " f"Example IDs: {bad_ids}"
            )

    print(
        f"Common activity cohort: {len(retained_ids):,} unique IDs; "
        f"all {len(data):,} activity rows retained"
    )
    return data.reset_index(drop=True)


def calculate_population_shares(
    data: pd.DataFrame,
    group_column: str,
    group_categories: Iterable,
    person_column: str = "ID",
) -> pd.Series:
    categories = list(group_categories)
    people = data[[person_column, group_column]].drop_duplicates(subset=[person_column])
    counts = people[group_column].value_counts().reindex(categories, fill_value=0).astype("float64")
    if counts.sum() <= 0:
        raise ValueError("Population count is zero.")
    return counts / counts.sum()


def _prepare_context_lookup(
    context_result: pd.DataFrame,
    context_columns: list[str],
    categories: list[str],
) -> tuple[pd.DataFrame, list[str]]:
    comp_cols = [f"COMPOSITION_{g}" for g in categories]
    validate_required_columns(context_result, context_columns, "context_result")

    # A composition column can be absent when that category has zero members in
    # the dataframe used to create/pivot the saved context table. Such an absent
    # category has composition zero in every context; add it explicitly so all
    # models use the same configured category vector and column order.
    present_comp_cols = [c for c in comp_cols if c in context_result.columns]
    if not present_comp_cols:
        available = [c for c in context_result.columns if c.startswith("COMPOSITION_")]
        raise KeyError(
            "context_result contains none of the expected composition columns "
            f"{comp_cols}. Available composition columns: {available}"
        )

    lookup = context_result[[*context_columns, *present_comp_cols]].copy()
    if "ACTIVITY_SUBZONE" in context_columns:
        lookup["ACTIVITY_SUBZONE"] = canonicalize_subzone(lookup["ACTIVITY_SUBZONE"])
        lookup = lookup.dropna(subset=["ACTIVITY_SUBZONE"])
    missing_comp_cols = [c for c in comp_cols if c not in lookup.columns]
    for c in missing_comp_cols:
        lookup[c] = 0.0
    if missing_comp_cols:
        warnings.warn(
            "The following configured categories are absent from context_result "
            f"and are treated as zero composition: {missing_comp_cols}",
            RuntimeWarning,
            stacklevel=2,
        )

    lookup = lookup[[*context_columns, *comp_cols]]
    if lookup.duplicated(context_columns).any():
        examples = lookup.loc[lookup.duplicated(context_columns, keep=False), context_columns].head(
            20
        )
        raise ValueError(
            f"Duplicate context keys after normalization: {context_columns}\n"
            f"Examples:\n{examples.to_string(index=False)}"
        )

    for c in comp_cols:
        lookup[c] = pd.to_numeric(lookup[c], errors="coerce").astype("float64")

    if lookup[comp_cols].isna().any().any():
        raise ValueError("Missing context compositions.")

    if not np.allclose(
        lookup[comp_cols].sum(axis=1).to_numpy(),
        1.0,
        rtol=0.0,
        atol=1e-9,
    ):
        raise ValueError("Some context compositions do not sum to one.")

    return lookup, comp_cols


def _initialize_accumulators(person_ids: pd.Index, n_groups: int) -> dict:
    return {
        "person_ids": person_ids,
        "person_to_code": pd.Series(
            np.arange(len(person_ids), dtype=np.int64),
            index=person_ids,
        ),
        "exposure_numerator": np.zeros((len(person_ids), n_groups), dtype=np.float64),
        "matched": np.zeros(len(person_ids), dtype=np.float64),
        "input": np.zeros(len(person_ids), dtype=np.float64),
        "contexts": np.zeros(len(person_ids), dtype=np.int64),
    }


def _accumulate_chunk(
    attendance_chunk: pd.DataFrame,
    context_lookup: pd.DataFrame,
    context_columns: list[str],
    comp_cols: list[str],
    accumulators: dict,
    person_column: str = "ID",
    attendance_column: str = "INDIVIDUAL_ATTENDANCE_HOURS",
) -> None:
    """Merge only one chunk and accumulate exposure numerators with np.add.at."""
    if attendance_chunk.empty:
        return

    chunk = attendance_chunk[[person_column, *context_columns, attendance_column]].copy()
    chunk[attendance_column] = pd.to_numeric(chunk[attendance_column], errors="coerce").astype(
        "float64"
    )
    chunk = chunk.dropna(subset=[person_column, *context_columns, attendance_column])
    chunk = chunk.loc[chunk[attendance_column] > 0]
    if chunk.empty:
        return

    person_codes = accumulators["person_to_code"].reindex(chunk[person_column]).to_numpy()
    if np.isnan(person_codes).any():
        raise ValueError("Attendance contains an unknown person ID.")
    person_codes = person_codes.astype(np.int64)
    weights = chunk[attendance_column].to_numpy(dtype=np.float64)
    np.add.at(accumulators["input"], person_codes, weights)

    merged = chunk.merge(
        context_lookup,
        on=context_columns,
        how="left",
        validate="many_to_one",
        sort=False,
    )

    matched_mask = merged[comp_cols[0]].notna().to_numpy()
    if not matched_mask.any():
        del chunk, merged
        gc.collect()
        return

    matched = merged.loc[matched_mask]
    codes = accumulators["person_to_code"].reindex(matched[person_column]).to_numpy(dtype=np.int64)
    hours = matched[attendance_column].to_numpy(dtype=np.float64)
    compositions = matched[comp_cols].to_numpy(dtype=np.float64, copy=False)

    np.add.at(accumulators["matched"], codes, hours)
    np.add.at(accumulators["contexts"], codes, 1)

    for j in range(len(comp_cols)):
        np.add.at(
            accumulators["exposure_numerator"][:, j],
            codes,
            hours * compositions[:, j],
        )

    del chunk, merged, matched, person_codes, weights, codes, hours, compositions
    gc.collect()


def _finalize_individual_scores(
    accumulators: dict,
    population_shares: pd.Series,
    categories: list[str],
    score_column: str,
    person_column: str = "ID",
) -> pd.DataFrame:
    matched = accumulators["matched"]
    total_input = accumulators["input"]
    no_input = total_input <= 0
    if no_input.any():
        ids = accumulators["person_ids"][no_input][:20].tolist()
        raise ValueError(
            f"{no_input.sum():,} IDs have no positive attendance for {score_column}. "
            f"Example IDs: {ids}"
        )

    unmatched = total_input - matched
    tolerance = np.maximum(1e-9, 1e-10 * total_input)
    incomplete = np.abs(unmatched) > tolerance
    '''if incomplete.any():
        ids = accumulators["person_ids"][incomplete]
        audit = pd.DataFrame(
            {
                person_column: ids[:20].to_numpy(),
                "TOTAL_INPUT_ATTENDANCE_HOURS": total_input[incomplete][:20],
                "MATCHED_ATTENDANCE_HOURS": matched[incomplete][:20],
                "UNMATCHED_ATTENDANCE_HOURS": unmatched[incomplete][:20],
            }
        )
        raise ValueError(
            f"{incomplete.sum():,} IDs have unmatched schedule time for "
            f"{score_column}. Paper-consistent exposure requires the complete "
            f"schedule. Total unmatched time: {unmatched[incomplete].sum():,.6f} "
            f"hours. Examples:\n{audit.to_string(index=False)}"
        )
    '''
    # Paper: tau_iq = sum_c(t_i,c * composition_q,c) / sum_c(t_i,c).
    exposure = accumulators["exposure_numerator"] / total_input[:, None]

    reference = population_shares.reindex(categories).to_numpy(dtype=np.float64)
    denominator = 2.0 * (1.0 - reference.min())
    scores = np.abs(exposure - reference[None, :]).sum(axis=1) / denominator

    out = pd.DataFrame(
        {
            person_column: accumulators["person_ids"].to_numpy(),
            score_column: np.clip(scores, 0.0, 1.0),
            "MATCHED_ATTENDANCE_HOURS": matched,
            "TOTAL_INPUT_ATTENDANCE_HOURS": total_input,
            "N_MATCHED_CONTEXTS": accumulators["contexts"],
        }
    )

    for j, category in enumerate(categories):
        out[f"EXPOSURE_{category}"] = exposure[:, j]

    out["UNMATCHED_ATTENDANCE_HOURS"] = (
        out["TOTAL_INPUT_ATTENDANCE_HOURS"] - out["MATCHED_ATTENDANCE_HOURS"]
    ).clip(lower=0.0)

    out["MATCHED_ATTENDANCE_SHARE"] = (
        out["MATCHED_ATTENDANCE_HOURS"] / out["TOTAL_INPUT_ATTENDANCE_HOURS"]
    )
    out["UNMATCHED_ATTENDANCE_SHARE"] = (
        out["UNMATCHED_ATTENDANCE_HOURS"] / out["TOTAL_INPUT_ATTENDANCE_HOURS"]
    )

    return out


def select_attribute_context(
    context_result: pd.DataFrame,
    attribute_label: str,
    name: str,
) -> pd.DataFrame:
    """Select attribute rows robustly to capitalization and whitespace."""
    validate_required_columns(context_result, ["ATTRIBUTE"], name)
    labels = context_result["ATTRIBUTE"].astype("string").str.strip().str.upper()
    selected = context_result.loc[labels.eq(attribute_label.strip().upper())].copy()
    if selected.empty:
        raise ValueError(
            f"{name} contains no rows for {attribute_label}. "
            f"Available attributes: {sorted(labels.dropna().unique().tolist())}"
        )
    return selected


def calculate_non_temporal_individual_model(
    data: pd.DataFrame,
    context_result: pd.DataFrame,
    context_columns: list[str],
    population_shares: pd.Series,
    categories: list[str],
    score_column: str,
    chunk_size: int = 2_000_000,
    person_column: str = "ID",
) -> pd.DataFrame:
    """Low-RAM place or place-purpose individual segregation."""
    lookup, comp_cols = _prepare_context_lookup(context_result, context_columns, categories)

    person_ids = pd.Index(data[person_column].drop_duplicates())
    acc = _initialize_accumulators(person_ids, len(categories))

    group_columns = [person_column, *context_columns]
    attendance = (
        data.groupby(group_columns, observed=True, sort=False)["ACTIVITY_DURATION_HOURS"]
        .sum()
        .rename("INDIVIDUAL_ATTENDANCE_HOURS")
        .reset_index()
    )

    for start in range(0, len(attendance), chunk_size):
        _accumulate_chunk(
            attendance.iloc[start : start + chunk_size],
            lookup,
            context_columns,
            comp_cols,
            acc,
            person_column=person_column,
        )

    out = _finalize_individual_scores(
        acc,
        population_shares,
        categories,
        score_column,
        person_column,
    )

    del attendance, lookup, acc
    gc.collect()
    return out


def calculate_temporal_individual_models(
    data: pd.DataFrame,
    place_time_result: pd.DataFrame,
    place_purpose_time_result: pd.DataFrame,
    population_shares: pd.Series,
    categories: list[str],
    person_column: str = "ID",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Calculate place-time and place-purpose-time individual segregation
    without constructing a full hourly-expanded dataframe.

    Only one hour is materialized at a time.
    """
    pt_contexts = ["ACTIVITY_SUBZONE", "ACTIVITY_TIME_SLOT"]
    ppt_contexts = [
        "ACTIVITY_SUBZONE",
        "ACTIVITY_PURPOSE",
        "ACTIVITY_TIME_SLOT",
    ]

    pt_lookup, pt_comp_cols = _prepare_context_lookup(place_time_result, pt_contexts, categories)
    ppt_lookup, ppt_comp_cols = _prepare_context_lookup(
        place_purpose_time_result, ppt_contexts, categories
    )

    person_ids = pd.Index(data[person_column].drop_duplicates())
    pt_acc = _initialize_accumulators(person_ids, len(categories))
    ppt_acc = _initialize_accumulators(person_ids, len(categories))

    starts = data["ACTIVITY_STARTTIME"].to_numpy(dtype=np.float64)
    ends = data["ACTIVITY_ENDTIME"].to_numpy(dtype=np.float64)
    durations = data["ACTIVITY_DURATION_HOURS"].to_numpy(dtype=np.float64)
    inclusive_slot_counts = ends - starts + 1.0
    base = data[[person_column, "ACTIVITY_SUBZONE", "ACTIVITY_PURPOSE"]]

    # The observation day contains the 24 inclusive hourly slots 0 through 23.
    for hour in range(24):
        mask = (starts <= hour) & (ends >= hour)
        if not mask.any():
            continue

        hour_rows = base.loc[mask].copy()
        hour_rows["ACTIVITY_TIME_SLOT"] = np.int8(hour)
        # STARTTIME and ENDTIME are inclusive hourly slots. Because preparation
        # verifies duration = end - start + 1, each occupied slot receives one
        # attendance-hour and total attendance is conserved exactly.
        hour_rows["INDIVIDUAL_ATTENDANCE_HOURS"] = durations[mask] / inclusive_slot_counts[mask]

        ppt_att = hour_rows.groupby(
            [
                person_column,
                "ACTIVITY_SUBZONE",
                "ACTIVITY_PURPOSE",
                "ACTIVITY_TIME_SLOT",
            ],
            observed=True,
            sort=False,
            as_index=False,
        )["INDIVIDUAL_ATTENDANCE_HOURS"].sum()

        pt_att = ppt_att.groupby(
            [person_column, "ACTIVITY_SUBZONE", "ACTIVITY_TIME_SLOT"],
            observed=True,
            sort=False,
            as_index=False,
        )["INDIVIDUAL_ATTENDANCE_HOURS"].sum()

        _accumulate_chunk(
            pt_att,
            pt_lookup,
            pt_contexts,
            pt_comp_cols,
            pt_acc,
            person_column=person_column,
        )
        _accumulate_chunk(
            ppt_att,
            ppt_lookup,
            ppt_contexts,
            ppt_comp_cols,
            ppt_acc,
            person_column=person_column,
        )

        del hour_rows, pt_att, ppt_att, mask
        gc.collect()

    pt = _finalize_individual_scores(
        pt_acc,
        population_shares,
        categories,
        "INDIVIDUAL_PLACE_TIME_SEGREGATION",
        person_column,
    )
    ppt = _finalize_individual_scores(
        ppt_acc,
        population_shares,
        categories,
        "INDIVIDUAL_PLACE_PURPOSE_TIME_SEGREGATION",
        person_column,
    )

    del pt_lookup, ppt_lookup, pt_acc, ppt_acc
    gc.collect()
    return pt, ppt


def calculate_four_individual_models_for_attribute_low_ram(
    activities: pd.DataFrame,
    all_place_segregation: pd.DataFrame,
    all_place_purpose_segregation: pd.DataFrame,
    all_place_time_segregation: pd.DataFrame,
    all_place_purpose_time_segregation: pd.DataFrame,
    attribute_label: str,
    group_column: str,
    group_categories: Iterable,
    chunk_size: int = 2_000_000,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    categories = list(group_categories)
    data = prepare_activity_sample(activities, group_column, categories)
    shares = calculate_population_shares(data, group_column, categories)
    person_group = data[["ID", group_column]].drop_duplicates(subset=["ID"])

    place_context = select_attribute_context(
        all_place_segregation, attribute_label, "all_place_segregation"
    )
    pp_context = select_attribute_context(
        all_place_purpose_segregation,
        attribute_label,
        "all_place_purpose_segregation",
    )
    pt_context = select_attribute_context(
        all_place_time_segregation, attribute_label, "all_place_time_segregation"
    )
    ppt_context = select_attribute_context(
        all_place_purpose_time_segregation,
        attribute_label,
        "all_place_purpose_time_segregation",
    )

    place = calculate_non_temporal_individual_model(
        data,
        place_context,
        ["ACTIVITY_SUBZONE"],
        shares,
        categories,
        "INDIVIDUAL_PLACE_SEGREGATION",
        chunk_size,
    )

    place_purpose = calculate_non_temporal_individual_model(
        data,
        pp_context,
        ["ACTIVITY_SUBZONE", "ACTIVITY_PURPOSE"],
        shares,
        categories,
        "INDIVIDUAL_PLACE_PURPOSE_SEGREGATION",
        chunk_size,
    )

    place_time, place_purpose_time = calculate_temporal_individual_models(
        data,
        pt_context,
        ppt_context,
        shares,
        categories,
    )

    outputs = [place, place_purpose, place_time, place_purpose_time]
    for i, output in enumerate(outputs):
        output = output.merge(
            person_group,
            on="ID",
            how="left",
            validate="one_to_one",
        )
        output["ATTRIBUTE"] = attribute_label
        outputs[i] = output

    del data, shares, person_group
    gc.collect()
    return tuple(outputs)


def calculate_all_individual_segregation_models_low_ram(
    activities: pd.DataFrame,
    all_place_segregation: pd.DataFrame,
    all_place_purpose_segregation: pd.DataFrame,
    all_place_time_segregation: pd.DataFrame,
    all_place_purpose_time_segregation: pd.DataFrame,
    chunk_size: int = 2_000_000,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Run every attribute on the same IDs and complete schedule-time support."""
    activities = prepare_common_activity_cohort(activities)
    expected_ids = pd.Index(activities["ID"].dropna().unique())
    place_results = []
    pp_results = []
    pt_results = []
    ppt_results = []

    for attribute_label, group_column, categories in ATTRIBUTE_CONFIGURATIONS:
        print(f"Calculating low-RAM individual {attribute_label} segregation...")

        place, pp, pt, ppt = calculate_four_individual_models_for_attribute_low_ram(
            activities=activities,
            all_place_segregation=all_place_segregation,
            all_place_purpose_segregation=all_place_purpose_segregation,
            all_place_time_segregation=all_place_time_segregation,
            all_place_purpose_time_segregation=all_place_purpose_time_segregation,
            attribute_label=attribute_label,
            group_column=group_column,
            group_categories=categories,
            chunk_size=chunk_size,
        )

        place_results.append(place)
        pp_results.append(pp)
        pt_results.append(pt)
        ppt_results.append(ppt)

        del place, pp, pt, ppt
        gc.collect()

    outputs = (
        pd.concat(place_results, ignore_index=True, sort=False),
        pd.concat(pp_results, ignore_index=True, sort=False),
        pd.concat(pt_results, ignore_index=True, sort=False),
        pd.concat(ppt_results, ignore_index=True, sort=False),
    )

    resolution_names = ["place", "place-purpose", "place-time", "place-purpose-time"]
    failures = []
    for resolution_name, output in zip(resolution_names, outputs):
        for attribute_label, _, _ in ATTRIBUTE_CONFIGURATIONS:
            output_ids = pd.Index(
                output.loc[output["ATTRIBUTE"].eq(attribute_label), "ID"].unique()
            )
            missing = expected_ids.difference(output_ids)
            unexpected = output_ids.difference(expected_ids)
            if len(missing) or len(unexpected):
                failures.append(
                    {
                        "RESOLUTION": resolution_name,
                        "ATTRIBUTE": attribute_label,
                        "N_EXPECTED": len(expected_ids),
                        "N_OUTPUT": len(output_ids),
                        "N_MISSING": len(missing),
                        "N_UNEXPECTED": len(unexpected),
                        "EXAMPLE_MISSING_IDS": missing[:20].tolist(),
                    }
                )
    if failures:
        raise RuntimeError(
            "ID-cohort inconsistency detected:\n" f"{pd.DataFrame(failures).to_string(index=False)}"
        )

    print(
        f"All four resolutions and all attributes retain all " f"{len(expected_ids):,} input IDs."
    )
    return outputs


# ============================================================
# COMBINE FOUR INDIVIDUAL SCORES BY ATTRIBUTE
# ============================================================


def extract_individual_score_for_attribute(
    dataframe: pd.DataFrame,
    attribute: str,
    input_score_column: str,
    output_score_column: str,
    id_column: str = "ID",
    attribute_column: str = "ATTRIBUTE",
) -> pd.DataFrame:
    """Extract one score for one attribute, retaining one row per ID."""

    validate_required_columns(
        dataframe,
        [id_column, attribute_column, input_score_column],
        input_score_column,
    )

    result = dataframe.loc[
        dataframe[attribute_column].astype("string").str.upper().eq(attribute.upper()),
        [id_column, input_score_column],
    ].copy()

    if result.empty:
        raise ValueError(f"No {attribute} rows were found for {input_score_column}.")

    result[input_score_column] = pd.to_numeric(
        result[input_score_column],
        errors="coerce",
    )

    result = result.dropna(subset=[id_column, input_score_column]).copy()

    duplicate_mask = result.duplicated(
        subset=[id_column],
        keep=False,
    )

    if duplicate_mask.any():
        examples = result.loc[duplicate_mask].head(20)
        raise ValueError(
            f"Duplicate IDs found for {attribute} and "
            f"{input_score_column}.\nExamples:\n{examples}"
        )

    return result.rename(columns={input_score_column: output_score_column}).reset_index(drop=True)


def combine_individual_segregation_by_attribute(
    attribute: str,
    all_individual_place_segregation: pd.DataFrame,
    all_individual_place_purpose_segregation: pd.DataFrame,
    all_individual_place_time_segregation: pd.DataFrame,
    all_individual_place_purpose_time_segregation: pd.DataFrame,
    id_column: str = "ID",
) -> pd.DataFrame:
    """
    Combine the four individual segregation scores for one attribute.

    Final columns:
        ID
        PLACE_SEGREGATION
        PLACE_PURPOSE_SEGREGATION
        PLACE_TIME_SEGREGATION
        PLACE_PURPOSE_TIME_SEGREGATION
    """

    place = extract_individual_score_for_attribute(
        dataframe=all_individual_place_segregation,
        attribute=attribute,
        input_score_column="INDIVIDUAL_PLACE_SEGREGATION",
        output_score_column="PLACE_SEGREGATION",
        id_column=id_column,
    )

    place_purpose = extract_individual_score_for_attribute(
        dataframe=all_individual_place_purpose_segregation,
        attribute=attribute,
        input_score_column="INDIVIDUAL_PLACE_PURPOSE_SEGREGATION",
        output_score_column="PLACE_PURPOSE_SEGREGATION",
        id_column=id_column,
    )

    place_time = extract_individual_score_for_attribute(
        dataframe=all_individual_place_time_segregation,
        attribute=attribute,
        input_score_column="INDIVIDUAL_PLACE_TIME_SEGREGATION",
        output_score_column="PLACE_TIME_SEGREGATION",
        id_column=id_column,
    )

    place_purpose_time = extract_individual_score_for_attribute(
        dataframe=all_individual_place_purpose_time_segregation,
        attribute=attribute,
        input_score_column="INDIVIDUAL_PLACE_PURPOSE_TIME_SEGREGATION",
        output_score_column="PLACE_PURPOSE_TIME_SEGREGATION",
        id_column=id_column,
    )

    combined = (
        place.merge(
            place_purpose,
            on=id_column,
            how="outer",
            validate="one_to_one",
        )
        .merge(
            place_time,
            on=id_column,
            how="outer",
            validate="one_to_one",
        )
        .merge(
            place_purpose_time,
            on=id_column,
            how="outer",
            validate="one_to_one",
        )
    )

    score_columns = [
        "PLACE_SEGREGATION",
        "PLACE_PURPOSE_SEGREGATION",
        "PLACE_TIME_SEGREGATION",
        "PLACE_PURPOSE_TIME_SEGREGATION",
    ]

    for column in score_columns:
        combined[column] = pd.to_numeric(
            combined[column],
            errors="coerce",
        )

    combined = combined[[id_column, *score_columns]].sort_values(id_column).reset_index(drop=True)

    return combined


def validate_combined_individual_results(
    dataframe: pd.DataFrame,
    attribute: str,
    id_column: str = "ID",
) -> dict:
    """Validate one final attribute-level dataframe."""

    score_columns = [
        "PLACE_SEGREGATION",
        "PLACE_PURPOSE_SEGREGATION",
        "PLACE_TIME_SEGREGATION",
        "PLACE_PURPOSE_TIME_SEGREGATION",
    ]

    validate_required_columns(
        dataframe,
        [id_column, *score_columns],
        f"individual_segregation_{attribute.lower()}",
    )

    available_scores = dataframe[score_columns].stack()

    return {
        "ATTRIBUTE": attribute,
        "N_ROWS": len(dataframe),
        "N_UNIQUE_IDS": dataframe[id_column].nunique(),
        "N_DUPLICATE_IDS": int(dataframe.duplicated(subset=[id_column]).sum()),
        "N_COMPLETE_ROWS": int(dataframe[score_columns].notna().all(axis=1).sum()),
        "N_ROWS_WITH_MISSING_SCORE": int(dataframe[score_columns].isna().any(axis=1).sum()),
        "ALL_AVAILABLE_SCORES_IN_0_1": bool(available_scores.between(0.0, 1.0).all()),
    }


def validate_individual_attendance_conservation(
    all_individual_place_segregation: pd.DataFrame,
    all_individual_place_purpose_segregation: pd.DataFrame,
    all_individual_place_time_segregation: pd.DataFrame,
    all_individual_place_purpose_time_segregation: pd.DataFrame,
    id_column: str = "ID",
    attribute_column: str = "ATTRIBUTE",
    tolerance: float = 1e-9,
) -> pd.DataFrame:
    """Verify that every individual uses identical attendance in all resolutions."""
    specifications = [
        ("PLACE", all_individual_place_segregation),
        ("PLACE_PURPOSE", all_individual_place_purpose_segregation),
        ("PLACE_TIME", all_individual_place_time_segregation),
        ("PLACE_PURPOSE_TIME", all_individual_place_purpose_time_segregation),
    ]
    keys = [id_column, attribute_column]
    merged = None
    hour_columns = []
    for label, dataframe in specifications:
        required = keys + ["TOTAL_INPUT_ATTENDANCE_HOURS", "MATCHED_ATTENDANCE_HOURS"]
        validate_required_columns(dataframe, required, label)
        if dataframe.duplicated(keys).any():
            raise ValueError(f"{label} has duplicate ID × ATTRIBUTE rows.")
        total_column = f"{label}_TOTAL_INPUT_ATTENDANCE_HOURS"
        matched_column = f"{label}_MATCHED_ATTENDANCE_HOURS"
        current = dataframe[required].rename(
            columns={
                "TOTAL_INPUT_ATTENDANCE_HOURS": total_column,
                "MATCHED_ATTENDANCE_HOURS": matched_column,
            }
        )
        hour_columns.append(total_column)
        merged = current if merged is None else merged.merge(
            current, on=keys, how="outer", validate="one_to_one"
        )

    if merged[hour_columns].isna().any().any():
        bad = merged.loc[merged[hour_columns].isna().any(axis=1), keys + hour_columns].head(20)
        raise ValueError(
            "The four individual models do not contain identical ID × ATTRIBUTE cohorts.\n"
            f"Examples:\n{bad.to_string(index=False)}"
        )

    hour_values = merged[hour_columns].to_numpy(dtype=np.float64)
    merged["MAX_ATTENDANCE_DIFFERENCE"] = hour_values.max(axis=1) - hour_values.min(axis=1)
    scale = np.maximum(1.0, hour_values.max(axis=1))
    merged["ATTENDANCE_CONSISTENT"] = merged["MAX_ATTENDANCE_DIFFERENCE"].to_numpy() <= (
        tolerance * scale
    )
    if not merged["ATTENDANCE_CONSISTENT"].all():
        bad = merged.loc[~merged["ATTENDANCE_CONSISTENT"], keys + hour_columns].head(20)
        raise ValueError(
            "Individual attendance is not conserved across resolutions.\n"
            f"Examples:\n{bad.to_string(index=False)}"
        )

    return (
        merged.groupby(attribute_column, observed=True, sort=False)
        .agg(
            N_INDIVIDUALS=(id_column, "nunique"),
            MAX_ATTENDANCE_DIFFERENCE=("MAX_ATTENDANCE_DIFFERENCE", "max"),
            SHARE_ATTENDANCE_CONSISTENT=("ATTENDANCE_CONSISTENT", "mean"),
        )
        .reset_index()
    )


def analyze_individual_resolution_ordering(
    dataframe: pd.DataFrame,
    attribute: str,
    id_column: str = "ID",
    tolerance: float = 1e-10,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Calculate empirical cross-resolution gaps for individuals.

    These comparisons are diagnostics, not universal convexity results, because
    individual context weights differ from the population weights used to form
    lower-resolution place compositions. No ordering is imposed between the
    purpose-resolved and time-resolved scores.
    """
    score_columns = [
        "PLACE_SEGREGATION",
        "PLACE_PURPOSE_SEGREGATION",
        "PLACE_TIME_SEGREGATION",
        "PLACE_PURPOSE_TIME_SEGREGATION",
    ]
    validate_required_columns(dataframe, [id_column, *score_columns], attribute)
    result = dataframe.copy()
    if result[id_column].duplicated().any():
        raise ValueError(f"{attribute} contains duplicate IDs.")
    if result[score_columns].isna().any().any():
        raise ValueError(f"{attribute} contains missing individual segregation scores.")

    gap_definitions = {
        "PURPOSE_MINUS_PLACE_GAP": (
            "PLACE_PURPOSE_SEGREGATION", "PLACE_SEGREGATION"
        ),
        "TIME_MINUS_PLACE_GAP": (
            "PLACE_TIME_SEGREGATION", "PLACE_SEGREGATION"
        ),
        "PURPOSE_TIME_MINUS_PLACE_GAP": (
            "PLACE_PURPOSE_TIME_SEGREGATION", "PLACE_SEGREGATION"
        ),
        "PURPOSE_TIME_MINUS_PURPOSE_GAP": (
            "PLACE_PURPOSE_TIME_SEGREGATION", "PLACE_PURPOSE_SEGREGATION"
        ),
        "PURPOSE_TIME_MINUS_TIME_GAP": (
            "PLACE_PURPOSE_TIME_SEGREGATION", "PLACE_TIME_SEGREGATION"
        ),
    }
    flag_definitions = {
        "PLACE_LE_PURPOSE": "PURPOSE_MINUS_PLACE_GAP",
        "PLACE_LE_TIME": "TIME_MINUS_PLACE_GAP",
        "PLACE_LE_PURPOSE_TIME": "PURPOSE_TIME_MINUS_PLACE_GAP",
        "PURPOSE_LE_PURPOSE_TIME": "PURPOSE_TIME_MINUS_PURPOSE_GAP",
        "TIME_LE_PURPOSE_TIME": "PURPOSE_TIME_MINUS_TIME_GAP",
    }
    for gap_column, (higher_column, lower_column) in gap_definitions.items():
        result[gap_column] = result[higher_column] - result[lower_column]
    for flag_column, gap_column in flag_definitions.items():
        result[flag_column] = result[gap_column] >= -tolerance

    summary_row = {
        "ATTRIBUTE": attribute,
        "N_INDIVIDUALS": len(result),
    }
    for gap_column in gap_definitions:
        summary_row[f"MEAN_{gap_column}"] = result[gap_column].mean()
        summary_row[f"MEDIAN_{gap_column}"] = result[gap_column].median()
        summary_row[f"MIN_{gap_column}"] = result[gap_column].min()
        summary_row[f"MAX_{gap_column}"] = result[gap_column].max()
    for flag_column in flag_definitions:
        summary_row[f"SHARE_{flag_column}"] = result[flag_column].mean()

    return result, pd.DataFrame([summary_row])


# ============================================================
# RUN
# ============================================================

for frac in ['0.1']:#['0.2', '0.4', '0.6', 'true']:
    print(f"frac {frac}____________________________________________________")

    activities = pd.read_csv(
        path_segregation + f"val_segregation_activities_{frac}.csv"
    )

    file_paths = {
        "place": os.path.join(
            path_segregation,
            f"val_all_place_segregation_{frac}.csv",

        ),
        "purpose": os.path.join(
            path_segregation,
            f"val_all_place_purpose_segregation_{frac}.csv",
        ),
        "time": os.path.join(
            path_segregation,
            f"val_all_place_time_segregation_{frac}.csv",
        ),
        "purpose_time": os.path.join(
            path_segregation,
            f"val_all_place_purpose_time_segregation_{frac}.csv",
        ),
    }

    all_place_segregation = pd.read_csv(file_paths["place"])
    all_place_purpose_segregation = pd.read_csv(file_paths["purpose"])
    all_place_time_segregation = pd.read_csv(file_paths["time"])
    all_place_purpose_time_segregation = pd.read_csv(file_paths["purpose_time"])

    (
        all_individual_place_segregation,
        all_individual_place_purpose_segregation,
        all_individual_place_time_segregation,
        all_individual_place_purpose_time_segregation,
    ) = calculate_all_individual_segregation_models_low_ram(
        activities=activities,
        all_place_segregation=all_place_segregation,
        all_place_purpose_segregation=all_place_purpose_segregation,
        all_place_time_segregation=all_place_time_segregation,
        all_place_purpose_time_segregation=all_place_purpose_time_segregation,
        # Reduce this to 500_000 or 250_000 if RAM is still limited.
        chunk_size=1_000_000,
    )

    print(all_individual_place_segregation.head())
    print(all_individual_place_purpose_segregation.head())
    print(all_individual_place_time_segregation.head())
    print(all_individual_place_purpose_time_segregation.head())

    attendance_conservation_summary = validate_individual_attendance_conservation(
        all_individual_place_segregation=all_individual_place_segregation,
        all_individual_place_purpose_segregation=all_individual_place_purpose_segregation,
        all_individual_place_time_segregation=all_individual_place_time_segregation,
        all_individual_place_purpose_time_segregation=(
            all_individual_place_purpose_time_segregation
        ),
    )
    print("\nIndividual attendance conservation across resolutions")
    print(attendance_conservation_summary.to_string(index=False))

    # ============================================================
    # CREATE FINAL AGE, GENDER, AND INCOME DATAFRAMES
    # ============================================================
    start_time = time.time()

    individual_segregation_age = combine_individual_segregation_by_attribute(
        attribute="AGE",
        all_individual_place_segregation=all_individual_place_segregation,
        all_individual_place_purpose_segregation=(all_individual_place_purpose_segregation),
        all_individual_place_time_segregation=(all_individual_place_time_segregation),
        all_individual_place_purpose_time_segregation=(
            all_individual_place_purpose_time_segregation
        ),
    )

    gc.collect()

    individual_segregation_gender = combine_individual_segregation_by_attribute(
        attribute="GENDER",
        all_individual_place_segregation=all_individual_place_segregation,
        all_individual_place_purpose_segregation=(all_individual_place_purpose_segregation),
        all_individual_place_time_segregation=(all_individual_place_time_segregation),
        all_individual_place_purpose_time_segregation=(
            all_individual_place_purpose_time_segregation
        ),
    )

    gc.collect()

    individual_segregation_income = combine_individual_segregation_by_attribute(
        attribute="INCOME",
        all_individual_place_segregation=all_individual_place_segregation,
        all_individual_place_purpose_segregation=(all_individual_place_purpose_segregation),
        all_individual_place_time_segregation=(all_individual_place_time_segregation),
        all_individual_place_purpose_time_segregation=(
            all_individual_place_purpose_time_segregation
        ),
    )

    gc.collect()

    # ============================================================
    # EMPIRICAL INDIVIDUAL RESOLUTION ORDERING
    # ============================================================

    individual_segregation_age, individual_ordering_summary_age = (
        analyze_individual_resolution_ordering(individual_segregation_age, "AGE")
    )
    individual_segregation_gender, individual_ordering_summary_gender = (
        analyze_individual_resolution_ordering(individual_segregation_gender, "GENDER")
    )
    individual_segregation_income, individual_ordering_summary_income = (
        analyze_individual_resolution_ordering(individual_segregation_income, "INCOME")
    )
    individual_ordering_summary = pd.concat(
        [
            individual_ordering_summary_age,
            individual_ordering_summary_gender,
            individual_ordering_summary_income,
        ],
        ignore_index=True,
    )
    print("\nEmpirical individual resolution-ordering summary")
    print(
        individual_ordering_summary.to_string(
            index=False,
            float_format=lambda value: f"{value:.12g}",
        )
    )

    # ============================================================
    # VALIDATE FINAL DATAFRAMES
    # ============================================================

    combined_validation = pd.DataFrame(
        [
            validate_combined_individual_results(
                individual_segregation_age,
                "AGE",
            ),
            validate_combined_individual_results(
                individual_segregation_gender,
                "GENDER",
            ),
            validate_combined_individual_results(
                individual_segregation_income,
                "INCOME",
            ),
        ]
    )

    print("\nCombined individual segregation validation")
    print(combined_validation.to_string(index=False))

    # ============================================================
    # SAVE FINAL CSV FILES
    # ============================================================

    os.makedirs(path_segregation, exist_ok=True)

    age_output_path = os.path.join(
        path_segregation,
        f"val_individual_segregation_age_{frac}.csv",

    )

    gender_output_path = os.path.join(
        path_segregation,
        f"val_individual_segregation_gender_{frac}.csv",
    )

    income_output_path = os.path.join(
        path_segregation,
        f"val_individual_segregation_income_{frac}.csv",
    )

    ordering_summary_output_path = os.path.join(
        path_segregation,
        f"val_individual_segregation_ordering_summary_{frac}.csv",
    )

    individual_segregation_age.to_csv(
        age_output_path,
        index=False,
    )

    individual_segregation_gender.to_csv(
        gender_output_path,
        index=False,
    )

    individual_segregation_income.to_csv(
        income_output_path,
        index=False,
    )

    individual_ordering_summary.to_csv(ordering_summary_output_path, index=False)

    print("\nSaved files")
    print(age_output_path)
    print(gender_output_path)
    print(income_output_path)
    print(ordering_summary_output_path)

    print("\nAGE")
    print(individual_segregation_age)

    print("\nGENDER")
    print(individual_segregation_gender)

    print("\nINCOME")
    print(individual_segregation_income)

    del individual_segregation_age, individual_segregation_gender, individual_segregation_income
    gc.collect()

    print("Time take %s seconds" % (time.time() - start_time))
    print("Done!")

# ============================================================
# FINAL RETAINED OUTPUTS
# ============================================================
#
# Detailed outputs:
#   all_individual_place_segregation
#   all_individual_place_purpose_segregation
#   all_individual_place_time_segregation
#   all_individual_place_purpose_time_segregation
#
# Final attribute-level outputs:
#   individual_segregation_age
#   individual_segregation_gender
#   individual_segregation_income
#   individual_ordering_summary
#   attendance_conservation_summary
#
# Saved files:
#   individual_segregation_age_{reg}_M_{M}_nb_{nb}_seed_{seed}.csv
#   individual_segregation_gender_{reg}_M_{M}_nb_{nb}_seed_{seed}.csv
#   individual_segregation_income_{reg}_M_{M}_nb_{nb}_seed_{seed}.csv
#   individual_segregation_ordering_summary_{reg}_M_{M}_nb_{nb}_seed_{seed}.csv
#

frac 0.1____________________________________________________
Common activity cohort: 10,537 unique IDs; all 37,344 activity rows retained
Calculating low-RAM individual AGE segregation...
Calculating low-RAM individual GENDER segregation...
Calculating low-RAM individual INCOME segregation...
All four resolutions and all attributes retain all 10,537 input IDs.
    ID  INDIVIDUAL_PLACE_SEGREGATION  MATCHED_ATTENDANCE_HOURS  \
0  562                      0.073787                      26.0   
1  563                      0.059471                      24.0   
2  564                      0.008977                      26.0   
3  565                      0.038538                      22.0   
4  566                      0.089653                      25.0   

   TOTAL_INPUT_ATTENDANCE_HOURS  N_MATCHED_CONTEXTS  EXPOSURE_age_0  \
0                          26.0                   2             0.0   
1                          24.0                   2             0.0   
2                          